# 5-fluorouracil geometry from a very deformed start (Module 1, third notebook) on a Colab L4

Same notebook as the second Module 1 notebook (xTB basin search, then L-BFGS on Skala-1.1 at def2-SVP, with the Gemini agent and its guard), with **a much more strained start**: about 6 eV above the minimum by GFN2-xTB. It is the stress-test case ("Run 3") next to the mild start of the first notebook ("Run 1") and the 2.5 eV start of the second ("Run 2").

| What | Value |
|---|---|
| Start `D2` | the xTB minimum with a stronger mix of its vibration modes: atoms moved up to about 0.5 to 0.8 A, ring clearly puckered, bonds and angles strongly distorted. The bond pattern is still the same (checked), and no two atoms are closer than 0.9 A. |
| Size of the deformation | about 6 eV above the minimum by GFN2-xTB. This is close to the largest strain at which the molecule still has all its bonds. At 10 eV the mode mixing already stretches bonds past their normal range, and at 50 eV the molecule cannot be written down without breaking or crushing bonds. |
| What it is for | a stress test: does the optimizer still come back to the same structure from a very strained start? It may fail, and if it does the notebook reports that. |

## What happens when you press Run all
1. **Automatic smoke test** (as in the other notebooks).
2. **Start report:** distance of `D2` from the reference, bond-pattern check, and its Skala energy.
3. **Real run:** one Gemini-agent run from `D2` (arm `GS`), def2-SVP, gradient budget 400 (more than the other notebooks, because the start is much farther away).
4. **Result:** energy released, whether it is at least 5 eV, and the comparison with the first notebook's result (shape difference in A, energy difference in meV).
5. **Files:** `MyDrive/fu5_handoff/<SHAPE>_svp_deformed6_run.xyz` and `.json`, `<SHAPE>_deformed6_energy_release.png`. These names differ from the other notebooks', so **Module 2 and the second notebook's files are untouched**.

## Before you start
1. Runtime > Change runtime type > **L4 GPU**.
2. Colab *Secrets*: `GEMINI_API_KEY` (and `GEMINI_API_KEY_2`), notebook access on.
3. Set `SHAPE` in the settings cell (`T1` first), run all. Run the first notebook for the same shape beforehand, so the comparison has something to compare with.

## What this does and does not show
- One run per shape. "Reached the same structure" is a check that could fail; if it does, that is reported and nothing is hidden.
- The strain was set with xTB, so the Skala energy released is known only after the run. The notebook does not change the start if the number is lower than expected.
- This is a stress test of the optimizer. It does not replace the 2.5 eV run as the input to Module 2.
- No speed claim, no comparison between agent and rules.


## Setup

In [ ]:
# @title 0. Install (same packages as the earlier tested notebook, plus the Gemini SDK)
SKALA_VERSION = "2026.9"   # forces verified consistent on 2026.9 (gpu4pyscf 1.8.1, pyscf 2.14.0). 2026.4 gave forces ~2.7x off: do not use it. Restart the session after changing it.
!pip install -q skala-cuda12x=={SKALA_VERSION}
# Only what Colab does not already have. mace-torch is NOT installed: this notebook does not use MACE, and installing it can change torch/e3nn.
!pip install -q ase tblite google-genai

In [ ]:
# @title 1. Settings: edit this cell, then Runtime > Run all
# Which molecule (one Colab session per molecule; results go to separate Drive folders) ----
SHAPE = "T1"             # "T1" diketo (the drug) | "T3" enol, O-H toward C5-F | "T3B" enol, O-H toward N3  <- the only line to change between sessions
MOLECULE = "5FU_" + SHAPE
RUN_MODE = "auto"       # "auto" = automatic smoke test first, then the real run (default) | "smoke" = 1 start, 2 arms, <= 4 steps per pass (wiring check, minutes)
                          # "n1"    = 1 start (R3), arms A0e/A0r/A1/A3, full steps: the go/no-go run (printed at the end)
                          # "full"  = all starts in STARTS below

# What to run ---------------------------------------------------------------
REQUIREMENT = "R0"        # "R0" = def2-SVP only (fast screen, about 20 min per run)  |  "R1" = def2-SVP then 6-31+G(d,p)  |  "R2" = def2-SVP then def2-TZVP
ARMS   = ["A0e", "A0r", "A1", "A2", "A3"]          # optional extras: "A0c" cautious novice, "A0p" deliberately poor schedule, "A3p" Gemini agent started from the poor schedule
STARTS = ["R1", "R2", "R3", "R4", "R5", "Q1", "P0"]  # five designed starts + two controls (Q1 squeezed, P0 relaxed reference), all nudged
RUN_TAG = "_L4_5FU_deformed6"             # suffix for the Drive results folder: set when one experiment is split over two Colab sessions
R0_GRADIENT_BUDGET = 400   # gradient cap per run when REQUIREMENT == 'R0' (150 in the annulene screen; 400 here because the very deformed start is farther from the minimum; 12 atoms in def2-SVP are cheap)
N1_START = "D2"          # start used in "n1" mode ("P0" = relaxed reference: used for the replication check)
N1_ARMS_OVERRIDE = ["GS"]   # HS/H3p = LLM + rules mode | GS/A3p = LLM mode | A2p = rules only. Optional: "P1p" = plain memory-20 baseline on the poor schedule   # e.g. ["A0e", "A1"] to run only some of the four go/no-go arms in this session
SMOKE_TEST = RUN_MODE in ("smoke", "auto")      # auto starts as a smoke test; the grid cell then switches to the real run
N1_TEST = RUN_MODE == "n1"

# Where the chemistry runs --------------------------------------------------
BACKEND    = "skala"      # "skala" = real Skala-1.1 DFT on the GPU (Colab L4)
                          # "standin_xtb" = CPU dry run with GFN2-xTB instead of DFT (wiring test only, NOT a result)
AGENT_MODE = "gemini"     # "gemini" = live Gemini function calling | "mock" = rule-based stand-in (no API)

# Golden rule (fixed, do not change) -----------------------------------------
SPHERICAL_D = True        # pure (spherical) d functions in EVERY basis (PySCF cart=False)
NUDGE_RMS_A = 0.01        # the start files are already nudged out of perfect symmetry by this RMS (A)

# Skala compatibility ---------------------------------------------------------
# Your tested v6/v7 code sets small_rho_cutoff = 1e-10 (drops grid points with negligible density). Skala 2026.9 (the current
# Colab install) rejects such pruned grids ("Grid size mismatch"). 0.0 keeps the full grid: works on 2026.9, energies differ from
# the pruned grid only at the level of the discarded density. To reproduce the old numbers exactly, instead pin the Skala version
# you used when v6 was tested (install cell: !pip install skala-cuda12x==<version>) and set this back to 1e-10.
SMALL_RHO_CUTOFF_L3 = 0.0

# Optimizer and budget ---------------------------------------------------------
GRADIENT_BUDGET_PER_RUN = 400   # one gradient is about 15-20 s on the L4, so 400 is about 2 h per run at most
FORCE_UNIT_FIX = False    # False = keep the tested v7 force convention (eV/Bohr labelled eV/A) so thresholds match earlier runs
WARM_START = True         # arms A1-A3: reuse the previous density as the SCF guess (falls back to 'minao' if it fails)
FIXES_MEMORY = 20         # arms A1-A3: L-BFGS memory (A0 arms keep the v7 value 1)
ENDPOINT_FMAX = 0.02      # common endpoint for every arm (same units as the v7 thresholds)
PROBATION_STEPS = 5
COOLDOWN_STEPS = 5

# Gemini: rotation, back-off and patience ---------------------------------------
GEMINI_KEY_SECRETS = ["GEMINI_API_KEY", "GEMINI_API_KEY_2"]   # Colab secret names (you have 2 keys)
MODEL_FAST   = "gemini-3.8-flash"     # routine in-loop calls
MODEL_PRO    = "gemini-pro-latest"    # middle tier; edit to the Pro ID you have
MODEL_STRONG = "gemini-4-argon"       # high-stakes calls (basin choice, negative curvature)
API_PATIENCE_S    = 1800   # wait at most this long for ONE decision before the run is flagged 'degraded'
API_BACKOFF_BASE_S = 5     # first back-off; doubles each failed round (with jitter)
API_BACKOFF_MAX_S  = 180
API_MODEL_ROTATE_AFTER = 2 # failed rounds on the primary model (both keys) before trying the next model
AGENT_VERSION = "v2"      # "v1" = logic of the first runs (kept for reproducibility) | "v2" = refined: advisor in the packet, no shrinking while the energy falls, step ramp-up, fewer calls
MAX_TOOL_ROUNDS = 6 if AGENT_VERSION == "v1" else 4        # function-calling rounds per decision
RERUN_DEGRADED = True      # on re-running this notebook, redo runs whose agent calls were degraded by API limits

# Google Drive ---------------------------------------------------------------------
USE_DRIVE = True
assert MOLECULE in ("5FU_T1", "5FU_T3", "5FU_T3B", "C10H6", "C11H8"), "SHAPE must be 'T1', 'T3' or 'T3B'"
assert RUN_MODE in ("auto", "smoke", "n1", "full"), "RUN_MODE must be 'auto', 'smoke', 'n1' or 'full'"
DRIVE_SUBDIR = f"agentic_skala_{MOLECULE.lower()}"   # inside MyDrive; one folder per molecule, so two sessions never collide

# Per-molecule facts (the geometry itself comes from the starts cell) -----------------------------
MOL_PROFILES = {
    "C10H6": dict(n_atoms=16, desc="a didehydroannulene ring, 16 atoms, planar, 'System 2' in the v7 notebook",
                  plane_atoms=None,            # planarity is measured over all atoms
                  permute_symmetry=True,       # ring symmetry: RMSD may relabel equivalent atoms
                  nao={"def2-svp": 170, "6-31+g(d,p)": 210, "def2-tzvp": 346}),
    "C11H8": dict(n_atoms=19, desc="an annulene derivative with a CH2 group, 19 atoms, 'System 7' in the v6 notebook; ten ring carbons are planar and the CH2 hydrogens sit out of that plane",
                  plane_atoms=list(range(10)), # planarity is measured over the ten ring carbons (atoms 0-9 of every start file)
                  permute_symmetry=False,      # no usable symmetry: RMSD does not relabel atoms
                  nao={"def2-svp": 194, "6-31+g(d,p)": 238, "def2-tzvp": 389}),
    "5FU_T1": dict(n_atoms=12, formula="C4H3FN2O2", fixed_shape=True, desc="5-fluorouracil, diketo tautomer T1 (5-fluoropyrimidine-2,4(1H,3H)-dione), a planar six-membered ring with two C=O groups, 12 atoms",
                   plane_atoms=None, permute_symmetry=False,
                   nao={"def2-svp": 141, "6-31+g(d,p)": 177, "def2-tzvp": 297}),
    "5FU_T3": dict(n_atoms=12, formula="C4H3FN2O2", fixed_shape=True, desc="5-fluorouracil, 4-hydroxy-2-oxo enol tautomer T3 with the O-H toward C5-F, a planar ring with one C=O and one O-H, 12 atoms",
                   plane_atoms=None, permute_symmetry=False,
                   nao={"def2-svp": 141, "6-31+g(d,p)": 177, "def2-tzvp": 297}),
    "5FU_T3B": dict(n_atoms=12, formula="C4H3FN2O2", fixed_shape=True, desc="5-fluorouracil, 4-hydroxy-2-oxo enol tautomer T3B with the O-H toward N3 (rotamer of T3), a planar ring with one C=O and one O-H, 12 atoms",
                   plane_atoms=None, permute_symmetry=False,
                   nao={"def2-svp": 141, "6-31+g(d,p)": 177, "def2-tzvp": 297}),
}
MOL = MOL_PROFILES[MOLECULE]

if SMOKE_TEST:
    STARTS, ARMS = ["R3"], ["A0e", "A3"]
elif N1_TEST:
    STARTS, ARMS = [N1_START], (N1_ARMS_OVERRIDE or ["A0e", "A1", "A3", "A0r"])    # go/no-go set: expert, random, random+fixes, random+fixes+Gemini (A2 added later if go)

BASIS_PATH = {"R0": ("def2-svp", "def2-svp"), "R1": ("def2-svp", "6-31+g(d,p)"), "R2": ("def2-svp", "def2-tzvp")}[REQUIREMENT]
if REQUIREMENT == "R0":
    GRADIENT_BUDGET_PER_RUN = R0_GRADIENT_BUDGET     # fast screen: a run that is not converged after 150 gradients counts as not converged
print(f"Molecule {MOLECULE} | run mode {RUN_MODE}")
print(f"Requirement {REQUIREMENT}: passes 1-3 in {BASIS_PATH[0]}, passes 4-6 in {BASIS_PATH[1]} | backend={BACKEND} | agent={AGENT_MODE}")
print(f"Arms: {ARMS} | starts: {STARTS} | spherical d: {SPHERICAL_D} | start nudge: {NUDGE_RMS_A} A RMS")
if SMOKE_TEST:
    print("SMOKE TEST = WIRING ONLY: 4 steps per pass is far too short to show any benefit of the agent. Do not read results from it.")
if BACKEND != "skala":
    print("WARNING: BACKEND is not 'skala'. Numbers from this run are a wiring test only and must not be reported.")

In [ ]:
# @title 2. Google Drive, results folder, GPU check
import os, json, time, platform
RESULTS_ROOT = f"/content/agentic_skala_{MOLECULE.lower()}"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_ROOT = f"/content/drive/MyDrive/{DRIVE_SUBDIR}"
        print("Google Drive mounted.")
    except Exception as e:
        print(f"Drive not mounted ({type(e).__name__}); results go to {RESULTS_ROOT} and will vanish when Colab disconnects.")
EXPERIMENT_DIR = os.path.join(RESULTS_ROOT, ("SMOKE_" if SMOKE_TEST else "N1_" if N1_TEST else "") + f"requirement_{REQUIREMENT}" + RUN_TAG + ("" if BACKEND == "skala" else "_STANDIN"))
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
print("Every run writes readable JSON files under:", EXPERIMENT_DIR)

GPU_NAME = "none"
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
        if "L4" not in GPU_NAME:
            print("Note: this notebook was written for the Colab L4; other GPUs work but timings will differ.")
    elif BACKEND == "skala":
        raise RuntimeError("BACKEND='skala' needs a GPU. Use Runtime > Change runtime type > L4 GPU.")
except ImportError:
    if BACKEND == "skala":
        raise
import importlib.metadata as _imd
def _pkg(p):
    try: return _imd.version(p)
    except Exception: return None
json.dump({"gpu": GPU_NAME, "python": platform.python_version(), "started": time.strftime("%Y-%m-%dT%H:%M:%S"),
           "packages": {p: _pkg(p) for p in ("skala", "skala-cuda12x", "gpu4pyscf-cuda12x", "pyscf", "torch", "cupy-cuda12x", "e3nn", "ase", "google-genai")}},
          open(os.path.join(EXPERIMENT_DIR, "environment.json"), "w"), indent=2)

## Chemistry engine: GPU utilities, Skala calculator, starts, settings

In [ ]:
# @title 3. GPU memory utilities (verbatim from the earlier tested v7 notebook)
import gc, os
# Compatibility fix: e3nn (used by Skala) loads a constants file with torch.load; PyTorch >= 2.6 refuses it by default
# ("Weights only load failed ... slice"). This is the standard workaround for a trusted, pip-installed package file.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
try:
    import torch
    try: torch.serialization.add_safe_globals([slice])
    except Exception: pass
    import cupy as cp
    HAVE_GPU_LIBS = True
except Exception:
    HAVE_GPU_LIBS = False

if HAVE_GPU_LIBS:
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    def print_gpu_memory_status():
        py_alloc = torch.cuda.memory_allocated() / 1e9
        py_res = torch.cuda.memory_reserved() / 1e9
        cp_pool = cp.get_default_memory_pool()
        print(f"   GPU memory | PyTorch active {py_alloc:.2f} GB, reserved {py_res:.2f} GB | CuPy active {cp_pool.used_bytes()/1e9:.2f} GB, pool {cp_pool.total_bytes()/1e9:.2f} GB")

    def hard_reset_gpu_allocator_pool(verbose=True):
        """Forces a complete hard-reset of the GPU memory pool (tested v7 routine)."""
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
        cp.get_default_memory_pool().set_limit(0)
        cp.get_default_memory_pool().set_limit(None)
        if verbose:
            torch.cuda.synchronize()
            print(f"   GPU allocator reset: PyTorch allocated {torch.cuda.memory_allocated()/(1024**2):.1f} MB")

    def dynamic_step_evacuation():
        torch.cuda.synchronize()
        cp.cuda.Stream.null.synchronize()
        gc.collect()
        torch.cuda.empty_cache()
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
else:
    def print_gpu_memory_status(): pass
    def hard_reset_gpu_allocator_pool(verbose=True): pass
    def dynamic_step_evacuation(): gc.collect()

In [ ]:
# @title 4. Skala-1.1 GPU calculator (the earlier tested v7 calculator + spherical-d rule + telemetry)
import io, gc, os, time
import numpy as np
from ase.calculators.calculator import Calculator, all_changes

HARTREE_TO_EV = 27.211386
BOHR_TO_ANG = 0.529177210903
WARM = {"mol": None, "dm": None}      # density from the previous SCF, reused as the next guess when WARM_START is on

def reset_warm_start():
    WARM["mol"], WARM["dm"] = None, None

def _to_numpy(x):
    if hasattr(x, "get"):             # cupy array
        x = x.get()
    return np.asarray(x, dtype=float)

class NativeGPUSkalaCalculator(Calculator):
    """v7 NativeGPUSkalaCalculator. Changes: (1) cart=False is explicit (golden rule: pure d functions),
    (2) telemetry counters, (3) optional density warm start, (4) optional force-unit fix. The SCF/grid/memory settings
    are exactly the tested ones."""
    implemented_properties = ['energy', 'forces']

    def __init__(self, fixed_spin=0, fixed_charge=0, basis_set="def2-svp", grid_level=3, conv_tol=1e-5, verbose=0,
                 warm_start=False, grid_response=False, align=None, rho_cut=None, **kwargs):
        super().__init__(**kwargs)
        self.rho_cut = rho_cut                  # small_rho_cutoff override (diagnostic only); None = SMALL_RHO_CUTOFF_L3
        self.align = align                      # grids.alignment; None = library default (diagnostic only)
        self.grid_response = grid_response      # diagnostic only (force check cell); off in every production run
        self.fixed_spin, self.fixed_charge = fixed_spin, fixed_charge
        self.basis_set, self.grid_level = basis_set, grid_level
        self.conv_tol, self.verbose, self.warm_start = conv_tol, verbose, warm_start
        self.n_scf = 0                 # every SCF (energy-only or with gradient)
        self.n_gradients = 0           # SCFs that also produced a gradient
        self.last_scf = {}
        self.nao = None

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        from pyscf import gto, dft
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"   # e3nn / PyTorch>=2.6 loading fix (see GPU utilities cell)
        from skala.gpu4pyscf import SkalaKS, SkalaUKS
        super().calculate(atoms, properties, system_changes)
        t0 = time.perf_counter()
        gc.collect(); torch.cuda.empty_cache(); cp.get_default_memory_pool().free_all_blocks()

        atom_list = [[self.atoms.symbols[i], self.atoms.positions[i]] for i in range(len(self.atoms))]
        cpu_mol = gto.M(atom=atom_list, basis=self.basis_set, charge=self.fixed_charge, spin=self.fixed_spin,
                        cart=False, verbose=0)                      # GOLDEN RULE: spherical (pure) d functions
        self.nao = int(cpu_mol.nao)
        gpu_mol = cpu_mol.to_gpu()

        if self.fixed_spin > 0:
            base_ks = SkalaUKS(gpu_mol, xc="skala-1.1"); initial_density_guess = 'atom'
        else:
            base_ks = SkalaKS(gpu_mol, xc="skala-1.1");  initial_density_guess = 'minao'

        base_ks.grids.level = self.grid_level
        if self.align is not None: base_ks.grids.alignment = self.align      # newer Skala/gpu4pyscf builds: padded grids break the per-atom grid-size check (error 'Grid size mismatch')
        if self.grid_level >= 3:
            base_ks.grids.prune = dft.gen_grid.nwchem_prune
            base_ks.small_rho_cutoff = SMALL_RHO_CUTOFF_L3 if self.rho_cut is None else self.rho_cut   # tested value 1e-10; see settings cell
            base_ks.max_memory = 2000        # tested: memory-safe chunking at grid level 3
        else:
            base_ks.grids.prune = dft.gen_grid.sg1_prune
            base_ks.small_rho_cutoff = 1e-8
            base_ks.max_memory = 4000
        base_ks.verbose = self.verbose
        base_ks.conv_tol = self.conv_tol

        scf_engine = base_ks.density_fit(auxbasis="def2-universal-jkfit")
        scf_engine.init_guess = initial_density_guess

        warm_used, warm_failed = False, False
        energy_ha = None
        if self.warm_start and self.fixed_spin == 0 and WARM["dm"] is not None:
            try:
                from pyscf.scf import addons
                dm0 = cp.asarray(addons.project_dm_nr2nr(WARM["mol"], WARM["dm"], cpu_mol))
                energy_ha = float(scf_engine.kernel(dm0=dm0))
                warm_used = bool(getattr(scf_engine, "converged", True))
                if not warm_used:
                    warm_failed = True; energy_ha = None
            except Exception as e:
                warm_failed = True; energy_ha = None
                if self.verbose: print("   warm start failed, using minao:", repr(e)[:120])
        if energy_ha is None:
            scf_engine.init_guess = initial_density_guess
            energy_ha = float(scf_engine.kernel())
        self.n_scf += 1
        self.results['energy'] = energy_ha * HARTREE_TO_EV

        cycles = getattr(scf_engine, "cycles", None)
        self.last_scf = {"converged": bool(getattr(scf_engine, "converged", True)),
                         "cycles": int(cycles) if isinstance(cycles, (int, np.integer)) else None,
                         "conv_tol": float(self.conv_tol), "warm_start_used": warm_used, "warm_start_failed": warm_failed}
        if self.warm_start and self.fixed_spin == 0:
            try:
                WARM["dm"], WARM["mol"] = _to_numpy(scf_engine.make_rdm1()), cpu_mol
            except Exception:
                WARM["dm"], WARM["mol"] = None, None

        if 'forces' in properties:
            grad_analyzer = scf_engine.nuc_grad_method()
            if self.grid_response:
                try: grad_analyzer.grid_response = True
                except Exception: pass
            forces_ha_per_bohr = -_to_numpy(grad_analyzer.kernel())
            # v7 convention: Eh/Bohr x 27.211386 (so "eV/A" thresholds are really eV/Bohr). FORCE_UNIT_FIX converts properly.
            self.results['forces'] = forces_ha_per_bohr * HARTREE_TO_EV / (BOHR_TO_ANG if FORCE_UNIT_FIX else 1.0)
            self.n_gradients += 1
        self.last_scf["wall_s"] = round(time.perf_counter() - t0, 2)

        del gpu_mol, scf_engine
        torch.cuda.synchronize(); cp.cuda.Stream.null.synchronize()
        gc.collect(); torch.cuda.empty_cache(); cp.get_default_memory_pool().free_all_blocks()

class StandInCalculator(Calculator):
    """CPU dry-run stand-in: GFN2-xTB. In the second basis it adds a small bond-length shift so the basis switch changes
    the forces, as it does with DFT. It exists only to test the notebook's wiring."""
    implemented_properties = ['energy', 'forces']

    def __init__(self, basis_set="def2-svp", conv_tol=1e-5, warm_start=False, **kw):
        super().__init__(**kw)
        from tblite.ase import TBLite
        self.basis_set, self.conv_tol, self.warm_start = basis_set, conv_tol, warm_start
        self.tb = TBLite(method="GFN2-xTB", verbosity=0)
        self.n_scf = self.n_gradients = 0
        self.last_scf = {}
        self.nao = MOL["nao"].get(basis_set)

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        t0 = time.perf_counter()
        a = self.atoms.copy(); a.calc = self.tb
        e, f = a.get_potential_energy(), a.get_forces()
        if self.basis_set != BASIS_PATH[0]:
            k, shift = 15.0, 0.012
            pos = self.atoms.positions
            for i, j in BOND_PAIRS:
                d = pos[j] - pos[i]; r = np.linalg.norm(d); u = d / r
                e += 0.5 * k * (r - REF_BOND_LEN[(i, j)] - shift) ** 2
                fij = -k * (r - REF_BOND_LEN[(i, j)] - shift) * u
                f[j] += fij; f[i] -= fij
        self.n_scf += 1; self.n_gradients += 1
        self.results['energy'] = float(e); self.results['forces'] = np.array(f)
        self.last_scf = {"converged": True, "cycles": 9, "conv_tol": float(self.conv_tol), "warm_start_used": False,
                         "warm_start_failed": False, "wall_s": round(time.perf_counter() - t0, 3)}

def make_calculator(basis_set, conv_tol, grid_level=3, warm_start=False, grid_response=False, align=None, rho_cut=None):
    if BACKEND == "skala":
        return NativeGPUSkalaCalculator(fixed_spin=0, fixed_charge=0, basis_set=basis_set, grid_level=grid_level,
                                        conv_tol=conv_tol, verbose=0, warm_start=warm_start, grid_response=grid_response, align=align, rho_cut=rho_cut)
    return StandInCalculator(basis_set=basis_set, conv_tol=conv_tol, warm_start=warm_start)

print("Calculator ready. Force convention:", "eV/Angstrom (corrected)" if FORCE_UNIT_FIX else "v7 convention (Eh/Bohr x 27.211386, labelled eV/A)")
print("Spherical d enforced: cart=False in every PySCF molecule." if BACKEND == "skala" else "(stand-in backend: no PySCF molecule is built)")

In [ ]:
# @title 5a. The nudged starting geometries (fixed files, identical for every arm) and the reference minimum
# Each start = xTB reference minimum + a designed deformation (see the design document) + a seeded 0.01 A RMS nudge off symmetry.
# Both molecules are stored here; MOLECULE (settings cell) picks one.
START_XYZ_ALL = {
    "C10H6": {
        "R1": """16
R1 nudged 0.01 A RMS
C      0.01667481     2.58581684     0.00893752
C      0.16408488     1.96530402    -1.25120173
C     -0.20243103     1.96459638     1.23554465
C      0.21720824     0.62015171    -1.40242668
C     -0.05812620     0.60531833     1.38913247
C     -0.02945451    -0.62012914    -1.39566565
C     -0.08898894    -0.61761796     1.40082829
C     -0.27136936    -1.96321037    -1.25308250
C     -0.00262955    -2.58552736    -0.00047070
C      0.26232013    -1.95954609     1.25018045
H     -0.14254450     3.68139117     0.00672302
H      0.13041999     2.58569334    -2.12647346
H     -0.28304936     2.57737773     2.12872597
H     -0.87087456    -2.59430989    -2.13226086
H      0.12339595    -3.67038317     0.00331989
H      0.89634812    -2.57583416     2.13778713""",
        "R2": """16
R2 nudged 0.01 A RMS
C      0.00473147     2.47069396    -0.11244337
C      0.00630717     1.91022048    -1.35438850
C     -0.00059966     1.89370550     1.18906398
C      0.00273350     0.67144643    -1.07135519
C     -0.00240968     0.56352965     1.47454292
C      0.00454626    -0.55231130    -1.78627965
C      0.00423860    -0.69291885     1.58017656
C     -0.00435604    -1.85465457    -1.18807904
C     -0.00256428    -2.36660106    -0.01910242
C     -0.00585583    -2.00647977     1.33848079
H     -0.00261880     3.60411022     0.00637445
H     -0.00438251     2.22854582    -2.47145292
H     -0.00752003     2.63616619     1.99230793
H      0.01264893    -2.49812518    -2.10532157
H     -0.00627602    -3.45283209    -0.01103014
H     -0.00000839    -2.92510664     2.03719208""",
        "R3": """16
R3 nudged 0.01 A RMS
C     -0.11063324     2.70690281     0.05890061
C     -0.26126091     2.08202292    -1.20128936
C      0.17947977     1.90488046     1.30281681
C      0.07480673     0.76426349    -1.64398885
C      0.17000338     0.56451165     1.07144581
C      0.17484988    -0.49872356    -1.14860792
C     -0.02639288    -0.66372190     1.09620679
C      0.19027650    -2.00250821    -1.07852089
C     -0.18392295    -2.81044215     0.17009018
C     -0.16116182    -2.10220272     1.32515658
H     -0.21850176     3.84221777     0.17670846
H     -0.44282399     2.74652160    -2.08765565
H      0.33123310     2.39519397     2.27387730
H      0.31332992    -2.44499178    -2.04945237
H     -0.37945620    -3.86793452    -0.04910062
H     -0.29251604    -2.27835585     2.49065580""",
        "R4": """16
R4 nudged 0.01 A RMS
C      0.12903840     2.54710601    -0.06977212
C     -0.07972623     1.88502558    -1.20183637
C      0.41592979     1.98675324     1.17865804
C     -0.20884107     0.56269712    -1.19889605
C     -0.30580550     0.61497606     1.67891660
C     -0.21615942    -0.64496196    -1.70621518
C     -0.26063626    -0.64084816     1.41870632
C      0.15785893    -1.95049390    -1.30590160
C      0.34017864    -2.49158852    -0.02006460
C     -0.12537275    -1.86154901     1.26519420
H      0.26351977     3.67279187     0.05599027
H      0.12400467     2.37203212    -2.20646441
H      0.53814584     2.73782088     1.95958286
H      0.41772158    -2.67614310    -2.10806281
H      0.65252042    -3.56935777    -0.11642957
H     -0.32249950    -2.55073214     2.08300763""",
        "R5": """16
R5 nudged 0.01 A RMS
C     -0.03679687     2.47054427     0.09425548
C     -0.00867543     1.95972152    -1.32526458
C     -0.00629990     1.90179774     1.22323710
C     -0.02477864     0.70878074    -1.81038397
C      0.21586187     0.49989200     1.20596909
C     -0.09472952    -0.52704697    -0.98290958
C      0.02075303    -0.67279676     1.43552067
C      0.14541367    -1.89241565    -1.18766031
C     -0.03637305    -2.53428652     0.02692438
C     -0.10013380    -1.95906973     1.31321907
H     -0.16599888     3.51471221     0.02071254
H      0.08500205     3.00317926    -1.91106411
H     -0.31404540     2.42116538     2.15553556
H      0.17195322    -2.31798396    -2.21250755
H     -0.02882652    -3.57972475    -0.05339523
H     -0.50080523    -2.62577978     2.12392674""",
        "Q1": """16
Q1 nudged 0.01 A RMS
C     -0.00291133     2.58638922    -0.00196489
C      0.00034379     1.96929097    -1.09554694
C     -0.00180309     1.97482352     1.08292395
C     -0.00246722     0.61787955    -1.22787305
C     -0.00161848     0.60271905     1.21645539
C      0.00798946    -0.60699042    -1.22313479
C     -0.00296756    -0.62574564     1.22659474
C      0.00277473    -1.97811767    -1.08916662
C     -0.00144725    -2.58287889     0.00450472
C      0.00548785    -1.96823111     1.09132718
H     -0.00380443     3.67817120     0.00485302
H      0.00143175     2.58206481    -1.87490468
H      0.00722567     2.59065528     1.88574238
H     -0.00895850    -2.58525806    -1.88428647
H      0.00031725    -3.67080429    -0.00020942
H      0.00040737    -2.58396732     1.88448207""",
        "P0": """16
P0 nudged 0.01 A RMS
C      0.00663695     2.57592914     0.00763305
C      0.01078728     1.96914448    -1.23088784
C     -0.00397375     1.95305739     1.24179189
C      0.00831811     0.62380642    -1.39379506
C     -0.01140277     0.62246988     1.39063883
C      0.00029495    -0.61764689    -1.39095838
C      0.00333591    -0.61873672     1.39448428
C     -0.00376038    -1.95651138    -1.25533445
C     -0.00104727    -2.58994667     0.00038280
C      0.00275034    -1.96976140     1.24092248
H     -0.00593254     3.66593399    -0.00061027
H     -0.00219599     2.59022020    -2.13512372
H     -0.00257632     2.58462455     2.12662179
H     -0.00152136    -2.58529083    -2.12765058
H     -0.00061948    -3.66840644     0.00049386
H      0.00090632    -2.57888551     2.13116741""",
    },
    "C11H8": {
        "R1": """19
R1 nudged 0.01 A RMS
C      0.26736258    -0.07823611     2.59802996
C     -0.16213986     1.19425529     1.97620600
C     -0.30070146    -1.29854857     1.96349624
C      0.05089594     1.60568766     0.67976476
C      0.11608982    -1.42627043     0.60643633
C      0.03598610     1.59002333    -0.69041015
C      0.09786454    -1.41903931    -0.60356627
C     -0.20076426     1.19836222    -1.99901557
C     -0.23195684    -1.29603290    -1.97098574
C      0.28844558    -0.06895140    -2.58021944
C      0.00856234     2.93335130     0.00844144
H      0.50683833    -0.06935736     3.67676651
H     -0.22788685     2.01101824     2.69180258
H      0.20850963    -2.21006152     2.56698255
H     -0.19208906     2.02287234    -2.69230208
H     -0.31485085    -2.19227932    -2.56173408
H      0.37811623    -0.07400316    -3.66872106
H     -0.92157839     3.48799372    -0.00799716
H      0.89441471     3.54554278     0.00854657""",
        "R2": """19
R2 nudged 0.01 A RMS
C      0.00425155     0.02818062     2.87265200
C      0.00612658     1.20245691     1.95801603
C     -0.00116141    -1.11069180     2.01569383
C      0.00223350     1.31226214     0.64740643
C     -0.00335493    -1.25180420     0.57945191
C      0.00483044     1.50622754    -0.79094383
C      0.00268322    -1.30061242    -0.61361463
C     -0.00554255     1.05000887    -2.07076728
C     -0.00301861    -1.18504596    -2.00579257
C     -0.00608703    -0.04548511    -2.88471984
C     -0.00352419     2.73921450     0.26660524
H     -0.00537164    -0.22512949     3.85977955
H     -0.00892736     2.33233807     2.35377008
H      0.01083595    -2.18194587     2.47293972
H     -0.00552117     1.93896420    -2.76824613
H      0.00051525    -2.18779193    -2.53112941
H      0.00492632    -0.04303970    -3.92316224
H     -0.90838951     3.32466635     0.40120099
H      0.91343801     3.32163102     0.40871321""",
        "R3": """19
R3 nudged 0.01 A RMS
C      0.16948822    -0.15439846     2.39524178
C      0.20051619     1.23361803     2.01638873
C      0.03260171    -1.41737391     1.86884268
C     -0.03535557     1.96264202     0.79583062
C     -0.25825564    -1.39274643     0.50022192
C     -0.08222997     1.59348008    -0.48340468
C     -0.25399720    -1.37233441    -0.71567105
C      0.21406061     1.06667732    -1.77329329
C      0.01162929    -1.58915911    -2.04911625
C      0.16104830    -0.18299033    -2.41746447
C     -0.25844985     3.21138278    -0.09105041
H      0.31847062    -0.30229166     3.44565466
H      0.53571091     1.91457249     2.96747835
H      0.12047464    -2.41534128     2.31355338
H      0.44060383     1.90683762    -2.51037482
H      0.14418840    -2.60705025    -2.44913875
H      0.30660045    -0.22330425    -3.53679339
H     -1.31779947     3.56392199    -0.47715009
H      0.48541909     4.02555826    -0.09805287""",
        "R4": """19
R4 nudged 0.01 A RMS
C     -0.47698487    -0.04107565     2.68573979
C     -0.24977246     1.21527056     1.96410203
C      0.10144050    -1.22848115     2.10192328
C      0.08050565     1.18865759     0.51087895
C      0.34315802    -1.46338164     0.67919685
C      0.06033805     1.67953147    -0.82236727
C      0.33975616    -1.26904207    -0.54830266
C     -0.24056769     1.26451892    -2.09648111
C      0.06743797    -1.21828429    -1.97847123
C     -0.43407777    -0.05390550    -2.66801599
C      0.42454132     2.83557563     0.16721937
H     -0.51093034     0.03148746     3.69099012
H     -0.28330271     2.23475546     2.49170459
H      0.09364464    -2.17235240     2.70924520
H     -0.31113417     1.97623754    -2.95273465
H      0.16293434    -2.21397068    -2.49254794
H     -0.75222591     0.00286172    -3.75911753
H     -0.29571343     3.69195794     0.25089477
H      1.48000220     3.18211039     0.24614242""",
        "R5": """19
R5 nudged 0.01 A RMS
C      0.19816091    -0.11431049     2.38872487
C      0.07343806     1.11637528     1.84601094
C      0.17804446    -1.41879703     1.96353699
C     -0.19134011     1.76530147     0.74800874
C     -0.22021821    -1.67377159     0.65902044
C     -0.05930751     1.67760234    -0.73558048
C     -0.31450009    -1.48926643    -0.51390134
C      0.02405718     1.31046096    -1.87166378
C     -0.07643125    -1.23627700    -1.80944631
C      0.35299204    -0.01391945    -2.43625880
C     -0.10451135     2.99791183    -0.18035532
H      0.38299546    -0.18229414     3.49092482
H      0.15327608     1.79274223     2.72784186
H      0.11944635    -2.22467015     2.64951921
H      0.03689463     2.15009885    -2.57324487
H      0.02743805    -2.09372869    -2.49635764
H      0.92774792    -0.11045619    -3.49370145
H     -0.97723945     3.62889853    -0.50427530
H      0.90452669     3.55572489    -0.29453331""",
        "Q1": """19
Q1 nudged 0.01 A RMS
C     -0.00281578    -0.07896204     2.27569640
C      0.00061713     1.19854176     1.74608085
C     -0.00162837    -1.28997718     1.72315892
C     -0.00236445     1.59999686     0.60525555
C     -0.00141540    -1.43010075     0.52697825
C      0.00874752     1.60512795    -0.60332345
C     -0.00286866    -1.42880197    -0.53676135
C      0.00323772     1.18037980    -1.74381078
C     -0.00129710    -1.29344900    -1.72909350
C      0.00607612    -0.07915881    -2.27956375
C     -0.00384055     2.92935865     0.00530289
H      0.00174738    -0.07629620     3.23145465
H      0.00786722     2.02486161     2.38288222
H     -0.00923198    -2.20317807     2.25091145
H      0.00064164     2.02004398    -2.37183065
H      0.00060012    -2.20179426    -2.25045469
H     -0.00069873    -0.07069788    -3.23599381
H     -0.90701477     3.52540940     0.00601638
H      0.90305543     3.52443782    -0.00282696""",
        "P0": """19
P0 nudged 0.01 A RMS
C      0.00657870    -0.08852991     2.59516923
C      0.01058436     1.19832209     1.99741938
C     -0.00368287    -1.31163371     1.96943891
C      0.00818788     1.60576210     0.68881858
C     -0.01085974    -1.40957805     0.60728437
C      0.00045860     1.59439672    -0.68752089
C      0.00338858    -1.42073072    -0.60998596
C     -0.00345712     1.20277757    -1.99971774
C     -0.00089431    -1.30057432    -1.97053360
C      0.00281373    -0.08006041    -2.59006207
C     -0.00567297     2.91726446    -0.00100953
H     -0.00201054    -0.06785825     3.66704935
H     -0.00239359     2.01878483     2.69039909
H     -0.00133543    -2.20286319     2.57132077
H     -0.00040952     2.02254202    -2.69513739
H      0.00093114    -2.19664266    -2.56883811
H      0.00356901    -0.08085625    -3.67084774
H     -0.90845370     3.52366686    -0.00628043
H      0.90207228     3.53155250     0.01311239""",
    },
}
REF_XYZ_ALL = {
    "C10H6": """16
xTB planar minimum
C     -0.00000000     2.58929588    -0.00001371
C     -0.00000000     1.96512017    -1.24238638
C      0.00000000     1.96511997     1.24235860
C      0.00000000     0.61485239    -1.39350530
C      0.00000000     0.61485166     1.39347572
C     -0.00000000    -0.61485138    -1.39351049
C      0.00000000    -0.61485186     1.39347658
C     -0.00000000    -1.96511964    -1.24238696
C      0.00000000    -2.58929521    -0.00001322
C      0.00000000    -1.96511954     1.24235918
H     -0.00000000     3.67216736    -0.00001383
H      0.00000000     2.58473131    -2.13112118
H     -0.00000000     2.58472838     2.13109593
H     -0.00000000    -2.58473280    -2.13112072
H      0.00000000    -3.67216764    -0.00001377
H      0.00000000    -2.58472905     2.13109573""",
    "C11H8": """19
xTB minimum of the MACE structure
C     -0.00000560    -0.07539459     2.58821923
C     -0.00001920     1.19461564     1.98673922
C      0.00000841    -1.29976135     1.97044296
C     -0.00002449     1.59728160     0.68955443
C      0.00002592    -1.41676489     0.61048688
C      0.00001599     1.59729465    -0.68953716
C      0.00000106    -1.41677786    -0.61050658
C      0.00002753     1.19463196    -1.98672397
C     -0.00003709    -1.29975269    -1.97046185
C     -0.00000671    -0.07537438    -2.58821549
C     -0.00008477     2.92349172     0.00002334
H     -0.00004087    -0.07298306     3.67138069
H     -0.00005561     2.01907613     2.69518713
H     -0.00001909    -2.20213205     2.56841549
H      0.00003342     2.01909038    -2.69517377
H     -0.00010352    -2.20211096    -2.56845231
H     -0.00003167    -0.07294863    -3.67137621
H     -0.90448942     3.52205570    -0.00000737
H      0.90422020     3.52220436     0.00008396""",
}
# ---- 5-fluorouracil: RDKit/MMFF starts (structures/5fu_test), nudged 0.01 A RMS (seeded), reference = GFN2-xTB relaxed minimum ----
START_XYZ_ALL.update({
    "5FU_T1": {
        "R1": """12
R1 5FU_T1 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.21135433    -1.44035682    -0.31082074
C     -1.18157395    -0.76922219    -0.16670623
N     -1.21444601     0.58863803     0.07508180
C     -0.09973349     1.34825559     0.24399939
C      1.10246342     0.80627658     0.17157524
F      2.22767551     1.52653718     0.34160482
C      1.24715002    -0.65057106    -0.08191088
O      2.34232794    -1.20180962    -0.15197678
N      0.05657698    -1.31940591    -0.23150766
H     -2.14144080     1.00504867     0.12406213
H     -0.25883006     2.40314987     0.41746080
H      0.12050333    -2.31781121    -0.40235773""",
        "R3": """12
R3 5FU_T1 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.20093001    -1.43608433    -0.31061609
C     -1.19227106    -0.76163479    -0.17726554
N     -1.22782155     0.57943334     0.05962723
C     -0.09887639     1.34454712     0.23149418
C      1.10851118     0.81219416     0.16954533
F      2.24815249     1.52701245     0.33090887
C      1.25149293    -0.64459341    -0.08719900
O      2.33458139    -1.20196773    -0.14332507
N      0.06583388    -1.31957948    -0.24645597
H     -2.15034451     0.99663907     0.12988186
H     -0.24099351     2.41175846     0.42644104
H      0.11848708    -2.31744736    -0.40953126""",
        "P0": """12
P0 5FU_T1 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.21392465    -1.42733487    -0.30947659
C     -1.18243417    -0.77327857    -0.16231107
N     -1.21971742     0.58698971     0.07563247
C     -0.09691485     1.36651415     0.23498168
C      1.11188753     0.81865304     0.16596478
F      2.23921523     1.52301051     0.33237296
C      1.25188713    -0.65547221    -0.08736632
O      2.32915246    -1.20426884    -0.15609886
N      0.07348995    -1.32580634    -0.22390949
H     -2.15131773     0.99842195     0.12048326
H     -0.24965383     2.40433974     0.42555912
H      0.12117346    -2.31201768    -0.40588712""",
        "D2": """12
D2 5FU_T1 deliberately deformed start: xTB mode mixing, RMS shift 0.41 A from the xTB minimum, 6.0 eV above it by GFN2-xTB
O     -2.31438523    -1.38792103    -0.72260569
C     -1.24065063    -0.93577996     0.04222372
N     -1.08846736     0.51773590     0.10604974
C     -0.10714194     1.44136703     0.03193246
C      1.06248840     0.92672081     0.25360221
F      2.36311439     1.30345049     0.39243210
C      1.19692436    -0.47088036    -0.32455301
O      2.22894685    -0.92011407    -0.71372322
N     -0.02335550    -1.41622414     0.46093618
H     -2.10956857     0.99708231    -0.30484553
H     -0.18177241     2.38675094     0.82917165
H      0.21424001    -2.44217029    -0.05183105""",
    },
    "5FU_T3": {
        "R1": """12
R1 5FU_T3 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.36356833    -1.67401582    -0.39665474
C     -1.33622495    -1.00039519    -0.23509423
N     -0.06667401    -1.53043697    -0.28008820
C      0.91191051    -0.73601341    -0.09141661
O      2.16332042    -1.19778842    -0.13274776
C      0.81637151     0.70683518     0.17208282
F      1.95146502     1.40749194     0.33756912
C     -0.40261906     1.20299738     0.20319322
N     -1.47525802     0.36092009     0.00847834
H      2.84055820    -0.49235133     0.02751213
H     -0.62937106     2.24005787     0.37973080
H     -2.42059267     0.69142679     0.03594027""",
        "R3": """12
R3 5FU_T3 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.35314401    -1.66974333    -0.39645009
C     -1.34692206    -0.99280779    -0.24565354
N     -0.08004955    -1.53964166    -0.29554277
C      0.91276761    -0.73972188    -0.10392182
O      2.16936818    -1.19187084    -0.13477767
C      0.83684849     0.70731045     0.16138687
F      1.95580793     1.41346959     0.33228100
C     -0.41036561     1.20283927     0.21184493
N     -1.46600112     0.36074652    -0.00646997
H      2.83165449    -0.50076093     0.03333186
H     -0.61153451     2.24866646     0.38871104
H     -2.42260892     0.69179064     0.02876674""",
        "P0": """12
P0 5FU_T3 RDKit/MMFF start, nudged 0.01 A RMS
O     -2.36613865    -1.66099387    -0.39531059
C     -1.33708517    -1.00445157    -0.23069907
N     -0.07194542    -1.53208529    -0.27953753
C      0.91472915    -0.71775485    -0.10043432
O      2.17274453    -1.18541196    -0.13835822
C      0.82791123     0.70330851     0.16285096
F      1.95620213     1.40259079     0.33211368
C     -0.41579454     1.20053816     0.19907114
N     -1.45834505     0.35451966     0.01607651
H      2.83068127    -0.49897805     0.02393326
H     -0.62019483     2.24124774     0.38782912
H     -2.41992254     0.69722032     0.03241088""",
        "D2": """12
D2 5FU_T3 deliberately deformed start: xTB mode mixing, RMS shift 0.28 A from the xTB minimum, 6.0 eV above it by GFN2-xTB
O     -2.30673787    -1.63891037    -0.12760283
C     -1.21311942    -1.09921056    -0.42923146
N     -0.05643121    -1.56866553    -0.54585583
C      0.73458983    -0.79593763    -0.23065783
O      2.11447193    -1.13798042     0.12509233
C      0.88002945     0.57161279     0.12648632
F      2.03818442     1.27654499     0.29305355
C     -0.28953873     1.35237301     0.07387882
N     -1.58600575     0.39504352     0.16326001
H      2.75944404    -0.32775317     0.06292446
H     -0.42166228     2.50050216     0.60129664
H     -2.65287031     0.47224449    -0.11250321""",
    },
    "5FU_T3B": {
        "R1": """12
R1 5FU_T3B RDKit/MMFF start, nudged 0.01 A RMS
O     -2.36356833    -1.67401582    -0.39665474
C     -1.33622495    -1.00039519    -0.23509423
N     -0.06667401    -1.53043697    -0.28008820
C      0.91191051    -0.73601341    -0.09141661
O      2.16332042    -1.19778842    -0.13274776
C      0.81637151     0.70683518     0.17208282
F      1.95146502     1.40749194     0.33756912
C     -0.40261906     1.20299738     0.20319322
N     -1.47525802     0.36092009     0.00847834
H      2.22534320    -2.14256133    -0.29926487
H     -0.62937106     2.24005787     0.37973080
H     -2.42059267     0.69142679     0.03594027""",
        "R3": """12
R3 5FU_T3B RDKit/MMFF start, nudged 0.01 A RMS
O     -2.35314401    -1.66974333    -0.39645009
C     -1.34692206    -0.99280779    -0.24565354
N     -0.08004955    -1.53964166    -0.29554277
C      0.91276761    -0.73972188    -0.10392182
O      2.16936818    -1.19187084    -0.13477767
C      0.83684849     0.70731045     0.16138687
F      1.95580793     1.41346959     0.33228100
C     -0.41036561     1.20283927     0.21184493
N     -1.46600112     0.36074652    -0.00646997
H      2.21643949    -2.15097093    -0.29344514
H     -0.61153451     2.24866646     0.38871104
H     -2.42260892     0.69179064     0.02876674""",
        "P0": """12
P0 5FU_T3B RDKit/MMFF start, nudged 0.01 A RMS
O     -2.36613865    -1.66099387    -0.39531059
C     -1.33708517    -1.00445157    -0.23069907
N     -0.07194542    -1.53208529    -0.27953753
C      0.91472915    -0.71775485    -0.10043432
O      2.17274453    -1.18541196    -0.13835822
C      0.82791123     0.70330851     0.16285096
F      1.95620213     1.40259079     0.33211368
C     -0.41579454     1.20053816     0.19907114
N     -1.45834505     0.35451966     0.01607651
H      2.21546627    -2.14918805    -0.30284374
H     -0.62019483     2.24124774     0.38782912
H     -2.41992254     0.69722032     0.03241088""",
        "D2": """12
D2 5FU_T3B deliberately deformed start: xTB mode mixing, RMS shift 0.41 A from the xTB minimum, 6.0 eV above it by GFN2-xTB
O     -2.28769932    -2.07786510     0.33209362
C     -1.51471622    -0.94212444    -0.10584723
N     -0.18708017    -1.30112444    -0.60512890
C      0.90220379    -0.70307465    -0.28146576
O      2.03301659    -1.17939050     0.01389998
C      0.89033926     0.79665034     0.07343484
F      1.85948365     1.35596557     0.85043975
C     -0.38960875     1.13914212    -0.20021921
N     -1.41574039     0.38633733    -0.14764311
H      2.32070647    -2.10180754    -0.50781060
H     -0.50259270     1.99870354     0.66495926
H     -2.32284128     0.97781969    -0.41228829""",
    },
})
REF_XYZ_ALL.update({
    "5FU_T1": """12
GFN2-xTB relaxed minimum of T1 (reference for bond graph and RMSD)
O     -2.19947280    -1.42742611    -0.31367452
C     -1.20059632    -0.77304655    -0.17013148
N     -1.23295432     0.58776510     0.07181717
C     -0.11229491     1.34656252     0.23726640
C      1.10977231     0.78740275     0.17025777
F      2.20609177     1.54239051     0.33437533
C      1.29466803    -0.64363730    -0.08013800
O      2.33573444    -1.24137520    -0.15888317
N      0.07073728    -1.29579331    -0.22932411
H     -2.14978390     1.00961069     0.12250725
H     -0.24550097     2.40165926     0.42196818
H      0.12359938    -2.29411035    -0.40604082""",
    "5FU_T3": """12
GFN2-xTB relaxed minimum of T3 (reference for bond graph and RMSD)
O     -2.32368883    -1.67910695    -0.40154828
C     -1.33479317    -1.01903371    -0.24062959
N     -0.06037972    -1.48148672    -0.27335090
C      0.96548670    -0.71372949    -0.09123202
O      2.17570366    -1.25192441    -0.14041997
C      0.80588625     0.69047278     0.15904571
F      1.90785607     1.45999242     0.34459042
C     -0.44552753     1.21017148     0.20316122
N     -1.48402042     0.38059440     0.00921941
H      2.85930039    -0.58049479     0.01015850
H     -0.63072195     2.25858116     0.38733193
H     -2.43510245     0.72596484     0.03367458""",
    "5FU_T3B": """12
GFN2-xTB relaxed minimum of T3B (reference for bond graph and RMSD)
O     -2.31378655    -1.69119485    -0.40335754
C     -1.32902183    -1.02278518    -0.24108218
N     -0.05134352    -1.48288099    -0.27323992
C      0.97779530    -0.71334454    -0.09066074
O      2.18866313    -1.24130188    -0.13794921
C      0.83233465     0.68792755     0.15965665
F      1.90257464     1.47819744     0.34770584
C     -0.42691241     1.19568735     0.20127042
N     -1.47069712     0.36930906     0.00769775
H      2.10967325    -2.19199401    -0.31500851
H     -0.61445229     2.24324979     0.38519089
H     -2.42004323     0.71892125     0.03300054""",
})
START_XYZ = START_XYZ_ALL[MOLECULE]
REF_XYZ = REF_XYZ_ALL[MOLECULE]
assert START_XYZ["R3"].split()[0] == str(MOL["n_atoms"]), "start files do not match the molecule"
print(f"{MOLECULE} starts:", ", ".join(START_XYZ))

In [ ]:
# @title 5b. Geometry helpers: bond graph, planarity, transannular distances, symmetry-aware RMSD
import numpy as np
from ase import Atoms
from ase.data import covalent_radii, atomic_numbers
from scipy.optimize import linear_sum_assignment

def parse_xyz(text):
    lines = text.strip().splitlines()
    n = int(lines[0]); rows = [l.split() for l in lines[2:2 + n]]
    return [r[0] for r in rows], np.array([[float(x) for x in r[1:4]] for r in rows])

SYMBOLS, REF_POS = parse_xyz(REF_XYZ)
N_ATOMS = len(SYMBOLS)

def start_atoms(name):
    sym, pos = parse_xyz(START_XYZ[name]); return Atoms(sym, positions=pos)

def _graph(pos):
    z = [atomic_numbers[s] for s in SYMBOLS]; pairs = set()
    for i in range(N_ATOMS):
        for j in range(i + 1, N_ATOMS):
            if np.linalg.norm(pos[i] - pos[j]) < 1.2 * (covalent_radii[z[i]] + covalent_radii[z[j]]):
                pairs.add((i, j))
    return pairs

BOND_GRAPH = _graph(REF_POS)
BOND_PAIRS = sorted(BOND_GRAPH)
REF_BOND_LEN = {(i, j): float(np.linalg.norm(REF_POS[i] - REF_POS[j])) for i, j in BOND_PAIRS}
_nbr = {i: set() for i in range(N_ATOMS)}
for i, j in BOND_PAIRS: _nbr[i].add(j); _nbr[j].add(i)
_C = [i for i, s in enumerate(SYMBOLS) if s == "C"]
_TRANS_PAIRS = [(i, j) for i in _C for j in _C if i < j and (i, j) not in BOND_GRAPH and not (_nbr[i] & _nbr[j])]

def same_bond_graph(pos): return _graph(pos) == BOND_GRAPH
def fmax_of(F): return float(np.linalg.norm(F, axis=1).max())
def min_pair_distance(pos):
    d = np.linalg.norm(pos[:, None] - pos[None], axis=-1); d[np.diag_indices(len(pos))] = 9; return float(d.min())
def plane_rms(pos):
    """RMS distance (A) of the atoms from their best-fit plane. Frame independent: a tilted flat molecule gives 0."""
    p = pos if MOL["plane_atoms"] is None else pos[MOL["plane_atoms"]]     # C11H8: the ten ring carbons only
    c = p - p.mean(0); return float(np.sqrt(max(np.linalg.eigvalsh(c.T @ c / len(c))[0], 0.0)))
def transannular(pos, k=2):
    return sorted(float(np.linalg.norm(pos[i] - pos[j])) for i, j in _TRANS_PAIRS)[:k]

def _kabsch(A, B):
    A0, B0 = A - A.mean(0), B - B.mean(0)
    U, S, Vt = np.linalg.svd(B0.T @ A0)
    d = np.sign(np.linalg.det(U @ Vt)); D = np.diag([1, 1, d])
    R = U @ D @ Vt
    return B0 @ R, A0

def rmsd_sym(A, B, allow_reflection=True):
    """Standard RMSD (sqrt of the mean squared atom distance) minimised over rotations, reflection and same-element atom
    permutations (the ring's symmetry). Note: v7's overlay divides by 3N, which is sqrt(3) smaller."""
    best = 1e9
    for sgn in ([1, -1] if allow_reflection else [1]):
        Bm = B * np.array([1, 1, sgn]); perm = np.arange(len(A))
        for _ in range(4 if MOL["permute_symmetry"] else 0):
            Bal, A0 = _kabsch(A, Bm[perm]); newperm = perm.copy()
            for el in set(SYMBOLS):
                idx = np.array([i for i, s in enumerate(SYMBOLS) if s == el])
                _, c = linear_sum_assignment(np.linalg.norm(A0[idx][:, None] - Bal[idx][None], axis=-1))
                newperm[idx] = perm[idx][c]
            if np.array_equal(newperm, perm): break
            perm = newperm
        Bal, A0 = _kabsch(A, Bm[perm])
        best = min(best, float(np.sqrt(((A0 - Bal) ** 2).sum(1).mean())))
    return best

def fingerprint(pos, energy=None):
    return {"plane_rms_A": round(plane_rms(pos), 4), "transannular_A": [round(x, 3) for x in transannular(pos)],
            "rmsd_to_reference_A": round(rmsd_sym(REF_POS, pos), 4), "bond_graph_ok": same_bond_graph(pos),
            "energy_eV": None if energy is None else float(energy)}

print(f"Reference {MOLECULE}: {N_ATOMS} atoms, {len(BOND_PAIRS)} bonds, planarity (ring atoms) {plane_rms(REF_POS):.4f} A; transannular C...C = {[round(x,2) for x in transannular(REF_POS)]} A")

In [ ]:
# @title 6. Settings per pass: expert-tuned (A0e), untuned random draw (A0r), cautious novice (A0c)
from dataclasses import dataclass, asdict, replace

@dataclass
class PassSettings:
    basis: str
    maxstep: float          # A, largest atomic move per step
    steps: int              # step budget of the pass
    fmax: float             # pass ends when the largest atomic force is below this
    conv_tol: float         # SCF energy tolerance (Eh)
    grid_level: int = 3

# expert_hand_tuned_custom_runtime_settings (passes 1-3) and the JSON run's settings (passes 4-6): hand-tuned from observation
EXPERT_HAND_TUNED_CUSTOM_RUNTIME_SETTINGS = [(0.025, 50, 0.10, 1e-4), (0.010, 100, 0.05, 1e-5), (0.001, 100, 0.01, 1e-6),
          (0.010, 100, 0.10, 1e-6), (0.006, 100, 0.03, 1e-6), (0.001, 100, 0.02, 1e-6)]

def _with_basis(rows):
    b = BASIS_PATH
    return [PassSettings(b[0] if i < 3 else b[1], ms, st, fm, ct) for i, (ms, st, fm, ct) in enumerate(rows)]

# Deliberately poor, written down before any run: stress test only (arm A0p / A3p), never a baseline claim.
POOR_PRACTICE_CUSTOM_RUNTIME_SETTINGS = [(0.002, 60, 0.10, 1e-4), (0.050, 40, 0.02, 1e-3), (0.020, 60, 0.005, 1e-3),
                                         (0.002, 60, 0.10, 1e-5), (0.050, 40, 0.01, 1e-3), (0.010, 100, 0.02, 1e-5)]

def expert_schedule():   return _with_basis(EXPERT_HAND_TUNED_CUSTOM_RUNTIME_SETTINGS)
def poor_schedule():     return _with_basis(POOR_PRACTICE_CUSTOM_RUNTIME_SETTINGS)
def cautious_schedule(): return _with_basis([(0.005, 200, ENDPOINT_FMAX, 1e-6)] * 6)

# Random ranges, written down BEFORE any Skala run and never re-rolled after seeing an outcome
RANDOM_RANGES = dict(maxstep_A=(0.002, 0.05), steps=[25, 50, 100, 150, 200], fmax_passes_1_to_5=(0.01, 0.2),
                     pass6_fmax=ENDPOINT_FMAX, conv_tol=(1e-8, 1e-4))
def random_schedule(seed):   # = random_custom_runtime_settings
    r = np.random.default_rng(seed)
    ms = np.exp(r.uniform(np.log(0.002), np.log(0.05), 6))                       # log-uniform step caps
    st = r.choice(RANDOM_RANGES["steps"], 6)                                       # step budgets
    fm = np.sort(np.exp(r.uniform(np.log(0.01), np.log(0.2), 5)))[::-1]            # fmax falls (never rises)
    fm = np.append(fm, ENDPOINT_FMAX)                                              # same final criterion for every arm
    ct = np.sort(np.exp(r.uniform(np.log(1e-8), np.log(1e-4), 6)))[::-1]           # SCF tolerance tightens
    r2 = lambda x, n: float(f"{x:.{n}g}")                                        # readable numbers: 2 significant digits
    return _with_basis([(r2(a, 2), int(b), r2(c, 2), r2(d, 1)) for a, b, c, d in zip(ms, st, fm, ct)])

ARM_SPEC = {
    "A0e": dict(label="Expert-tuned baseline", schedule="expert",   memory=1,            warm=False, agent=None,     explore=False),
    "A0r": dict(label="Untuned baseline",      schedule="random",   memory=1,            warm=False, agent=None,     explore=False),
    "A0c": dict(label="Cautious novice",       schedule="cautious", memory=1,            warm=False, agent=None,     explore=False),
    "A0p": dict(label="Poor-practice stress test", schedule="poor",     memory=1,            warm=False, agent=None,     explore=False),
    "A3p": dict(label="Gemini agent from poor setup", schedule="poor",  memory=FIXES_MEMORY, warm=True,  agent="gemini", explore=True),
    "AF":  dict(label="Fault injection (safety test)", schedule="expert", memory=FIXES_MEMORY, warm=True, agent="faulty", explore=True),
    "A1":  dict(label="Baseline plus fixes",   schedule="random",   memory=FIXES_MEMORY, warm=True,  agent=None,     explore=False),
    "A2":  dict(label="Rule-adaptive",         schedule="random",   memory=FIXES_MEMORY, warm=True,  agent="rules",  explore=True),
    "A3":  dict(label="Gemini agent",          schedule="random",   memory=FIXES_MEMORY, warm=True,  agent="gemini", explore=True),
}
START_INDEX = {n: i for i, n in enumerate(["R1", "R2", "R3", "R4", "R5", "Q1", "P0"])}
RANDOM_SEED_BASE = 5000

def schedule_for(arm, start):
    kind = ARM_SPEC[arm]["schedule"]
    sched = {"expert": expert_schedule, "cautious": cautious_schedule, "poor": poor_schedule}.get(kind, lambda: random_schedule(RANDOM_SEED_BASE + START_INDEX[start]))()
    if SMOKE_TEST:
        sched = [replace(s, steps=min(s.steps, 4)) for s in sched]
    if arm == "AF":      # fault-injection test: short passes, all six pass starts must be reached
        sched = [replace(s, steps=min(s.steps, 10)) for s in sched]
    return sched

print("Random baseline ranges (fixed before any Skala run):", RANDOM_RANGES)
for _s in STARTS[:1]:
    print(f"Untuned draw for start {_s} (seed {RANDOM_SEED_BASE + START_INDEX[_s]}):")
    for i, p in enumerate(random_schedule(RANDOM_SEED_BASE + START_INDEX[_s]), 1):
        print(f"   pass {i}: {p.basis:12s} maxstep {p.maxstep:.4f} A | steps {p.steps:3d} | fmax {p.fmax:.3f} | conv_tol {p.conv_tol:.1e}")
# ---- Part B: step-cap sweep arms (all with L-BFGS memory FIXES_MEMORY = 20 and the density warm start, like the earlier agent arms) ----
# A cell = one starting cap. P = plain optimizer (no agent, a memory-20 + warm-start control), G = Gemini agent (same package as A3/A3p), G2 = second agent repeat.
SWEEP_CAPS = {"S": 0.005, "M": 0.025, "B": 0.3}     # too small | reasonable | too big (A, largest atomic move per step), the same cap in all six passes
for _c, _cap in SWEEP_CAPS.items():
    _sched = f"cap:{_cap}"
    ARM_SPEC[f"P{_c}"]  = dict(label=f"Plain L-BFGS, cap {_cap} A, memory {FIXES_MEMORY}, warm start", schedule=_sched, memory=FIXES_MEMORY, warm=True, agent=None,     explore=False)
    ARM_SPEC[f"G{_c}"]  = dict(label=f"Gemini agent from cap {_cap} A",                                  schedule=_sched, memory=FIXES_MEMORY, warm=True, agent="gemini", explore=True)
    ARM_SPEC[f"G{_c}2"] = dict(label=f"Gemini agent from cap {_cap} A (repeat)",                         schedule=_sched, memory=FIXES_MEMORY, warm=True, agent="gemini", explore=True)
_schedule_for_old = schedule_for
def schedule_for(arm, start):
    kind = ARM_SPEC[arm]["schedule"]
    if isinstance(kind, str) and kind.startswith("cap:"):
        sched = _with_basis([(float(kind[4:]), 200, ENDPOINT_FMAX, 1e-6)] * 6)      # uniform cap, endpoint criterion in every pass (like the cautious novice, but with the swept cap)
        return [replace(s, steps=min(s.steps, 4)) for s in sched] if SMOKE_TEST else sched
    return _schedule_for_old(arm, start)
print("Sweep arms:", {k: v["label"] for k, v in ARM_SPEC.items() if k[0] in "PG" and k[1:2] in SWEEP_CAPS})

# ---- Arms of the hybrid-supervisor test (all memory 20 + warm start) ----
ARM_SPEC["HS"]  = dict(label="LLM + rules (hybrid) from cap 0.005 A",          schedule="cap:0.005", memory=FIXES_MEMORY, warm=True, agent="hybrid", explore=True)
ARM_SPEC["H3p"] = dict(label="LLM + rules (hybrid) from the poor schedule",     schedule="poor",      memory=FIXES_MEMORY, warm=True, agent="hybrid", explore=True)
ARM_SPEC["A2p"] = dict(label="Rules only from the poor schedule (no LLM)",      schedule="poor",      memory=FIXES_MEMORY, warm=True, agent="rules",  explore=True)
ARM_SPEC["RS"]  = dict(label="Rules controller from cap 0.005 A (no LLM)",      schedule="cap:0.005", memory=FIXES_MEMORY, warm=True, agent="rules",  explore=True)
ARM_SPEC["P1p"] = dict(label="Plain L-BFGS on the poor schedule, memory 20, warm start", schedule="poor", memory=FIXES_MEMORY, warm=True, agent=None, explore=False)
print("Hybrid-test arms:", {k: ARM_SPEC[k]["label"] for k in ("HS", "GS", "H3p", "A3p", "A2p", "P1p")})

## Logging, rate-limit-safe Gemini client, surrogate tools

In [ ]:
# @title 7. Run logger: readable JSON for every run (observer only, never changes the optimization)
import json, time, platform
from pathlib import Path

class RunLogger:
    """Writes, per run folder (flushed after every line, so a Colab disconnect loses nothing):
         run_meta.json    arm, start, requirement, settings, seeds, models, versions, GPU
         steps.jsonl      one line per optimizer step           (also steps.json, pretty-printed, at the end)
         events.jsonl     one line per agent-related event      (also events.json, pretty-printed, at the end)
         geometries.npy   positions after every step (for the optional energy-map figure)
         summary.json     final energy, gradients, minutes, curvature check, success flags (written by the orchestrator)
       API waiting (rate limits, back-off) is measured separately: t_opt_s = wall time minus API time."""

    def __init__(self, run_dir, meta):
        self.dir = Path(run_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.t0 = time.time(); self.grad = 0; self.scf = 0; self.api_s = 0.0; self._geoms = []
        meta = {**meta, "started": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version()}
        (self.dir / "run_meta.json").write_text(json.dumps(meta, indent=2, default=str))
        for f in ("steps.jsonl", "events.jsonl"):
            (self.dir / f).write_text("")
        self._fs = open(self.dir / "steps.jsonl", "a"); self._fe = open(self.dir / "events.jsonl", "a")

    @staticmethod
    def _w(f, rec):
        f.write(json.dumps(rec, default=float) + "\n"); f.flush()

    def t_wall(self): return time.time() - self.t0
    def t_opt(self):  return max(0.0, self.t_wall() - self.api_s)
    def add_api_time(self, seconds): self.api_s += float(seconds)

    def step(self, pass_id, step, energy_eV, fmax_eV_A, max_move_A, maxstep_A, fmax_target, conv_tol,
             basis=None, scf_cycles=None, positions=None, **extra):
        rec = dict(pass_id=pass_id, step=step, energy_eV=energy_eV, fmax=fmax_eV_A, max_move=max_move_A,
                   maxstep=maxstep_A, fmax_target=fmax_target, conv_tol=conv_tol, basis=basis, scf_cycles=scf_cycles,
                   gradients=self.grad, t_s=round(self.t_wall(), 3), t_opt_s=round(self.t_opt(), 3), **extra)
        self._w(self._fs, rec)
        if positions is not None: self._geoms.append(np.asarray(positions, float).copy())

    def event(self, kind, pass_id=None, step=None, **rec):
        """kind: pass_start | trigger | agent_call | validator | action_applied | probation | probe | rollback | pass_end | note"""
        rec.update(kind=kind, pass_id=pass_id, step=step, gradients=self.grad,
                   t_s=round(self.t_wall(), 3), t_opt_s=round(self.t_opt(), 3))
        self._w(self._fe, rec)

    def close(self):
        if self._geoms: np.save(self.dir / "geometries.npy", np.array(self._geoms))
        self._fs.close(); self._fe.close()
        for name in ("steps", "events"):                                      # human-readable copies
            rows = [json.loads(l) for l in open(self.dir / f"{name}.jsonl") if l.strip()]
            (self.dir / f"{name}.json").write_text(json.dumps(rows, indent=1, default=float))

def load_run(run_dir):
    d = Path(run_dir); rd = lambda n: [json.loads(l) for l in open(d / n) if l.strip()] if (d / n).exists() else []
    out = dict(dir=d, meta=json.loads((d / "run_meta.json").read_text()), steps=rd("steps.jsonl"), events=rd("events.jsonl"))
    if (d / "summary.json").exists(): out["summary"] = json.loads((d / "summary.json").read_text())
    return out
print("Logger ready.")

In [ ]:
# @title 8. Rate-limit-safe Gemini client: key rotation, model rotation, exponential back-off
import os, re, time, random, json
from collections import defaultdict

class ApiExhausted(Exception):
    """No Gemini answer within API_PATIENCE_S. The orchestrator flags the run 'degraded' (it is never counted silently).
    .elapsed_s = time spent inside the pool (excluded from the wall-clock metrics)."""
    elapsed_s = 0.0

class ApiBadRequest(Exception):
    """The request itself was rejected (schema/argument error). Retrying cannot help."""
    elapsed_s = 0.0

def _load_keys(secret_names):
    """Reads keys from Colab secrets, then environment variables. Keys are never printed; only 'key1', 'key2' labels."""
    keys = []
    for name in secret_names:
        val = None
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            pass
        val = val or os.environ.get(name)
        if val: keys.append((f"key{len(keys)+1}({name})", val))
    return keys

def _retry_after_s(err_text):
    m = re.search(r"retry(?:Delay)?\D{0,12}?([0-9]+(?:\.[0-9]+)?)\s*s", err_text, re.I)
    return float(m.group(1)) if m else None

def _classify(exc):
    """-> 'rate' | 'server' | 'dead_model' | 'bad_key' | 'bad_request' | 'other'"""
    txt = f"{type(exc).__name__} {exc}"
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    low = txt.lower()
    if code == 429 or "resource_exhausted" in low or "quota" in low or "rate limit" in low or "too many requests" in low: return "rate"
    if code in (500, 502, 503, 504) or "unavailable" in low or "overloaded" in low or "deadline" in low or "timed out" in low or "timeout" in low or "internal" in low: return "server"
    if code == 404 or "not_found" in low or "is not found" in low: return "dead_model"
    if code in (401, 403) or "api key" in low and "invalid" in low or "permission_denied" in low: return "bad_key"
    if code == 400 or "invalid_argument" in low: return "bad_request"
    return "other"

class GeminiPool:
    """One object that every agent call goes through.

    Policy (designed so a low-tier, shared, rate-limited account does not change WHAT the agent decides, only WHEN):
      1. A rate-limit on one key -> immediately try the same model on the other key.
      2. Both keys limited -> exponential back-off with jitter (honouring the server's retry-after hint).
      3. After API_MODEL_ROTATE_AFTER failed rounds on the preferred model -> rotate to the next model in the role's list,
         but return to the preferred model as soon as its cool-down ends.
      4. Past API_PATIENCE_S -> raise ApiExhausted (the run is flagged 'degraded' and re-run later if RERUN_DEGRADED).
    All waiting happens while the optimizer is paused, so the DFT trajectory is unaffected; the waiting time is logged and
    is subtracted from every wall-clock metric (t_opt_s), so a slow account does not penalise the agent arm."""

    ROLE_MODELS = {"fast": [MODEL_FAST, MODEL_PRO, MODEL_STRONG], "strong": [MODEL_STRONG, MODEL_PRO, MODEL_FAST]}

    def __init__(self, key_secrets=GEMINI_KEY_SECRETS, sleep=time.sleep, now=time.time, verbose=True):
        self.keys = _load_keys(key_secrets)
        if not self.keys: raise RuntimeError("No Gemini API key found. Add Colab secrets GEMINI_API_KEY (and GEMINI_API_KEY_2).")
        from google import genai
        self._genai = genai
        self.clients = {}
        self.cool = {}                         # (model, key_index) -> time when usable again
        self.dead_models, self.dead_keys = set(), set()
        self.sleep, self.now, self.verbose = sleep, now, verbose
        self.next_key = 0
        self.stats = defaultdict(float)        # calls, ok, rate_limited, server_errors, key_rotations, model_rotations, waited_s, latency_s
        if verbose: print(f"Gemini pool ready: {len(self.keys)} key(s) {[k[0] for k in self.keys]} | models fast={self.ROLE_MODELS['fast']}")

    def _client(self, ki):
        if ki not in self.clients: self.clients[ki] = self._genai.Client(api_key=self.keys[ki][1])
        return self.clients[ki]

    def _usable_keys(self, model):
        ks = [ki for ki in range(len(self.keys)) if ki not in self.dead_keys and self.cool.get((model, ki), 0) <= self.now()]
        return sorted(ks, key=lambda ki: (ki - self.next_key) % len(self.keys))     # alternate keys

    # ---- raw call log: every API call (prompt, answer, tokens, retries) goes to llm_calls.jsonl in the experiment folder ----
    @staticmethod
    def _ser_content(c):
        out = []
        for p in (getattr(c, "parts", None) or []):
            if getattr(p, "function_call", None): out.append(dict(function_call=p.function_call.name, args=dict(p.function_call.args or {})))
            elif getattr(p, "function_response", None): out.append(dict(function_response=p.function_response.name, response=p.function_response.response))
            elif getattr(p, "text", None): out.append(dict(text=p.text))
        return dict(role=getattr(c, "role", None), parts=out)

    def _log_call(self, role, contents, config, label, resp, meta, error=None):
        try:
            d = globals().get("EXPERIMENT_DIR")
            if not d: return
            os.makedirs(d, exist_ok=True)
            sp = os.path.join(d, "llm_system_prompt.txt")
            if not os.path.exists(sp) and getattr(config, "system_instruction", None): open(sp, "w").write(str(config.system_instruction))
            um = getattr(resp, "usage_metadata", None) if resp is not None else None
            rec = dict(t=time.strftime("%Y-%m-%d %H:%M:%S"), label=label, role=role, ok=error is None, error=error,
                       tokens=dict(prompt=getattr(um, "prompt_token_count", None), output=getattr(um, "candidates_token_count", None),
                                   thinking=getattr(um, "thoughts_token_count", None), total=getattr(um, "total_token_count", None)) if um else None,
                       meta={k: v for k, v in (meta or {}).items()},
                       prompt=[self._ser_content(c) for c in contents],
                       answer=self._ser_content(resp.candidates[0].content) if (resp is not None and resp.candidates) else None)
            with open(os.path.join(d, "llm_calls.jsonl"), "a") as f: f.write(json.dumps(rec, default=str) + "\n")
        except Exception as e:
            if self.verbose: print("   (llm log write skipped:", str(e)[:100], ")")

    def generate(self, role, contents, config, label=""):
        """Returns (response, meta); meta has model, key, latency, waited_s, elapsed_s (everything spent in here), rotations, attempts, errors.
        On failure raises ApiExhausted/ApiBadRequest carrying .elapsed_s."""
        t0_all = self.now()
        try:
            resp, meta = self._generate(role, contents, config, label)
            meta["elapsed_s"] = round(self.now() - t0_all, 2)
            self._log_call(role, contents, config, label, resp, meta); return resp, meta
        except (ApiExhausted, ApiBadRequest) as e:
            e.elapsed_s = self.now() - t0_all
            self._log_call(role, contents, config, label, None, dict(elapsed_s=round(e.elapsed_s, 1)), error=f"{type(e).__name__}: {str(e)[:300]}"); raise

    def _generate(self, role, contents, config, label=""):
        t_start, waited, attempts, backoff = self.now(), 0.0, 0, API_BACKOFF_BASE_S
        mfail = defaultdict(int)               # consecutive failed rounds per model within this decision
        rot_key = rot_model = 0
        meta = dict(role=role, errors=[])
        while True:
            if self.now() - t_start > API_PATIENCE_S:
                self.stats["exhausted"] += 1
                raise ApiExhausted(f"no answer within {API_PATIENCE_S}s; last errors: {meta['errors'][-3:]}")
            if len(self.dead_keys) == len(self.keys): raise ApiExhausted("every API key was rejected (invalid or no permission)")
            order = [m for m in self.ROLE_MODELS[role] if m not in self.dead_models]
            if not order: raise ApiExhausted("every model in the rotation reported 'not found'")
            # a short cool-down on the preferred model is waited out (so the decision comes from the same model whenever possible)
            pref = order[0]
            if mfail[pref] < API_MODEL_ROTATE_AFTER and not self._usable_keys(pref):
                soon = min((self.cool.get((pref, ki), 0) for ki in range(len(self.keys)) if ki not in self.dead_keys), default=self.now()) - self.now()
                if 0 < soon <= 30:
                    w = soon + 0.5
                    if self.verbose: print(f"   API: {pref} cooling for {soon:.0f}s -> waiting for it (same model keeps the decision comparable)")
                    self.sleep(w); waited += w; self.stats["waited_s"] += w
                    continue
            # stay on the preferred model until it has failed API_MODEL_ROTATE_AFTER rounds, then use the next one
            pick = next((m for m in order if self._usable_keys(m) and mfail[m] < API_MODEL_ROTATE_AFTER), None)
            if pick is not None:
                if pick != order[0] and mfail[order[0]] >= API_MODEL_ROTATE_AFTER:
                    rot_model += 1; self.stats["model_rotations"] += 1
                    if self.verbose: print(f"   API: {order[0]} stays limited -> using {pick} for this decision")
                round_ok = False
                for ki in self._usable_keys(pick):
                    attempts += 1; self.stats["calls"] += 1; t0 = self.now()
                    try:
                        resp = self._client(ki).models.generate_content(model=pick, contents=contents, config=config)
                        lat = self.now() - t0; self.stats["ok"] += 1; self.stats["latency_s"] += lat; self.next_key = ki
                        meta.update(model=pick, key=self.keys[ki][0], latency_s=round(lat, 2), waited_s=round(waited, 1), attempts=attempts,
                                    key_rotations=rot_key, model_rotations=rot_model, preferred_model=order[0], used_fallback_model=(pick != self.ROLE_MODELS[role][0]))
                        return resp, meta
                    except Exception as e:
                        kind = _classify(e); hint = _retry_after_s(str(e))
                        meta["errors"].append(f"{self.keys[ki][0]}/{pick}: {kind}")
                        if kind == "bad_request": raise ApiBadRequest(str(e)[:400])
                        if kind == "dead_model":
                            self.dead_models.add(pick); rot_model += 1; self.stats["model_rotations"] += 1
                            if self.verbose: print(f"   API: model {pick} is not available for this account -> rotating to the next model")
                            break
                        if kind == "bad_key":
                            self.dead_keys.add(ki); rot_key += 1; self.stats["key_rotations"] += 1
                            if self.verbose: print(f"   API: {self.keys[ki][0]} was rejected -> using the other key only")
                            continue
                        self.stats["rate_limited" if kind == "rate" else "server_errors"] += 1
                        self.cool[(pick, ki)] = self.now() + max(hint or 0.0, backoff if kind == "rate" else 3.0)
                        self.next_key = (ki + 1) % len(self.keys); rot_key += 1; self.stats["key_rotations"] += 1
                        if self.verbose: print(f"   API: {kind} limit on {self.keys[ki][0]} / {pick} -> trying the other key (server retry hint: {hint})")
                else:
                    mfail[pick] += 1                     # every usable key failed this round
                continue
            # nothing usable right now: wait (exponential back-off), then start a fresh cycle from the preferred model
            soonest = min((self.cool.get((m, ki), 0) for m in order for ki in range(len(self.keys)) if ki not in self.dead_keys), default=self.now())
            wait = min(API_BACKOFF_MAX_S, max(backoff, soonest - self.now())) * random.uniform(0.9, 1.1)
            if self.verbose: print(f"   API: all keys/models limited -> waiting {wait:.0f}s (optimizer paused; this time is excluded from the metrics)")
            self.sleep(wait); waited += wait; self.stats["waited_s"] += wait
            backoff = min(API_BACKOFF_MAX_S, backoff * 2); mfail.clear()

print("Rate-limit-safe Gemini client defined (keys are read only when the agent first runs).")

In [ ]:
# @title 9. Surrogate tools (GFN2-xTB): relaxation, soft modes, basin exploration, preconditioner
import numpy as np
from ase import Atoms
from ase.optimize import BFGS, LBFGS
from tblite.ase import TBLite

def xtb_atoms(pos):
    a = Atoms(SYMBOLS, positions=np.array(pos, float)); a.calc = TBLite(method="GFN2-xTB", verbosity=0); return a

def xtb_relax(pos, fmax=0.02, steps=400):
    a = xtb_atoms(pos); LBFGS(a, logfile=None, maxstep=0.1, memory=20).run(fmax=fmax, steps=steps)
    return a.positions.copy(), float(a.get_potential_energy())

def xtb_energy(pos): return float(xtb_atoms(pos).get_potential_energy())

def xtb_hessian(pos, delta=0.01):
    """Cartesian Hessian (eV/A^2) by central differences of xTB forces: 6 N cheap gradients (N atoms)."""
    a = xtb_atoms(pos); n3 = 3 * len(a); H = np.zeros((n3, n3)); x0 = a.positions.copy()
    for k in range(n3):
        for sgn in (+1, -1):
            x = x0.copy().reshape(-1); x[k] += sgn * delta; a.positions = x.reshape(-1, 3)
            H[k] += -sgn * a.get_forces().reshape(-1) / (2 * delta)
    return 0.5 * (H + H.T)

def surrogate_modes(pos, n_soft=6):
    """Softest vibrational modes of the xTB Hessian (the 6 rigid-body modes are removed). Each mode is a unit vector in 3N space."""
    H = xtb_hessian(pos); w, U = np.linalg.eigh(H)
    rigid = np.argsort(np.abs(w))[:6]; vib = [i for i in np.argsort(w) if i not in set(rigid)]
    normal = np.linalg.svd(pos - pos.mean(0))[2][2]                      # best-fit plane normal
    out = []
    for i in vib[:n_soft]:
        v = U[:, i].reshape(-1, 3); out.append(dict(index=int(i), curvature_eV_A2=float(w[i]), vec=U[:, i].copy(),
                                                    out_of_plane_share=float(((v @ normal) ** 2).sum() / (v ** 2).sum())))
    return out, H

def surrogate_preconditioner(pos, H=None):
    """Positive-definite starting Hessian from xTB: vibrational floor 0.3 eV/A^2, rigid-body modes 30 (tested on the proxy)."""
    H = xtb_hessian(pos) if H is None else H; w, U = np.linalg.eigh(H); idx = np.argsort(np.abs(w))
    lam = np.maximum(w, 0.3); lam[idx[:6]] = 30.0
    return (U * lam) @ U.T

def explore_basins(pos, amp_rms=0.15, seed=0, n_modes=6):
    """Surrogate search for basins near the start: relax the start with xTB, push +/- along the softest modes, relax each, cluster.
    Deterministic (the pushes are fixed vectors); seed only labels the call."""
    x_s, e_s = xtb_relax(pos)
    modes, _ = surrogate_modes(x_s, n_modes)
    seeds = [("start_relaxed", x_s)]
    for k, m in enumerate(modes):
        for sgn in (+1, -1):
            seeds.append((f"mode{k}{'+' if sgn > 0 else '-'}", x_s + sgn * amp_rms * np.sqrt(N_ATOMS) * m["vec"].reshape(-1, 3)))
    basins = []
    for label, x in seeds:
        try:
            xr, er = xtb_relax(x, fmax=0.01)
        except Exception:
            continue
        if not same_bond_graph(xr): continue
        for b in basins:
            if abs(b["E_xtb_eV"] - er) < 5e-4 and rmsd_sym(b["pos"], xr) < 0.05:
                b["n_seeds"] += 1; b["seeds"].append(label); break
        else:
            basins.append(dict(E_xtb_eV=er, pos=xr, n_seeds=1, seeds=[label], plane_rms_A=plane_rms(xr), rmsd_to_ref_A=rmsd_sym(REF_POS, xr)))
    basins.sort(key=lambda b: b["E_xtb_eV"])
    for i, b in enumerate(basins): b["id"] = i; b["rel_xtb_meV"] = 1000 * (b["E_xtb_eV"] - basins[0]["E_xtb_eV"])
    return basins, modes

print("Surrogate tools ready (GFN2-xTB via tblite).")

## The agent layer: monitor, six functions, validator, controllers

In [ ]:
# @title 10. Run state, per-step monitor, triggers, and the step_advisor function (all plain code, zero cost)
import numpy as np

class RunState:
    """Everything the orchestrator, the monitor and the six functions share for ONE run."""
    def __init__(self, arm, start, schedule, logger):
        self.arm, self.start, self.spec = arm, start, ARM_SPEC[arm]
        self.schedule = [replace(s) for s in schedule]          # mutable copy: agent actions edit this
        self.baseline_schedule = [replace(s) for s in schedule] # what the baseline would have used
        self.logger = logger
        self.atoms = None; self.calcs = []; self.p = 0; self.cfg = None
        self.recs, self.pass_recs = [], []
        self.step_total = 0
        self.optimizer_kind, self.memory = "lbfgs", self.spec["memory"]
        self.agent_enabled = self.spec["agent"] is not None
        self.agent_calls = 0; self.degraded_calls = 0; self.strikes = 0
        self.actions = []                                        # every proposed action and its fate
        self.cooldown = 0; self.probation = None
        self.basins = None; self.modes = None; self.current_basin = None; self.target_basin = None
        self.switched_basin = False
        self.surr_H = None; self.surr_pre = None
        self.final_curv = None; self.pending_geoms = {}; self.probe_cache = {}
        self.sec_per_grad = None; self.t_last_step = None
        self.event_name = None; self.api_log = []
        self.t_opt_end = None; self.eval_gradients = 0
        self.last_trigger = None; self.end_pass = False; self.rerun_last_pass = False; self.stop_flag = None; self.engine = None
        self.agent_errors = 0; self.max_agent_calls = 25; self.quiet = {}

    @property
    def gradients(self): return int(sum(c.n_gradients for c in self.calcs))
    @property
    def scf_total(self): return int(sum(c.n_scf for c in self.calcs))
    @property
    def forces(self): return self.atoms.get_forces()
    @property
    def target_fmax(self): return self.cfg.fmax
    def new_calculator(self, basis, conv_tol, grid_level=3):
        c = make_calculator(basis, conv_tol, grid_level, warm_start=(WARM_START and self.spec["warm"]))
        self.calcs.append(c); return c

def scf_noise_eV(cfg): return max(cfg.conv_tol * HARTREE_TO_EV, 1e-9)

def pass_features(state):
    """Trajectory features of the current pass (5-step window), computed after every step."""
    R, cfg = state.pass_recs, state.cfg; n = len(R); f = [r["fmax"] for r in R]
    out = dict(steps_in_pass=n, fmax_now=f[-1] if n else None, fmax_target=cfg.fmax, maxstep=cfg.maxstep)
    if n >= 2:
        ratios = [f[i] / f[i - 1] for i in range(max(1, n - 5), n) if f[i - 1] > 0]
        out["decay_per_step"] = float(np.exp(np.mean(np.log(ratios))))
        out["uphill_fmax_last4"] = int(sum(f[i] > f[i - 1] for i in range(max(1, n - 4), n)))
        out["oscillation_index"] = out["uphill_fmax_last4"] / min(4, n - 1)
        out["dE_last_eV"] = R[-1]["dE"]
    if n >= 1:
        out["cap_saturation"] = float(np.mean([r["move"] >= 0.95 * r["maxstep"] for r in R[-5:]]))
    out["plateau"] = bool(n >= 6 and f[-1] / f[-6] > 0.9 and f[-1] > cfg.fmax)
    out["energy_falling_last5"] = int(sum(r["dE"] < 0 for r in R[-5:]))       # steps (of the last 5) that lowered the energy
    out["dE_mean_last5_over_noise"] = round(float(np.mean([abs(r["dE"]) for r in R[-5:]])) / scf_noise_eV(cfg), 1) if n else None
    out["scf_converged_last"] = R[-1]["scf_converged"] if n else None
    return out

def detect_trigger(state):
    """-> trigger name or None. Cheap, runs every step. No trigger before 6 steps in a pass (except an SCF failure).
    A condition the agent has just looked at (and answered CONTINUE, or acted on) stays quiet for a while (state.quiet)."""
    R, cfg = state.pass_recs, state.cfg; n = len(R)
    if n == 0: return None
    if not R[-1]["scf_converged"]: return "scf_not_converged"
    if n < 6: return None
    ft = pass_features(state); noise = scf_noise_eV(cfg); found = []
    if R[-1]["dE"] > max(5 * noise, 1e-3): found.append("uphill_step")
    falling = AGENT_VERSION == "v2" and ft.get("energy_falling_last5", 0) >= min(5, n) and (ft.get("dE_mean_last5_over_noise") or 0) > 5
    if ft.get("uphill_fmax_last4", 0) >= 3 and not falling: found.append("oscillation")       # v2: fmax wobbling while the energy keeps falling is not oscillation
    ks = [r["ratio"] for r in R[-2:] if r.get("ratio") is not None]
    if len(ks) == 2 and (np.median(ks) < 0.25 or np.median(ks) > 2.0): found.append("curvature_mismatch")
    if ft["plateau"] and not falling: found.append("plateau")
    if ft.get("cap_saturation", 0) >= 1.0 and R[-1]["fmax"] > 3 * cfg.fmax: found.append("steps_pinned_at_cap")
    for name in found:
        if state.quiet.get(name, 0) <= state.step_total: return name
    return None

def step_advisor(state, candidate=None):
    """Deterministic advice from the measured curvature. Free (no DFT). Returns suggested settings plus a forecast."""
    cfg, R = state.cfg, state.pass_recs
    ms = float((candidate or {}).get("maxstep", cfg.maxstep)); F = state.forces
    valid = [r for r in R[-6:] if r.get("k_meas") is not None and r.get("k_assumed", 0) > 0]
    reasons, conf = [], "ok"
    if len(valid) < 3: conf = "low"; reasons.append("fewer than 3 usable steps")
    if valid:
        k_meas = float(np.median([r["k_meas"] for r in valid[-3:]])); k_ass = float(np.median([r["k_assumed"] for r in valid[-3:]]))
        ratio = float(np.median([r["ratio"] for r in valid[-3:]]))
    else:
        k_meas = k_ass = ratio = None
        if state.surr_H is not None:
            fv = F.reshape(-1); k_meas = float(fv @ state.surr_H @ fv / (fv @ fv)); reasons.append("curvature from the surrogate Hessian")
    k_eff = float(np.clip(k_meas if k_meas is not None else 30.0, 0.2, 80.0))
    D90 = float(np.percentile(np.linalg.norm(F, axis=1) / k_eff, 90))
    cap = float(np.mean([r["move"] >= 0.95 * r["maxstep"] for r in R[-5:]])) if R else 0.0
    last2 = R[-2:]
    pred_bad = len(last2) == 2 and all(r.get("pred_dE") and r["pred_dE"] < 0 and r["dE"] > 0.25 * r["pred_dE"] for r in last2)
    noise_ratio = float(np.median([abs(r["dE"]) for r in R[-3:]]) / scf_noise_eV(cfg)) if len(R) >= 3 else None
    if ratio is not None and not (0.3 <= ratio <= 3): conf = "low"; reasons.append("curvature ratio outside 0.3-3")
    if noise_ratio is not None and noise_ratio < 5: conf = "low"; reasons.append("energy changes are near the SCF noise")
    falling5 = bool(R) and sum(r["dE"] < 0 for r in R[-5:]) >= min(5, len(R)) and (noise_ratio or 0) > 5
    rec = dict(action="keep", maxstep=ms, reset_memory=False, optimizer=None, tighten_conv_tol=False, reason="settings look appropriate")
    if (ratio is not None and ratio > 2) or pred_bad:
        rec.update(action="shrink_step", maxstep=max(0.002, ms / 2), reset_memory=True, reason="steps overshoot the measured curvature (ratio>2 or energy drop under a quarter of predicted)")
    elif cap >= 0.6 and ratio is not None and 0.5 <= ratio <= 2:
        new = min(0.05, 1.5 * D90, 2 * ms)
        if new > 1.05 * ms: rec.update(action="raise_cap", maxstep=new, reason=f"{cap:.0%} of recent steps sit at the cap and the curvature model is right (ratio {ratio:.2f})")
    elif AGENT_VERSION == "v2" and cap >= 0.6 and falling5 and ms < 0.05 and R[-1]["fmax"] > 5 * cfg.fmax:
        new = min(0.05, 2 * ms, max(1.5 * D90, 1.2 * ms))
        rec.update(action="raise_cap", maxstep=new, reason="steps sit at the cap, the energy fell on every recent step and fmax is far above target: the step cap is too timid (raise again, probation checks it)")
    elif cap >= 0.6 and ratio is not None and ratio < 0.5:
        opt = "lbfgs_mem20" if state.memory < 10 else None      # surrogate_bfgs is not recommended: never validated on DFT forces
        rec.update(action="better_curvature_model", optimizer=opt, reason="steps are capped AND the optimizer's curvature is much too high (ratio<0.5): the model is the problem")
    elif noise_ratio is not None and noise_ratio < 5 and cfg.conv_tol > 1e-8 and R and R[-1]["fmax"] > cfg.fmax:
        rec.update(action="tighten_scf", tighten_conv_tol=True, reason="energy changes are within 5x of the SCF noise")
    fnow = R[-1]["fmax"] if R else float(np.linalg.norm(F, axis=1).max())
    decay = float(np.clip(pass_features(state).get("decay_per_step", 0.9), 0.5, 0.98)) if len(R) >= 2 else 0.9
    n_decay = np.log(max(cfg.fmax / fnow, 1e-9)) / np.log(decay) if fnow > cfg.fmax else 0.0
    steps_to_target = int(min(999, np.ceil(max(n_decay, D90 / max(rec["maxstep"], 1e-6)))))
    spg = state.sec_per_grad or 40.0
    return dict(computed_at_step=state.step_total, cap_saturation=round(cap, 2),
                k_assumed_eV_A2=None if k_ass is None else round(k_ass, 2), k_measured_eV_A2=None if k_meas is None else round(k_meas, 2),
                curvature_ratio=None if ratio is None else round(ratio, 2), D90_A=round(D90, 4),
                scf_noise_ratio=None if noise_ratio is None else round(noise_ratio, 1), recommendation=rec,
                forecast=dict(steps_to_target=steps_to_target, minutes=round(steps_to_target * spg / 60, 1)),
                confidence=conf, confidence_notes=reasons)

def pass_start_advice(state):
    """Recommended settings for the pass about to start (code-side evidence only): loose early passes, no over-converging in the
    first basin, step cap sized from the best curvature estimate available."""
    cfg, i = state.cfg, state.p
    adv = dict(maxstep=cfg.maxstep, steps=cfg.steps, fmax=cfg.fmax, conv_tol=cfg.conv_tol, notes=[])
    in_first_basis = cfg.basis == BASIS_PATH[0] and BASIS_PATH[0] != BASIS_PATH[1]
    last_pass = (i >= len(state.schedule) - 1)
    if in_first_basis and i == 0 and cfg.fmax < 0.1: adv["fmax"] = 0.1; adv["notes"].append("pass 1 only needs to remove the strain")
    if in_first_basis and i == 1 and cfg.fmax < 0.05: adv["fmax"] = 0.05; adv["notes"].append("converging harder in the cheaper basis is wasted: the target basis redoes it")
    if (not in_first_basis) and not last_pass and cfg.fmax < 3 * ENDPOINT_FMAX: adv["fmax"] = 3 * ENDPOINT_FMAX; adv["notes"].append("intermediate target-basis pass: the last pass sets the final accuracy")
    if last_pass: adv["fmax"] = min(cfg.fmax, ENDPOINT_FMAX)
    # curvature estimate: measured in the previous pass if there is one, otherwise the surrogate Hessian
    F = state.forces; prev = [r for r in state.recs[-6:] if r.get("k_meas")]
    if prev: k = float(np.clip(np.median([r["k_meas"] for r in prev[-3:]]), 0.2, 80)); src = "measured in the previous pass"
    elif state.surr_H is not None: fv = F.reshape(-1); k = float(np.clip(fv @ state.surr_H @ fv / (fv @ fv), 0.2, 80)); src = "surrogate Hessian"
    else: k, src = None, None
    if k:
        D90 = float(np.percentile(np.linalg.norm(F, axis=1) / k, 90))
        adv["maxstep"] = float(np.clip(1.5 * D90, 0.005, 0.05)); adv["notes"].append(f"step cap sized from D90 = {D90:.3f} A ({src})")
    adv["conv_tol"] = float(min(cfg.conv_tol, 1e-4 if adv["fmax"] >= 0.08 else 1e-5 if adv["fmax"] >= 0.03 else 1e-6))
    return adv

print("Monitor, triggers and step_advisor ready.")

In [ ]:
# @title 11. L-BFGS step engine and the six functions Gemini may call (+ per-event allow-lists, validator)
import time, copy
from ase.optimize import LBFGS, BFGS

KAPPA_TOL = 0.02          # eV/A^2: a measured curvature below -KAPPA_TOL along a soft mode means "not a minimum"

# ----------------------------------------------------------------------------------------------------------------
class StepEngine:
    """Thin wrapper over ASE's optimizer so the orchestrator can change the step cap, reset the memory, or swap in the
    surrogate-Hessian optimizer between steps without touching ASE internals."""
    def __init__(self, state):
        self.s = state; self.build()

    def build(self):
        s = self.s; cfg = s.cfg
        if s.optimizer_kind == "surrogate_bfgs":
            try:
                if s.surr_pre is None: s.surr_pre = surrogate_preconditioner(s.atoms.positions)
                self.dyn = BFGS(s.atoms, maxstep=cfg.maxstep, logfile=None); self.dyn.H0 = s.surr_pre.copy()
                return
            except Exception as e:
                print(f"   surrogate-Hessian optimizer unavailable ({type(e).__name__}); using L-BFGS memory 20 instead")
                s.optimizer_kind, s.memory = "lbfgs", max(s.memory, 20)
        self.dyn = LBFGS(s.atoms, maxstep=cfg.maxstep, memory=s.memory, logfile=None)

    def set_maxstep(self, ms): self.s.cfg.maxstep = float(ms); self.dyn.maxstep = float(ms)
    def reset_memory(self):    self.build()

    def take_step(self):
        s = self.s; a = s.atoms
        pos0 = a.positions.copy(); F0 = a.get_forces().copy(); E0 = float(a.get_potential_energy()); t0 = time.time()
        self.dyn.step()
        F1 = a.get_forces(); E1 = float(a.get_potential_energy())
        disp = a.positions - pos0; sv = disp.reshape(-1); ss = float(sv @ sv)
        k_ass = float(F0.reshape(-1) @ sv / ss) if ss > 0 else None
        k_meas = float((F0 - F1).reshape(-1) @ sv / ss) if ss > 0 else None
        wall = time.time() - t0
        s.sec_per_grad = wall if s.sec_per_grad is None else 0.7 * s.sec_per_grad + 0.3 * wall
        sc = getattr(a.calc, "last_scf", {}) or {}
        rec = dict(step=s.step_total + 1, pass_id=s.p + 1, step_in_pass=len(s.pass_recs) + 1, E=E1, dE=E1 - E0, fmax=fmax_of(F1),
                   move=float(np.linalg.norm(disp, axis=1).max()), maxstep=float(s.cfg.maxstep), k_assumed=k_ass, k_meas=k_meas,
                   ratio=(k_meas / k_ass) if (k_ass and k_ass > 0 and k_meas is not None) else None,
                   pred_dE=(-0.5 * float(F0.reshape(-1) @ sv)), scf_cycles=sc.get("cycles"), scf_converged=bool(sc.get("converged", True)),
                   wall_s=wall, warm_used=sc.get("warm_start_used"), pos=a.positions.copy())
        s.step_total += 1; s.recs.append(rec); s.pass_recs.append(rec)
        s.logger.grad = s.gradients; s.logger.scf = s.scf_total
        s.logger.step(rec["pass_id"], rec["step"], E1, rec["fmax"], rec["move"], rec["maxstep"], s.cfg.fmax, s.cfg.conv_tol, basis=s.cfg.basis,
                      scf_cycles=rec["scf_cycles"], positions=a.positions, dE_eV=rec["dE"], k_meas=k_meas, k_assumed=k_ass,
                      optimizer=s.optimizer_kind, memory=s.memory)
        return rec

# ----------------------------------------------------------------------------------------------------------------
# The six functions. Every function returns a small JSON-safe dict.
PROBE_KINDS = {"basin_choice": ["single_point"], "end_of_svp": ["basis_switch"], "final_check": ["soft_curvature", "scf_stability"]}
ALLOWED = {   # which functions each event may use (the model is only shown these)
    "basin_choice": ["get_evidence", "explore_surrogate", "run_dft_probe", "submit_action"],
    "pass_start":   ["get_evidence", "step_advisor", "submit_action"],
    "trigger":      ["get_evidence", "step_advisor", "preview_displacement", "submit_action"],
    "end_of_svp":   ["get_evidence", "step_advisor", "run_dft_probe", "submit_action"],
    "final_check":  ["get_evidence", "run_dft_probe", "preview_displacement", "submit_action"],
}
STRONG_EVENTS = {"basin_choice", "final_check"}   # high-stakes events go to the strongest model first

def _r(x, n=5):
    return None if x is None else round(float(x), n)

class Toolbox:
    def __init__(self, state, engine_ref):
        self.s, self.engine_ref = state, engine_ref; self.event = None; self.calls = []

    def call(self, name, args, event):
        self.event = event
        try:
            fn = getattr(self, "tool_" + name)
            out = fn(**{k: v for k, v in (args or {}).items()})
        except Exception as e:
            out = {"error": f"{type(e).__name__}: {str(e)[:200]}"}
        self.calls.append(name)
        self.s.logger.event("tool_call", self.s.p + 1, self.s.step_total, tool=name, args=_jsonable(args), result_keys=list(out)[:8], event=event)
        return out

    # 1 -----------------------------------------------------------------------------------------------------------
    def tool_get_evidence(self, topic="trajectory", window=10):
        s = self.s; window = int(min(max(window, 1), 40)); R = s.pass_recs[-window:]
        if topic == "trajectory":
            return dict(pass_id=s.p + 1, rows=[dict(step=r["step"], dE_eV=_r(r["dE"], 6), fmax=_r(r["fmax"]), move_A=_r(r["move"], 4)) for r in R])
        if topic == "step_diagnostics":
            return dict(rows=[dict(step=r["step"], k_assumed=_r(r["k_assumed"], 2), k_measured=_r(r["k_meas"], 2), ratio=_r(r["ratio"], 2), maxstep=_r(r["maxstep"], 4)) for r in R])
        if topic == "atom_forces":
            F = np.linalg.norm(s.forces, axis=1); order = np.argsort(-F)[:8]
            return dict(top_atoms=[dict(index=int(i), element=SYMBOLS[i], force=_r(F[i])) for i in order])
        if topic == "scf_health":
            return dict(rows=[dict(step=r["step"], converged=r["scf_converged"], cycles=r["scf_cycles"], warm_start_used=r.get("warm_used")) for r in R])
        if topic == "structure":
            return fingerprint(s.atoms.positions)
        if topic in ("basin", "basin_table"):
            return basin_summary(s)
        if topic == "run_log":
            return dict(actions=s.actions[-8:], strikes=s.strikes, agent_calls=s.agent_calls, gradients=s.gradients)
        return {"error": "unknown topic", "topics": ["trajectory", "step_diagnostics", "atom_forces", "scf_health", "structure", "basin_table", "run_log"]}

    # 2 -----------------------------------------------------------------------------------------------------------
    def tool_step_advisor(self, candidate_maxstep=None, candidate_steps=None):
        cand = {}
        if candidate_maxstep: cand["maxstep"] = candidate_maxstep
        s = self.s
        if not s.pass_recs:
            adv = pass_start_advice(s); return dict(kind="pass_start", suggested=adv)
        return step_advisor(s, cand or None)

    # 3 -----------------------------------------------------------------------------------------------------------
    def tool_run_dft_probe(self, kind="single_point", reason="", basin_ids=None):
        s = self.s
        if kind not in PROBE_KINDS.get(self.event, []): return {"error": f"probe '{kind}' is not allowed for event '{self.event}'", "allowed": PROBE_KINDS.get(self.event, [])}
        if not reason: return {"error": "a reason is required for a DFT probe"}
        g0, scf0, t0 = s.gradients, s.scf_total, time.time()
        if kind == "single_point": out = probe_single_points(s, basin_ids)
        elif kind == "basis_switch": out = probe_basis_switch(s)
        elif kind == "soft_curvature": out = probe_soft_curvature(s)
        else: out = dict(available=False, note="SCF stability analysis is not wired into this notebook; flag for a manual check if the gap is small.")
        out["cost"] = dict(gradients=s.gradients - g0, scf_calls=s.scf_total - scf0, seconds=round(time.time() - t0, 1))
        s.logger.grad = s.gradients; s.logger.event("probe", s.p + 1, s.step_total, probe=kind, reason=reason, result=_jsonable(out))
        return out

    # 4 -----------------------------------------------------------------------------------------------------------
    def tool_explore_surrogate(self, task="basins", mode=0, amplitude_A=0.1, seed=0):
        s = self.s
        if task == "basins":
            if s.basins is None: ensure_explored(s)
            return basin_summary(s)
        if task == "hessian":
            modes = s.modes or surrogate_modes(s.atoms.positions)[0]
            return dict(softest_modes=[dict(mode=i, curvature_eV_A2=_r(m["curvature_eV_A2"], 3), out_of_plane_share=_r(m["out_of_plane_share"], 2)) for i, m in enumerate(modes)])
        if task == "mode_scan":
            modes = s.modes or surrogate_modes(s.atoms.positions)[0]; v = modes[int(mode)]["vec"].reshape(-1, 3); x0 = s.atoms.positions
            amps = [-2, -1, 0, 1, 2]; e = [xtb_energy(x0 + a * amplitude_A * np.sqrt(N_ATOMS) * v) for a in amps]
            return dict(mode=int(mode), rms_amplitudes_A=[a * amplitude_A for a in amps], xtb_energy_eV_relative=[_r(x - e[2], 5) for x in e])
        return {"error": "unknown task", "tasks": ["basins", "mode_scan", "hessian"]}

    # 5 -----------------------------------------------------------------------------------------------------------
    def tool_preview_displacement(self, mode=0, amplitude_A=0.05, sign=1):
        s = self.s; modes = s.modes or surrogate_modes(s.atoms.positions)[0]; s.modes = modes
        v = modes[int(mode)]["vec"].reshape(-1, 3); x0 = s.atoms.positions
        x1 = x0 + (1 if sign >= 0 else -1) * amplitude_A * np.sqrt(N_ATOMS) * v          # amplitude = RMS displacement (A)
        pid = f"disp{len(s.pending_geoms)}"; s.pending_geoms[pid] = x1
        chk = dict(preview_id=pid, max_atom_shift_A=_r(np.linalg.norm(x1 - x0, axis=1).max(), 3), min_pair_distance_A=_r(min_pair_distance(x1), 3),
                   bond_graph_ok=same_bond_graph(x1), plane_rms_A=_r(plane_rms(x1), 4), xtb_energy_change_meV=_r(1000 * (xtb_energy(x1) - xtb_energy(x0)), 3))
        chk["passes_structure_check"] = bool(chk["bond_graph_ok"] and chk["max_atom_shift_A"] <= 0.3 and chk["min_pair_distance_A"] >= 0.7)
        return chk

def _jsonable(o):
    try: return json.loads(json.dumps(o, default=lambda x: float(x) if np.isscalar(x) else str(x)))
    except Exception: return str(o)[:300]

# ---- helpers behind the functions ------------------------------------------------------------------------------
def ensure_explored(s):
    t0 = time.time(); s.basins, s.modes = explore_basins(s.atoms.positions, seed=0)
    s.current_basin = 0 if s.basins else None; s.target_basin = s.current_basin
    s.logger.event("note", s.p + 1, s.step_total, note="surrogate exploration done", basins=len(s.basins), seconds=round(time.time() - t0, 1))
    print(f"   Explore: {len(s.basins)} distinct xTB basin(s) from {sum(b['n_seeds'] for b in s.basins)} seeds ({time.time()-t0:.0f}s); "
          + "; ".join(f"#{b['id']}: +{b['rel_xtb_meV']:.1f} meV, planarity {b['plane_rms_A']:.3f} A, {b['n_seeds']} seeds" for b in s.basins[:4]))

def basin_summary(s):
    if s.basins is None: return {"note": "run explore_surrogate(task='basins') first"}
    return dict(current_basin=s.current_basin, target_basin=s.target_basin, basins=[dict(id=b["id"], xtb_rel_meV=_r(b["rel_xtb_meV"], 2), plane_rms_A=_r(b["plane_rms_A"], 4),
                rmsd_to_planar_ref_A=_r(b["rmsd_to_ref_A"], 4), n_seeds=b["n_seeds"], dft_rel_meV=_r(b.get("dft_rel_meV"), 2)) for b in s.basins[:6]],
                geometry_now=dict(nearest_basin=nearest_basin(s), plane_rms_A=_r(plane_rms(s.atoms.positions), 4) if s.atoms is not None else None))

def nearest_basin(s):
    if not s.basins or s.atoms is None: return None
    d = [(rmsd_sym(b["pos"], s.atoms.positions), b["id"]) for b in s.basins]; d.sort()
    return dict(id=d[0][1], rmsd_A=_r(d[0][0], 3))

def probe_single_points(s, basin_ids=None):
    """DFT single-point energies (first basis) at the surrogate basin geometries: the DFT referee for close basins."""
    ensure_explored(s) if s.basins is None else None
    ids = list(dict.fromkeys(list(basin_ids or [b["id"] for b in s.basins[:3]]) + [s.current_basin])); calc = s.new_calculator(BASIS_PATH[0], 1e-6); res = []
    for i in ids:
        b = s.basins[int(i)]; a = Atoms(SYMBOLS, positions=b["pos"]); a.calc = calc
        b["dft_E_eV"] = float(a.get_potential_energy()); res.append(b)
    e0 = min(b["dft_E_eV"] for b in res)
    for b in res: b["dft_rel_meV"] = 1000 * (b["dft_E_eV"] - e0)
    best = min(res, key=lambda b: b["dft_E_eV"]); s.target_basin = best["id"]
    return dict(single_points=[dict(basin=b["id"], dft_rel_meV=_r(b["dft_rel_meV"], 2), xtb_rel_meV=_r(b["rel_xtb_meV"], 2)) for b in res],
                lowest_basin=best["id"], same_order_as_xtb=bool(best["id"] == s.basins[0]["id"]))

def probe_basis_switch(s):
    if BASIS_PATH[0] == BASIS_PATH[1]: return dict(available=False, note="both requirements use different bases only after pass 3")
    cfg_t = s.schedule[3]; a = s.atoms.copy(); calc = s.new_calculator(BASIS_PATH[1], cfg_t.conv_tol); a.calc = calc
    ft, fs = fmax_of(a.get_forces()), fmax_of(s.forces); r = ft / fs
    s.probe_cache["basis_switch"] = dict(pos=s.atoms.positions.copy(), calc=calc, atoms=a, conv_tol=cfg_t.conv_tol)
    sug = "skip_pass_3" if r >= 3 else ("finish_pass_3" if r < 1.5 else "cut_pass_3_to_forecast")
    return dict(fmax_first_basis=_r(fs), fmax_target_basis=_r(ft), ratio=_r(r, 2), suggestion=sug,
                note="ratio>=3: forces are dominated by the basis change, so more first-basis steps are wasted; <1.5: the bases agree")

def probe_soft_curvature(s, n_modes=3, delta=0.02):
    """Hessian-free minimum check: DFT gradient at x and at x + delta*v for each of the 3 softest xTB modes v.
    kappa = -(F(x+delta v) - F(x)).v / delta   (eV/A^2; negative = not a minimum along that mode). 4 gradients, conv_tol 1e-8."""
    pos = s.atoms.positions.copy(); modes, _ = surrogate_modes(pos, n_modes)
    calc = s.new_calculator(s.cfg.basis, 1e-8); a = Atoms(SYMBOLS, positions=pos); a.calc = calc; F0 = a.get_forces().copy()
    xa = xtb_atoms(pos); Fx0 = xa.get_forces().copy(); out = []
    for m in modes:
        v = m["vec"]; a.positions = pos + delta * v.reshape(-1, 3); dF = (a.get_forces() - F0).reshape(-1)
        xa.positions = pos + delta * v.reshape(-1, 3); dFx = (xa.get_forces() - Fx0).reshape(-1)
        kappa = float(-(dF @ v) / delta); cosine = float(dF @ dFx / (np.linalg.norm(dF) * np.linalg.norm(dFx) + 1e-30))
        out.append(dict(xtb_curvature_eV_A2=_r(m["curvature_eV_A2"], 3), kappa_dft_eV_A2=_r(kappa, 4), out_of_plane_share=_r(m["out_of_plane_share"], 2), force_change_cosine_with_xtb=_r(cosine, 2), negative=bool(kappa < -KAPPA_TOL)))
    s.modes = modes
    s.final_curv = dict(modes=out, any_negative=any(o["negative"] for o in out), at_step=s.step_total, vecs=[m["vec"] for m in modes], pos=pos)
    return dict(modes=out, any_negative=s.final_curv["any_negative"], note="verified along the 3 softest surrogate modes; this is not a frequency calculation")

print("Step engine and the six functions are defined.")

In [ ]:
# @title 12. Actions, validator (bounds, plausibility, evidence gate), probation and rollback
ACTION_RISK = {"CONTINUE": "none", "SET_PARAMS": "low", "RESET_LBFGS_MEMORY": "low", "TIGHTEN_SCF": "low",
               "END_PASS_AND_ESCALATE": "medium", "STOP_AND_FLAG": "medium", "DISPLACE_ALONG_MODE": "high", "SWITCH_BASIN": "high"}
OPTIMIZERS = {None, "lbfgs", "lbfgs_mem20", "surrogate_bfgs"}
SWITCH_MARGIN_MEV = 43.0     # 1 kcal/mol: a DFT referee must prefer another basin by at least this much
if MOL.get("fixed_shape"): SWITCH_MARGIN_MEV = 1e9   # 5-FU: T3 and T3B are separate shapes; the agent may not hop between O-H rotamers

def max_conv_tol(fmax):
    return 1e-4 if fmax >= 0.08 else 1e-5 if fmax >= 0.03 else 1e-6

def is_last_pass(s): return s.p >= len(s.schedule) - 1

def validate(s, dec, event):
    """-> (status, cleaned_decision, notes). status: accepted | clipped | rejected. Nothing reaches the optimizer unvalidated."""
    notes = []; a = (dec or {}).get("action"); p = dict((dec or {}).get("params") or {}); ev = list((dec or {}).get("evidence") or [])
    if a not in ACTION_RISK: return "rejected", None, [f"unknown action {a!r}"]
    cfg = s.cfg; last = is_last_pass(s)
    if a in ("SET_PARAMS", "RESET_LBFGS_MEMORY", "TIGHTEN_SCF"):
        out = {}
        if "maxstep" in p:
            hi = min(0.05, max(2 * cfg.maxstep, 0.02))          # at most double the current cap per decision (never above 0.05)
            v = float(np.clip(float(p["maxstep"]), 0.001, hi));  notes += [f"maxstep clipped to {v:.4f}"] if abs(v - float(p["maxstep"])) > 1e-12 else []; out["maxstep"] = v
        if "fmax" in p:
            v = float(np.clip(float(p["fmax"]), 0.005, min(0.5, 2 * cfg.fmax)))   # may loosen the target at most 2x per decision
            if last: v = min(v, ENDPOINT_FMAX)
            notes += [f"fmax set to {v}"] if abs(v - float(p["fmax"])) > 1e-12 else []; out["fmax"] = v
        if "steps" in p:
            rem = max(0, GRADIENT_BUDGET_PER_RUN - s.gradients); v = int(max(0, min(int(p["steps"]), rem)))
            if last and v == 0: v = 1; notes.append("the final pass cannot be skipped")
            notes += [f"steps set to {v}"] if v != int(p["steps"]) else []; out["steps"] = v
        fm = out.get("fmax", cfg.fmax); ct_req = float(p.get("conv_tol", cfg.conv_tol)); ct = min(ct_req, max_conv_tol(fm))
        if last: ct = min(ct, 1e-6)
        if ct != cfg.conv_tol or "conv_tol" in p: out["conv_tol"] = ct
        if ct != ct_req: notes.append(f"conv_tol tightened to {ct:.0e} to match fmax {fm}")
        if "grid_level" in p: out["grid_level"] = int(np.clip(int(p["grid_level"]), 3, 4))
        if p.get("optimizer") not in OPTIMIZERS: return "rejected", None, [f"unknown optimizer {p.get('optimizer')!r}"]
        if p.get("optimizer") == "surrogate_bfgs": notes.append("optimizer surrogate_bfgs dropped: not validated on DFT forces"); p["optimizer"] = None
        if p.get("optimizer"): out["optimizer"] = p["optimizer"]
        if a == "TIGHTEN_SCF" and "conv_tol" not in out: out["conv_tol"] = max(cfg.conv_tol * 0.1, 1e-9)
        clean = dict(dec, params=out)
        return ("clipped" if notes else "accepted"), clean, notes
    if a == "END_PASS_AND_ESCALATE":
        fnow = s.pass_recs[-1]["fmax"] if s.pass_recs else None
        if last and (fnow is None or fnow > cfg.fmax): return "rejected", None, ["the final pass cannot be ended before its target is met"]
        if fnow is None or not (pass_features(s).get("plateau") or fnow <= 2 * cfg.fmax): return "rejected", None, ["evidence gate: needs a plateau flag or fmax within 2x of the pass target"]
        return "accepted", dec, notes
    if a == "DISPLACE_ALONG_MODE":
        pid = p.get("preview_id"); x1 = s.pending_geoms.get(pid)
        if x1 is None: return "rejected", None, ["no such preview_id: call preview_displacement first"]
        if not (np.linalg.norm(x1 - s.atoms.positions, axis=1).max() <= 0.3 and min_pair_distance(x1) >= 0.7 and same_bond_graph(x1)):
            return "rejected", None, ["physical plausibility: shift <= 0.3 A, no pair < 0.7 A, bond graph unchanged"]
        neg = bool(s.final_curv and s.final_curv["any_negative"]); locked = plane_rms(s.atoms.positions) < 1e-3
        if not (neg or locked): return "rejected", None, ["evidence gate: needs negative curvature from the soft-mode probe, or an exactly planar (symmetry-locked) geometry"]
        return "accepted", dec, notes
    if a == "SWITCH_BASIN":
        bid = p.get("basin_id")
        if s.basins is None or bid is None or not (0 <= int(bid) < len(s.basins)): return "rejected", None, ["unknown basin_id"]
        b = s.basins[int(bid)]; cur = s.basins[s.current_basin or 0]
        if "dft_E_eV" not in b or "dft_E_eV" not in cur: return "rejected", None, ["evidence gate: needs DFT single points on both basins (run_dft_probe single_point)"]
        if 1000 * (cur["dft_E_eV"] - b["dft_E_eV"]) < SWITCH_MARGIN_MEV: return "rejected", None, [f"evidence gate: DFT must prefer the new basin by >= {SWITCH_MARGIN_MEV} meV"]
        return "accepted", dec, notes
    return "accepted", dec, notes     # CONTINUE, STOP_AND_FLAG

def checkpoint(s):
    return dict(pos=s.atoms.positions.copy(), E=float(s.atoms.get_potential_energy()), fmax=fmax_of(s.forces), cfg=replace(s.cfg),
                optimizer_kind=s.optimizer_kind, memory=s.memory, step=s.step_total)

def apply_action(s, eng, dec):
    """Executes a VALIDATED decision. eng is the StepEngine (None before a pass has started)."""
    a, p = dec["action"], dec.get("params") or {}; cfg = s.cfg; rebuild = False; new_calc = False
    if a in ("SET_PARAMS", "RESET_LBFGS_MEMORY", "TIGHTEN_SCF"):
        for k in ("fmax", "steps"):
            if k in p: setattr(cfg, k, p[k])
        for k in ("conv_tol", "grid_level"):
            if k in p and p[k] != getattr(cfg, k): setattr(cfg, k, p[k]); new_calc = True
        if "maxstep" in p: cfg.maxstep = float(p["maxstep"]); rebuild = rebuild or False
        if p.get("optimizer"):
            if p["optimizer"] == "lbfgs_mem20": s.optimizer_kind, s.memory = "lbfgs", max(s.memory, 20)
            else: s.optimizer_kind = p["optimizer"]
            rebuild = True
        if a == "RESET_LBFGS_MEMORY": rebuild = True
        if new_calc and s.atoms is not None: s.atoms.calc = s.new_calculator(cfg.basis, cfg.conv_tol, cfg.grid_level)
        if eng is not None:
            if rebuild: eng.build()
            elif "maxstep" in p: eng.set_maxstep(cfg.maxstep)
    elif a == "END_PASS_AND_ESCALATE": s.end_pass = True
    elif a == "DISPLACE_ALONG_MODE":
        s.atoms.set_positions(s.pending_geoms[p["preview_id"]]); s.rerun_last_pass = True
        if eng is not None: eng.build()
    elif a == "SWITCH_BASIN":
        s.atoms.set_positions(s.basins[int(p["basin_id"])]["pos"]); s.current_basin = s.target_basin = int(p["basin_id"]); s.switched_basin = True
        if eng is not None: eng.build()
    elif a == "STOP_AND_FLAG": s.stop_flag = (dec.get("rationale") or "agent requested stop")

def rollback(s, eng, ck):
    s.atoms.set_positions(ck["pos"]); s.optimizer_kind, s.memory = ck["optimizer_kind"], ck["memory"]
    tol_changed = (ck["cfg"].conv_tol, ck["cfg"].grid_level) != (s.cfg.conv_tol, s.cfg.grid_level)
    for k in ("maxstep", "steps", "fmax", "conv_tol", "grid_level"): setattr(s.cfg, k, getattr(ck["cfg"], k))
    if tol_changed: s.atoms.calc = s.new_calculator(s.cfg.basis, s.cfg.conv_tol, s.cfg.grid_level)
    eng.build()
print("Validator and action machinery ready.")

In [ ]:
# @title 13. Controllers: none (baselines), rule-based (A2 and the offline stand-in), Gemini function calling (A3)
SYSTEM_PROMPT = """You supervise an L-BFGS geometry optimization of {FORMULA} ({DESC}) with the Skala-1.1 DFT functional.
Every gradient costs about 15-20 seconds of GPU time on the L4. Code measures everything; you only choose among remedies.

Objectives, in this order:
1. Do not end in a local valley or a wrong basin. The lowest basin and a stable SCF matter more than saving time.
2. Then minimise wall-clock time (gradient evaluations).
API latency is not a cost. Never trade objective 1 for objective 2.

How you work:
- Use the functions you are shown. Free functions (get_evidence, step_advisor, preview_displacement) can be called as needed.
  run_dft_probe and explore_surrogate cost real time: call them only with a stated reason.
- Finish EVERY decision by calling submit_action exactly once. If nothing needs changing, submit CONTINUE.
- Call each free function at most once per decision (a repeated identical call returns the same answer). Then submit.
- The rationale must be one real sentence that names the evidence. Never write 'test', 'placeholder' or similar.
- Cite evidence: list the tool results (for example "step_advisor.curvature_ratio=0.31") that justify the action. Never invent numbers.
- Never write coordinates. Moves are made only through preview_displacement ids that code has already checked.
- Prefer low-risk actions: CONTINUE, SET_PARAMS, RESET_LBFGS_MEMORY, TIGHTEN_SCF. High-risk actions (DISPLACE_ALONG_MODE, SWITCH_BASIN)
  need hard evidence (negative curvature from the soft-mode probe, or a DFT single point that prefers another basin by >= 1 kcal/mol).
- Code validates every action, including settings chosen at the start of a pass, runs it on a probation of up to 5 steps, and rolls it back if the energy rises or the force grows. After two rollbacks you are switched off.
- Per decision the step cap can at most double and the force target can be loosened at most 2x; do not take bigger swings.

Rules of thumb that the code has already checked on a surrogate:
- Step cap pinned at the limit and measured curvature matches the optimizer's (ratio 0.5-2): raise maxstep, to at most 1.5 x D90.
- Ratio above 2 or oscillation: halve maxstep and reset L-BFGS memory.
- Steps pinned AND ratio below 0.5: the curvature estimate is unreliable; the safe options are optimizer 'lbfgs_mem20' or a smaller maxstep. Do not guess other optimizers.
- Do not converge hard in the first basis: the target basis redoes it. At the end of pass 2 probe the basis switch; ratio >= 3 means skip pass 3 (steps=0).
- The last pass must reach fmax <= 0.02; that is checked by code, not by you.
"""
SYSTEM_PROMPT_V2 = SYSTEM_PROMPT
_R = [
 ("- Call each free function at most once per decision (a repeated identical call returns the same answer). Then submit.",
  "- The packet already contains the step_advisor result and the trajectory features. Normally submit in your FIRST round. Call get_evidence only for a specific topic the packet does not cover, and each free function at most once."),
 ("- Step cap pinned at the limit and measured curvature matches the optimizer's (ratio 0.5-2): raise maxstep, to at most 1.5 x D90.",
  "- Step cap pinned at the limit (cap_saturation >= 0.6) with the energy falling on every recent step (energy_falling_last5 = 5) and fmax far above target: the cap is too timid. RAISE maxstep (at most 2x per decision, to at most 0.05 A and about 1.5 x D90), and raise it again at the next trigger if the steps are still pinned. Answering CONTINUE while the steps stay pinned and fmax is above 5x target wastes gradients."),
 ("- Ratio above 2 or oscillation: halve maxstep and reset L-BFGS memory.",
  "- Ratio above 2, or oscillation together with a rising or stalled ENERGY: halve maxstep and reset L-BFGS memory. If the energy falls on every step (energy_falling_last5 = 5), fmax wobbling is just a flat valley: do NOT shrink the step and do NOT reset memory; CONTINUE or tighten the SCF instead."),
 ("- Steps pinned AND ratio below 0.5: the curvature estimate is unreliable; the safe options are optimizer 'lbfgs_mem20' or a smaller maxstep. Do not guess other optimizers.",
  "- Steps pinned AND ratio below 0.5 with a falling energy: switch to optimizer 'lbfgs_mem20' once, and raise maxstep as above. Do not guess other optimizers."),
 ("- The last pass must reach fmax <= 0.02; that is checked by code, not by you.",
  "- An SCF tolerance (conv_tol) looser than 1e-5 makes the forces noisy: tighten it when a pass needs fmax below 0.05, or when the energy changes are within 5x of the SCF noise (scf_noise_ratio < 5).\n- The last pass must reach fmax <= 0.02; that is checked by code, not by you."),
]
for a, b in _R:
    assert a in SYSTEM_PROMPT_V2, a[:40]
    SYSTEM_PROMPT_V2 = SYSTEM_PROMPT_V2.replace(a, b)
if AGENT_VERSION == "v2": SYSTEM_PROMPT = SYSTEM_PROMPT_V2
SYSTEM_PROMPT = SYSTEM_PROMPT.replace("{FORMULA}", MOL.get("formula", MOLECULE)).replace("{DESC}", MOL["desc"])

_ARGS_DOC = {
    "get_evidence": "FREE. More detail on one item of the packet.",
    "step_advisor": "FREE. Deterministic advice from measured curvature: suggested maxstep/fmax/conv_tol, and a forecast of steps and minutes.",
    "run_dft_probe": "EXPENSIVE (real DFT gradients). One named measurement: single_point (basin referee), basis_switch, soft_curvature, scf_stability. A reason is required.",
    "explore_surrogate": "CHEAP (xTB, minutes). task = basins (cluster of nearby minima), mode_scan (energy along one soft mode), hessian (softest modes).",
    "preview_displacement": "FREE. Builds a displaced geometry along a soft mode and checks its structure. Never applies it; returns a preview_id.",
    "submit_action": "Hands your decision to the validator. Call exactly once to finish.",
}

def build_declarations(names):
    from google.genai import types as T
    S, Ty = T.Schema, T.Type
    def obj(props, req=None): return S(type=Ty.OBJECT, properties=props, required=req or [])
    spec = {
        "get_evidence": obj({"topic": S(type=Ty.STRING, enum=["trajectory", "step_diagnostics", "atom_forces", "scf_health", "structure", "basin_table", "run_log"]),
                             "window": S(type=Ty.INTEGER, description="number of recent steps (default 10, max 40)")}, ["topic"]),
        "step_advisor": obj({"candidate_maxstep": S(type=Ty.NUMBER, description="optional step cap to forecast with")}),
        "run_dft_probe": obj({"kind": S(type=Ty.STRING, enum=["single_point", "basis_switch", "soft_curvature", "scf_stability"]),
                              "reason": S(type=Ty.STRING), "basin_ids": S(type=Ty.ARRAY, items=S(type=Ty.INTEGER))}, ["kind", "reason"]),
        "explore_surrogate": obj({"task": S(type=Ty.STRING, enum=["basins", "mode_scan", "hessian"]), "mode": S(type=Ty.INTEGER),
                                  "amplitude_A": S(type=Ty.NUMBER), "seed": S(type=Ty.INTEGER)}, ["task"]),
        "preview_displacement": obj({"mode": S(type=Ty.INTEGER), "amplitude_A": S(type=Ty.NUMBER, description="RMS displacement, 0.02-0.1"),
                                     "sign": S(type=Ty.INTEGER, description="+1 or -1")}, ["mode"]),
        "submit_action": obj({
            "action": S(type=Ty.STRING, enum=list(ACTION_RISK)),
            "params": obj({"maxstep": S(type=Ty.NUMBER), "fmax": S(type=Ty.NUMBER), "conv_tol": S(type=Ty.NUMBER), "steps": S(type=Ty.INTEGER),
                           "optimizer": S(type=Ty.STRING, enum=["lbfgs", "lbfgs_mem20"]), "preview_id": S(type=Ty.STRING), "basin_id": S(type=Ty.INTEGER)}),
            "rationale": S(type=Ty.STRING, description="one or two sentences"),
            "evidence": S(type=Ty.ARRAY, items=S(type=Ty.STRING), description="tool results that justify the action"),
            "expected": obj({"fmax_after_5_steps": S(type=Ty.NUMBER), "steps_to_target": S(type=Ty.INTEGER)})}, ["action", "rationale"]),
    }
    return [T.FunctionDeclaration(name=n, description=_ARGS_DOC[n], parameters=spec[n]) for n in names]

def packet_for(s, event):
    cfg = s.cfg; R = s.pass_recs
    return dict(event=event, molecule=dict(formula=MOLECULE, n_atoms=N_ATOMS, start=s.start, arm=s.arm),
                requirement=dict(id=REQUIREMENT, basis_path=list(BASIS_PATH), spherical_d=True),
                run=dict(pass_label=f"{s.p+1}/{len(s.schedule)}", basis=cfg.basis, total_steps=s.step_total, gradients_used=s.gradients,
                         gradient_budget=GRADIENT_BUDGET_PER_RUN, minutes_so_far=round(s.logger.t_opt() / 60, 1), agent_calls=s.agent_calls, rollbacks=s.strikes),
                settings=dict(maxstep=cfg.maxstep, steps=cfg.steps, fmax=cfg.fmax, conv_tol=cfg.conv_tol, optimizer=s.optimizer_kind, memory=s.memory),
                features={k: (round(v, 4) if isinstance(v, float) else v) for k, v in pass_features(s).items()} if R else None,
                last_8_steps=[[r["step"], round(r["dE"], 5), round(r["fmax"], 4)] for r in R[-8:]],
                structure=fingerprint(s.atoms.positions), basin=basin_summary(s) if s.basins else None,
                past_actions=s.actions[-5:], allowed_functions=ALLOWED[event],
                **({"step_advisor": _jsonable(step_advisor(s))} if AGENT_VERSION == "v2" and event in ("trigger", "pass_start", "end_of_svp") else {}))

def _decision(action="CONTINUE", params=None, rationale="", evidence=None, expected=None, **meta):
    return dict(action=action, params=params or {}, rationale=rationale, evidence=evidence or [], expected=expected or {}, **meta)

# ------------------------------------------------------------------------------------------------------------------
class NullController:
    """Baselines A0e/A0r/A0c/A1: no decisions, the schedule runs as written."""
    name = "none"
    def decide(self, event, s, tb): return _decision(rationale="baseline arm: no agent")

class RuleController:
    """Fixed if-then rules over the same six functions (arm A2, and the offline stand-in for Gemini in mock mode)."""
    name = "rules"
    def decide(self, event, s, tb):
        cfg = s.cfg
        if event == "basin_choice":
            tb.call("explore_surrogate", dict(task="basins"), event)
            if len(s.basins) < 2: return _decision(rationale="only one basin found", evidence=["explore_surrogate.basins=1"])
            r = tb.call("run_dft_probe", dict(kind="single_point", reason="DFT referee for the explored basins"), event)
            cur, best = s.current_basin, r.get("lowest_basin")
            if best is not None and best != cur and "dft_E_eV" in s.basins[best] and "dft_E_eV" in s.basins[cur] and 1000 * (s.basins[cur]["dft_E_eV"] - s.basins[best]["dft_E_eV"]) >= SWITCH_MARGIN_MEV:
                return _decision("SWITCH_BASIN", dict(basin_id=int(best)), "DFT prefers another basin by more than 1 kcal/mol", ["run_dft_probe.single_points"])
            return _decision(rationale="DFT keeps the start's basin as the lowest", evidence=["run_dft_probe.lowest_basin=%s" % best])
        if event == "pass_start":
            adv = tb.call("step_advisor", {}, event).get("suggested", {})
            diff = {k: adv[k] for k in ("maxstep", "fmax", "conv_tol") if k in adv and abs(adv[k] - getattr(cfg, k)) > 1e-12 * max(1, abs(adv[k]))}
            if diff: return _decision("SET_PARAMS", diff, "; ".join(adv.get("notes", [])) or "pass-start advice", ["step_advisor.suggested"])
            return _decision(rationale="settings already match the advice")
        if event == "end_of_svp":
            r = tb.call("run_dft_probe", dict(kind="basis_switch", reason="decide whether another first-basis pass is worth its gradients"), event)
            sug = r.get("suggestion")
            if sug == "skip_pass_3": return _decision("SET_PARAMS", dict(steps=0), f"target-basis fmax is {r['ratio']}x the first-basis fmax: forces are dominated by the basis change", [f"run_dft_probe.ratio={r['ratio']}"])
            if sug == "cut_pass_3_to_forecast": return _decision("SET_PARAMS", dict(steps=min(cfg.steps, 12)), "bases partly disagree: cut the pass", [f"run_dft_probe.ratio={r['ratio']}"])
            return _decision(rationale="bases agree: finishing the pass is cheap insurance", evidence=[f"run_dft_probe.ratio={r.get('ratio')}"])
        if event == "trigger":
            adv = tb.call("step_advisor", {}, event); rec = adv.get("recommendation", {}); ev = [f"step_advisor.curvature_ratio={adv.get('curvature_ratio')}", f"step_advisor.cap_saturation={adv.get('cap_saturation')}"]
            tr = s.last_trigger
            if rec.get("action") == "shrink_step": return _decision("RESET_LBFGS_MEMORY", dict(maxstep=rec["maxstep"]), rec["reason"], ev, dict(fmax_after_5_steps=round(s.pass_recs[-1]["fmax"], 4)))
            if rec.get("action") == "raise_cap": return _decision("SET_PARAMS", dict(maxstep=rec["maxstep"]), rec["reason"], ev)
            if rec.get("action") == "better_curvature_model" and rec.get("optimizer"): return _decision("SET_PARAMS", dict(optimizer=rec["optimizer"]), rec["reason"], ev)
            if rec.get("action") == "tighten_scf": return _decision("TIGHTEN_SCF", dict(conv_tol=max(cfg.conv_tol * 0.1, 1e-9)), rec["reason"], ev)
            if tr == "plateau" and s.pass_recs[-1]["fmax"] <= 2 * cfg.fmax and not is_last_pass(s): return _decision("END_PASS_AND_ESCALATE", {}, "flat and within 2x of the pass target", ev)
            if tr in ("oscillation", "uphill_step"): return _decision("RESET_LBFGS_MEMORY", dict(maxstep=max(0.002, cfg.maxstep / 2)), f"{tr}: smaller steps and a clean memory", ev)
            return _decision(rationale=f"trigger {tr}: advisor sees nothing to fix", evidence=ev)
        if event == "final_check":
            r = tb.call("run_dft_probe", dict(kind="soft_curvature", reason="verify the final geometry is a minimum along the 3 softest modes"), event)
            if r.get("any_negative"):
                i = int(np.argmin([m["kappa_dft_eV_A2"] for m in r["modes"]]))
                best = None
                for sg in (+1, -1):
                    pv = tb.call("preview_displacement", dict(mode=i, amplitude_A=0.05, sign=sg), event)
                    if pv.get("passes_structure_check") and (best is None or pv["xtb_energy_change_meV"] < best["xtb_energy_change_meV"]): best = pv
                if best: return _decision("DISPLACE_ALONG_MODE", dict(preview_id=best["preview_id"]), "negative curvature along a soft mode: push and re-optimize", [f"run_dft_probe.modes[{i}].kappa={r['modes'][i]['kappa_dft_eV_A2']}"])
            return _decision(rationale="curvature verified along the 3 softest modes", evidence=["run_dft_probe.any_negative=False"])
        return _decision()

PLACEHOLDER_WORDS = {"test", "placeholder", "n/a", "none", "todo", "tbd", "ok", "continue"}

class GeminiController:
    """Arm A3. Function calling with per-event allow-lists. Every call goes through the rate-limit-safe pool."""
    name = "gemini"
    def __init__(self, pool): self.pool = pool; self.fallback = RuleController()

    def decide(self, event, s, tb):
        from google.genai import types as T
        names = ALLOWED[event]; role = "strong" if event in STRONG_EVENTS else "fast"
        packet = packet_for(s, event)
        contents = [T.Content(role="user", parts=[T.Part.from_text(text="Decision needed now.\n```json\n" + json.dumps(packet, default=float) + "\n```")])]
        tools_called, models, waited, elapsed, rot, errors = [], [], 0.0, 0.0, 0, []
        seen, rationale_retry = {}, False
        def _meta(): return dict(models=models, tools_called=tools_called, api_waited_s=round(waited, 1), api_elapsed_s=round(elapsed, 1), rotations=rot, errors=errors[-4:])
        for rnd in range(MAX_TOOL_ROUNDS):
            allowed = names if rnd < MAX_TOOL_ROUNDS - 1 else ["submit_action"]          # the last round must submit
            cfg_g = T.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.0,
                                            tools=[T.Tool(function_declarations=build_declarations(allowed))],
                                            tool_config=T.ToolConfig(function_calling_config=T.FunctionCallingConfig(mode="ANY", allowed_function_names=allowed)))
            try:
                resp, meta = self.pool.generate(role, contents, cfg_g, label=f"{s.arm}/{s.start}/{event}/round{rnd+1}")
            except ApiBadRequest as e:
                s.logger.add_api_time(elapsed + e.elapsed_s)
                s.logger.event("note", s.p + 1, s.step_total, note="Gemini rejected the request", detail=str(e)[:300])
                raise ApiExhausted("bad request: " + str(e)[:200])
            except ApiExhausted as e:
                s.logger.add_api_time(elapsed + e.elapsed_s); raise
            waited += meta.get("waited_s", 0.0); elapsed += meta.get("elapsed_s", 0.0); rot += meta.get("key_rotations", 0) + meta.get("model_rotations", 0)
            models.append(meta.get("model")); errors += meta.get("errors", [])
            parts = (resp.candidates[0].content.parts or []) if resp.candidates else []
            calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
            if not calls: contents.append(T.Content(role="user", parts=[T.Part.from_text(text="Call submit_action now.")])); continue
            contents.append(resp.candidates[0].content); replies = []
            for c in calls:
                args = dict(c.args or {}); tools_called.append(c.name)
                if c.name == "submit_action":
                    rat = str(args.get("rationale", "")).strip()
                    if (len(rat) < 20 or rat.lower().strip(". ") in PLACEHOLDER_WORDS) and not rationale_retry and rnd < MAX_TOOL_ROUNDS - 1:
                        rationale_retry = True      # one retry: ask for a real sentence instead of accepting 'test'
                        replies.append(T.Part.from_function_response(name=c.name, response={"error": "The rationale is empty, too short or a placeholder. Call submit_action again with the same decision and one real sentence naming the evidence."}))
                        continue
                    if len(rat) < 20 or rat.lower().strip(". ") in PLACEHOLDER_WORDS: rat = f"(no usable rationale given: {rat!r})"
                    dec = _decision(args.get("action", "CONTINUE"), dict(args.get("params") or {}), rat, list(args.get("evidence") or []), dict(args.get("expected") or {}))
                    dec["_meta"] = _meta(); s.logger.add_api_time(elapsed); return dec
                key = (c.name, json.dumps(args, sort_keys=True, default=str))
                if key in seen: result = {"repeat": "identical call already answered in this decision; reuse the earlier result and call submit_action", "earlier_result": seen[key]}
                else: result = seen[key] = tb.call(c.name, args, event)
                replies.append(T.Part.from_function_response(name=c.name, response={"result": result}))
            contents.append(T.Content(role="user", parts=replies))
        s.logger.add_api_time(elapsed)
        return _decision(rationale="no submit_action within the tool-round limit; continuing", _meta=_meta())

print("Controllers ready: NullController, RuleController, GeminiController.")

In [ ]:
# @title 13b. Hybrid controller: the rule set decides, the LLM is consulted only on the four hand-over conditions (H1-H4)
# Written on 13 Nov 2026 before any hybrid run. Do not edit the conditions after seeing results.
HANDOVER = {
    "H1": "no remedy: a warning trigger fired and the rule set would do nothing about it",
    "H2": "conflict: the rule's remedy points the opposite way from the trigger",
    "H3": "after a failure: an action in this run was already rolled back",
    "H4": "high stakes: more than one basin at the start, or a negative curvature at the final check",
}

_packet_for_plain = packet_for
def packet_for(s, event):
    """The usual evidence packet. In LLM + rules mode a hand-over adds one entry: why the LLM is consulted and what the rules would have done."""
    p = _packet_for_plain(s, event)
    if getattr(s, "handover", None): p["handover_from_rules"] = s.handover
    return p

def _brief(dec):
    return None if dec is None else dict(action=dec.get("action"), params=_jsonable(dec.get("params")), rationale=str(dec.get("rationale", ""))[:200])

class HybridController:
    """LLM + rules mode. Same events, same functions, same validator and probation as the other controllers."""
    name = "hybrid"
    def __init__(self, pool, mock=False):
        self.rules = RuleController(); self.mock = mock or pool is None
        self.llm = None if self.mock else GeminiController(pool)

    def handover_reason(self, event, s, rule_dec):
        """-> 'H1'..'H4' or None. rule_dec is the rule set's own decision (None when it was not computed)."""
        if s.strikes >= 1: return "H3"
        if event == "basin_choice" and s.basins and len(s.basins) >= 2: return "H4"
        if rule_dec is None: return None
        if event == "final_check" and rule_dec["action"] == "DISPLACE_ALONG_MODE": return "H4"
        if event == "trigger":
            tr = s.last_trigger; ms = (rule_dec.get("params") or {}).get("maxstep")
            if rule_dec["action"] == "CONTINUE": return "H1"
            if ms is not None and ms > s.cfg.maxstep * (1 + 1e-9) and tr in ("oscillation", "uphill_step"): return "H2"
            if ms is not None and ms < s.cfg.maxstep * (1 - 1e-9) and tr == "steps_pinned_at_cap": return "H2"
        return None

    def decide(self, event, s, tb):
        # H3 and the basin part of H4 are checked BEFORE the rules run, so that a costly probe is never made twice.
        code = self.handover_reason(event, s, None); rule_dec = None
        if code is None:
            rule_dec = self.rules.decide(event, s, tb)      # free at a trigger or pass start; at the final check it runs the curvature probe that is needed in any case
            code = self.handover_reason(event, s, rule_dec)
        if code is None:
            rule_dec["_meta"] = dict(decided_by="rules"); return rule_dec
        why = f"{code} {HANDOVER[code]}" + (f" (trigger: {s.last_trigger})" if event == "trigger" else "")
        print(f"   [hybrid] hand-over to the LLM: {why}")
        if self.mock:                                        # wiring test without an API: the rule decision stands in for the LLM's
            dec = rule_dec if rule_dec is not None else self.rules.decide(event, s, tb)
            dec["_meta"] = dict(decided_by="llm_mock", handover=code, handover_reason=why, rules_would_do=_brief(rule_dec)); return dec
        s.handover = dict(why=why, rules_would_do=_brief(rule_dec),
                          note="The rule set normally decides. You are consulted because of the condition in 'why'. Decide on the evidence; you may agree with the rules or not."
                               + (" The soft_curvature probe has just been run: do not repeat it; its result is in the run state and previews are available." if event == "final_check" and rule_dec is not None else ""))
        try:
            dec = self.llm.decide(event, s, tb)
        finally:
            s.handover = None
        meta = dec.get("_meta") or {}; meta.update(decided_by="llm", handover=code, handover_reason=why, rules_would_do=_brief(rule_dec)); dec["_meta"] = meta
        return dec

print("Hybrid controller ready. Hand-over conditions:"); [print(f"   {k}: {v}") for k, v in HANDOVER.items()]


## Orchestrator and the experiment grid

In [ ]:
# @title 14. Orchestrator: one optimization run (passes, triggers, agent calls, probation, rollback, evaluation, summary)
import traceback, shutil

def _fmt_min(sec): return f"{sec/60:.1f} min"

def make_controller(arm, pool):
    kind = ARM_SPEC[arm]["agent"]
    if kind is None: return NullController()
    if kind == "hybrid": return HybridController(pool, mock=(AGENT_MODE == "mock"))
    if kind == "rules" or AGENT_MODE == "mock": return RuleController()
    return GeminiController(pool)

def call_agent(s, controller, tb, event):
    """One agent decision: ask -> validate -> return the cleaned decision (or None). Never raises."""
    if not s.agent_enabled or s.agent_calls >= s.max_agent_calls: return None
    s.agent_calls += 1; n = s.agent_calls; s.event_name = event; t0 = time.time()
    print(f"   [agent #{n}] event '{event}' at pass {s.p+1}, step {s.step_total} ... asking {controller.name}")
    try:
        dec = controller.decide(event, s, tb)
    except ApiExhausted as e:
        s.degraded_calls += 1
        s.logger.event("agent_call", s.p + 1, s.step_total, trigger=event, status="degraded", detail=str(e)[:300])
        print(f"   [agent #{n}] Gemini unavailable ({str(e)[:120]}). Continuing with CONTINUE; this run is flagged 'degraded' and will be re-run later.")
        return None
    except Exception as e:
        s.agent_errors += 1
        s.logger.event("agent_call", s.p + 1, s.step_total, trigger=event, status="error", detail=f"{type(e).__name__}: {str(e)[:300]}")
        print(f"   [agent #{n}] controller error {type(e).__name__}: {str(e)[:150]} -> CONTINUE")
        return None
    meta = dec.pop("_meta", {})
    s.logger.event("agent_call", s.p + 1, s.step_total, trigger=event, status="ok", controller=controller.name, decision=_jsonable(dec),
                   seconds=round(time.time() - t0, 1), **_jsonable(meta))
    status, clean, notes = validate(s, dec, event)
    s.logger.event("validator", s.p + 1, s.step_total, status=status, notes=notes, action=dec.get("action"), params=_jsonable(dec.get("params")))
    act = dict(n=n, step=s.step_total, pass_id=s.p + 1, event=event, action=dec.get("action"), params=_jsonable(dec.get("params")), status=status, rationale=dec.get("rationale", "")[:160])
    if status == "rejected":
        s.actions.append(dict(act, outcome="rejected")); print(f"   [agent #{n}] {dec.get('action')} REJECTED by the validator: {'; '.join(notes)}"); return None
    s.actions.append(dict(act, outcome="pending"))
    print(f"   [agent #{n}] proposes {clean['action']} {clean.get('params') or ''} ({status}). Why: {dec.get('rationale','')[:140]}")
    return clean

def resolve_probation(s, eng, force=False):
    pr = s.probation
    if pr is None or (s.step_total < pr["end"] and not force): return
    ck = pr["ck"]; E, f = s.pass_recs[-1]["E"], s.pass_recs[-1]["fmax"]
    ok = (E <= ck["E"] + max(5 * scf_noise_eV(s.cfg), 1e-4)) and (f <= 1.5 * ck["fmax"] + 1e-9)
    s.logger.event("probation", s.p + 1, s.step_total, outcome="kept" if ok else "rolled_back", action=pr["action"], E_change_eV=E - ck["E"], fmax_before=ck["fmax"], fmax_after=f,
                   start_gradients=pr["g0"], start_t_opt_s=pr["t0"])
    s.actions[pr["idx"]]["outcome"] = "kept" if ok else "rolled_back"; s.probation = None
    if ok: print(f"   probation passed for {pr['action']}: energy {E - ck['E']:+.4f} eV, fmax {ck['fmax']:.3f} -> {f:.3f}"); return
    s.strikes += 1; rollback(s, eng, ck); s.logger.event("rollback", s.p + 1, s.step_total, strikes=s.strikes, level=pr.get("level", "trigger"))
    if pr.get("level") == "pass": s.pass_recs = []; s.cooldown = COOLDOWN_STEPS
    print(f"   probation FAILED for {pr['action']} (energy {E - ck['E']:+.4f} eV): rolled back. Strike {s.strikes} of 2.")
    if s.strikes >= 2:
        s.agent_enabled = False
        for i in range(s.p, len(s.schedule)):
            for k in ("maxstep", "steps", "fmax", "conv_tol", "grid_level"): setattr(s.schedule[i], k, getattr(s.baseline_schedule[i], k))
        s.logger.event("note", s.p + 1, s.step_total, note="agent switched off after two rollbacks; baseline settings finish the run")
        print("   Two rollbacks: the agent is switched off for the rest of this run; the baseline settings finish it.")

def run_pass(s, controller, tb, label=""):
    cfg = s.cfg; i = s.p
    cache = s.probe_cache.get("basis_switch")
    if cache is not None and cfg.basis == BASIS_PATH[1] and np.allclose(cache["pos"], s.atoms.positions) and cache["conv_tol"] == cfg.conv_tol:
        s.atoms.calc = cache["calc"]; print("   reusing the basis-switch probe as step 0 of this pass (no gradient wasted)")
    else:
        s.atoms.calc = s.new_calculator(cfg.basis, cfg.conv_tol, cfg.grid_level)
    s.probe_cache.pop("basis_switch", None)
    s.pass_recs = []; s.cooldown = 0; s.probation = None; s.end_pass = False
    s.logger.event("pass_start", i + 1, s.step_total, settings=asdict(cfg), label=label)
    print(f"\n=== Pass {i+1}/{len(s.schedule)} {label}| {cfg.basis} | maxstep {cfg.maxstep:.4f} A | up to {cfg.steps} steps | stop at fmax {cfg.fmax:.3f} | conv_tol {cfg.conv_tol:.0e} ===")
    hard_reset_gpu_allocator_pool(verbose=False); dynamic_step_evacuation()
    eng = StepEngine(s); s.engine = eng; tb.engine_ref = eng
    pa = getattr(s, "pass_action", None); s.pass_action = None
    if pa is not None and cfg.steps > 0:
        # Pass-start / basis-switch decisions get the same probation as mid-pass ones. The reference energy and force come from THIS pass's own
        # calculator (same basis), so the comparison never crosses a basis switch. A failed probation restores the baseline settings and optimizer.
        ck = checkpoint(s); ck.update(cfg=pa["base_cfg"], optimizer_kind=pa["optimizer_kind"], memory=pa["memory"])
        s.probation = dict(end=s.step_total + min(PROBATION_STEPS, cfg.steps), ck=ck, action=pa["action"], idx=pa["idx"], g0=s.gradients,
                           t0=round(s.logger.t_opt(), 2), level="pass")
    while True:
        if s.stop_flag: break
        if len(s.pass_recs) >= cfg.steps: print(f"   pass budget of {cfg.steps} steps used"); break
        if s.gradients >= GRADIENT_BUDGET_PER_RUN: s.stop_flag = "gradient budget reached"; break
        f_now = fmax_of(s.atoms.get_forces())
        if f_now <= cfg.fmax: print(f"   fmax {f_now:.4f} is already below the pass target {cfg.fmax}"); break
        rec = eng.take_step()
        print(f"   pass {i+1} step {rec['step_in_pass']:3d} | E {rec['E']:.5f} eV (dE {rec['dE']:+.5f}) | fmax {rec['fmax']:.4f} (target {cfg.fmax}) | move {rec['move']:.4f}/{cfg.maxstep:.4f} A | "
              f"{s.gradients} gradients | {_fmt_min(s.logger.t_opt())}")
        dynamic_step_evacuation()
        s.cooldown = max(0, s.cooldown - 1)
        resolve_probation(s, eng)
        if rec["fmax"] <= cfg.fmax: print(f"   pass target reached (fmax {rec['fmax']:.4f})"); break
        if s.agent_enabled and s.cooldown == 0 and s.probation is None:
            trig = detect_trigger(s)
            if trig:
                s.last_trigger = trig; s.logger.event("trigger", i + 1, s.step_total, trigger=trig, features=_jsonable(pass_features(s)))
                print(f"   trigger: {trig}")
                dec = call_agent(s, controller, tb, "trigger")
                if dec and dec["action"] != "CONTINUE":
                    ck = checkpoint(s); idx = len(s.actions) - 1; apply_action(s, eng, dec)
                    s.logger.event("action_applied", i + 1, s.step_total, action=dec["action"], params=_jsonable(dec.get("params")))
                    if dec["action"] in ("SET_PARAMS", "RESET_LBFGS_MEMORY", "TIGHTEN_SCF"):
                        s.probation = dict(end=s.step_total + PROBATION_STEPS, ck=ck, action=dec["action"], idx=idx, g0=s.gradients, t0=round(s.logger.t_opt(), 2))
                    else: s.actions[idx]["outcome"] = "applied"
                    s.cooldown = COOLDOWN_STEPS
                elif dec: s.actions[-1]["outcome"] = "continue"
                s.cooldown = max(s.cooldown, COOLDOWN_STEPS)        # never ask again within the cool-down
                s.quiet[trig] = s.step_total + ((25 if AGENT_VERSION == 'v2' else 15) if (dec is None or dec['action'] == 'CONTINUE') else 8)   # nor about this same condition for a while
    if s.probation is not None and s.pass_recs: resolve_probation(s, eng, force=True)
    F = s.atoms.get_forces()
    s.logger.event("pass_end", i + 1, s.step_total, steps=len(s.pass_recs), fmax=fmax_of(F), energy_eV=float(s.atoms.get_potential_energy()), target_met=bool(fmax_of(F) <= cfg.fmax))
    return fmax_of(F)

def evaluate_final(s):
    """Same curvature check for EVERY arm, charged to 'evaluation', not to the optimization cost."""
    if s.final_curv is not None and s.final_curv["at_step"] == s.step_total: return
    g0 = s.gradients; s.cfg = s.cfg or s.schedule[-1]
    print("\n--- Evaluation (same for all arms, not counted in compute): curvature check along the 3 softest modes ---")
    try:
        probe_soft_curvature(s)
        for m in s.final_curv["modes"]: print(f"   xTB curvature {m['xtb_curvature_eV_A2']} -> DFT kappa {m['kappa_dft_eV_A2']} eV/A^2 ({'NEGATIVE' if m['negative'] else 'ok'})")
    except Exception as e:
        print("   curvature check failed:", repr(e)[:150]); s.final_curv = None
    s.eval_gradients = s.gradients - g0

def run_one(arm, start, run_dir, pool=None):
    spec = ARM_SPEC[arm]; schedule = schedule_for(arm, start)
    meta = dict(arm=arm, arm_label=spec["label"], start=start, requirement=REQUIREMENT, basis_path=list(BASIS_PATH), backend=BACKEND, agent_mode=(AGENT_MODE if spec["agent"] == "gemini" else spec["agent"]), agent_version=AGENT_VERSION,
                spherical_d=SPHERICAL_D, nudge_rms_A=NUDGE_RMS_A, force_unit_fix=FORCE_UNIT_FIX,
                force_convention="eV/A (corrected)" if FORCE_UNIT_FIX else "v7: Eh/Bohr x 27.211386 labelled eV/A (about 1.89x smaller than true eV/A)",
                schedule=[asdict(p) for p in schedule], random_seed=(RANDOM_SEED_BASE + START_INDEX[start]) if spec["schedule"] == "random" else None,
                optimizer_memory=spec["memory"], warm_start=bool(WARM_START and spec["warm"]), endpoint_fmax=ENDPOINT_FMAX, gradient_budget=GRADIENT_BUDGET_PER_RUN,
                models=dict(fast=MODEL_FAST, pro=MODEL_PRO, strong=MODEL_STRONG) if spec["agent"] in ("gemini", "hybrid") else None, gpu=GPU_NAME,
                time_accounting="t_opt_s = wall time minus time spent waiting for the Gemini API (rate limits, back-off, latency)")
    shutil.rmtree(run_dir, ignore_errors=True)
    lg = RunLogger(run_dir, meta); s = RunState(arm, start, schedule, lg); reset_warm_start(); s.atoms = start_atoms(start)
    controller = make_controller(arm, pool); tb = Toolbox(s, None); status = "finished"
    print("\n" + "#" * 100 + f"\n# RUN: arm {arm} ({spec['label']}) | start {start} | requirement {REQUIREMENT} | {BACKEND}\n" + "#" * 100)
    try:
        s.cfg = s.schedule[0]; s.atoms.calc = s.new_calculator(s.cfg.basis, s.cfg.conv_tol, s.cfg.grid_level)
        if spec["explore"]:
            print("--- Explore stage (xTB surrogate): basins near the start, then the agent chooses the target basin ---")
            ensure_explored(s)
            s.surr_H = xtb_hessian(s.basins[s.current_basin]["pos"]); s.surr_pre = surrogate_preconditioner(None, H=s.surr_H)
            dec = call_agent(s, controller, tb, "basin_choice")
            if dec and dec["action"] != "CONTINUE":
                apply_action(s, None, dec); s.actions[-1]["outcome"] = "applied"; s.logger.event("action_applied", 0, 0, action=dec["action"], params=_jsonable(dec.get("params")))
            elif dec: s.actions[-1]["outcome"] = "continue"
        for p in range(len(s.schedule)):
            s.p = p; s.cfg = s.schedule[p]
            if s.agent_enabled:
                evname = "end_of_svp" if (p == 2 and BASIS_PATH[0] != BASIS_PATH[1]) else "pass_start"
                s.pass_action = None
                dec = call_agent(s, controller, tb, evname)
                if dec and dec["action"] != "CONTINUE":
                    base = replace(s.cfg); kind0, mem0 = s.optimizer_kind, s.memory
                    apply_action(s, None, dec); s.actions[-1]["outcome"] = "applied"; s.logger.event("action_applied", p + 1, s.step_total, action=dec["action"], params=_jsonable(dec.get("params")))
                    if dec["action"] in ("SET_PARAMS", "RESET_LBFGS_MEMORY", "TIGHTEN_SCF") and s.cfg.steps > 0:
                        s.pass_action = dict(idx=len(s.actions) - 1, base_cfg=base, optimizer_kind=kind0, memory=mem0, action=dec["action"])
                elif dec: s.actions[-1]["outcome"] = "continue"
            if s.cfg.steps <= 0:
                s.logger.event("pass_end", p + 1, s.step_total, steps=0, skipped=True); print(f"\n=== Pass {p+1}/{len(s.schedule)} skipped (0 steps) ==="); continue
            run_pass(s, controller, tb)
            if s.stop_flag: status = "stopped: " + s.stop_flag; break
        reps = 0
        while status == "finished" and not SMOKE_TEST and fmax_of(s.atoms.get_forces()) > ENDPOINT_FMAX and reps < 3:
            reps += 1; s.p = len(s.schedule) - 1; last = s.schedule[-1]
            s.cfg = replace(last, fmax=ENDPOINT_FMAX, steps=max(last.steps, 25)); s.schedule[-1] = s.cfg
            run_pass(s, controller, tb, label=f"(repeat {reps}: the final criterion fmax<={ENDPOINT_FMAX} is not met yet) ")
            if s.stop_flag: status = "stopped: " + s.stop_flag
        if status == "finished" and s.agent_enabled:
            s.p = len(s.schedule) - 1; s.cfg = s.schedule[-1]
            dec = call_agent(s, controller, tb, "final_check")
            if dec and dec["action"] == "DISPLACE_ALONG_MODE":
                apply_action(s, None, dec); s.actions[-1]["outcome"] = "applied"; s.logger.event("action_applied", s.p + 1, s.step_total, action=dec["action"], params=_jsonable(dec.get("params")))
                s.cfg = replace(s.schedule[-1], fmax=ENDPOINT_FMAX, steps=max(s.schedule[-1].steps, 40)); s.schedule[-1] = s.cfg; s.final_curv = None
                run_pass(s, controller, tb, label="(re-optimization after the displacement) ")
            elif dec: s.actions[-1]["outcome"] = "continue"
        s.t_opt_end = lg.t_opt(); g_opt = s.gradients
        evaluate_final(s)
    except Exception as e:
        status = f"error: {type(e).__name__}: {str(e)[:200]}"; print("RUN ERROR:\n", traceback.format_exc()[-1500:])
        s.t_opt_end = s.t_opt_end or lg.t_opt(); g_opt = s.gradients
    return finish_run(s, lg, run_dir, status, g_opt)

def finish_run(s, lg, run_dir, status, g_opt):
    a = s.atoms; pos = a.positions.copy()
    try: F = a.get_forces(); fm = fmax_of(F); E = float(a.get_potential_energy())
    except Exception: fm, E = None, None
    fp = fingerprint(pos, E)
    kappas = [m["kappa_dft_eV_A2"] for m in s.final_curv["modes"]] if s.final_curv else None
    degraded = s.degraded_calls > 0
    summary = dict(arm=s.arm, arm_label=s.spec["label"], start=s.start, requirement=REQUIREMENT, backend=BACKEND, status=status,
                   converged=bool(fm is not None and fm <= ENDPOINT_FMAX and status == "finished"), final_energy_eV=E, final_fmax=fm,
                   gradients_optimization=g_opt, gradients_evaluation=s.eval_gradients, gradients_total=s.gradients, scf_calls_total=s.scf_total,
                   minutes_optimization=round(s.t_opt_end / 60, 2), minutes_api_wait_excluded=round(lg.api_s / 60, 2), minutes_total_wall=round(lg.t_wall() / 60, 2),
                   steps_total=s.step_total, passes_with_steps=len({r["pass_id"] for r in s.recs}), final_structure=fp,
                   curvature_kappa_eV_A2=kappas, curvature_any_negative=(s.final_curv["any_negative"] if s.final_curv else None),
                   switched_basin=s.switched_basin, basins_found=(len(s.basins) if s.basins else None),
                   scf_not_converged_steps=sum(1 for r in s.recs if not r["scf_converged"]),
                   agent=dict(enabled_at_end=s.agent_enabled, calls=s.agent_calls, rollbacks=s.strikes, degraded_calls=s.degraded_calls, controller_errors=s.agent_errors,
                              actions=[{k: v for k, v in x.items()} for x in s.actions]),
                   agent_integrity=("degraded" if degraded else "ok" if s.spec["agent"] else "not_applicable"),
                   mean_seconds_per_gradient=round(float(np.mean([r["wall_s"] for r in s.recs])), 2) if s.recs else None)
    (Path(run_dir) / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    from ase.io import write as ase_write
    ase_write(str(Path(run_dir) / "final.xyz"), Atoms(SYMBOLS, positions=pos))
    lg.close()
    print(f"\n>>> {s.arm} on {s.start}: {status} | E {E} eV | fmax {fm} | {g_opt} gradients (+{s.eval_gradients} evaluation) | {summary['minutes_optimization']} min optimization "
          f"(+{summary['minutes_api_wait_excluded']} min of API time excluded) | integrity {summary['agent_integrity']}")
    return summary

print("Orchestrator ready.")

# Hybrid-supervisor runs (Skala)

In [ ]:
# @title 14b. Preflight on the GPU: basis sizes, spherical-d check, seconds per gradient, warm-start check (run once, about 10 minutes on an L4)
RUN_PREFLIGHT = True   # @param {type:"boolean"}
if RUN_PREFLIGHT:
    pf = {}
    a0 = start_atoms("P0")
    print("Preflight on start P0. This also answers the open question 'how long does one gradient take on the L4?'")
    for basis in dict.fromkeys(BASIS_PATH):
        calc = make_calculator(basis, 1e-6, 3, warm_start=False); a = a0.copy(); a.calc = calc; secs = []
        for k in range(3):
            a.positions = a0.positions + 0.003 * k * np.random.default_rng(k).standard_normal(a0.positions.shape)   # three slightly different geometries
            t0 = time.time(); F = a.get_forces(); secs.append(time.time() - t0)
        pf[basis] = dict(basis_functions=calc.nao, seconds_per_gradient=[round(x, 1) for x in secs], fmax_first=round(fmax_of(F), 4), scf_cycles=calc.last_scf.get("cycles"))
        print(f"   {basis:12s}: {calc.nao} basis functions | gradient seconds {pf[basis]['seconds_per_gradient']} | SCF cycles {calc.last_scf.get('cycles')}")
    expect = MOL["nao"]
    for b, v in pf.items():
        if BACKEND == "skala" and v["basis_functions"] != expect[b]: print(f"   WARNING: {b} has {v['basis_functions']} functions, expected {expect[b]} with spherical d")
    if BACKEND == "skala":
        reset_warm_start(); c1 = make_calculator(BASIS_PATH[0], 1e-6, 3, warm_start=True); a = a0.copy(); a.calc = c1; e1 = a.get_potential_energy()
        a.positions += 0.002; a.get_forces(); c2 = make_calculator(BASIS_PATH[0], 1e-6, 3, warm_start=False); b = a.copy(); b.calc = c2
        de = abs(a.get_potential_energy() - b.get_potential_energy()) / HARTREE_TO_EV
        pf["warm_start_check"] = dict(abs_energy_difference_Eh=de, warm_start_used=c1.last_scf.get("warm_start_used"), ok=bool(de < 1e-5))
        print(f"   warm start vs fresh guess: energy difference {de:.2e} Eh -> {'OK' if de < 1e-5 else 'TOO LARGE: set WARM_START=False'}")
    json.dump(pf, open(os.path.join(EXPERIMENT_DIR, "preflight.json"), "w"), indent=2)
    del a, a0
    hard_reset_gpu_allocator_pool(verbose=False)

In [ ]:
# @title 14c. The very deformed start D2: distance from the reference and Skala energy of the start
import numpy as np
a0 = start_atoms("D2")
disp = np.linalg.norm(a0.positions - REF_POS, axis=1)       # D2 was built by displacing the reference in the same frame
c14 = make_calculator(BASIS_PATH[0], 1e-6, 3, warm_start=False); a0.calc = c14
E_start = float(a0.get_potential_energy()); F0 = a0.get_forces()
DEFORM = dict(E_start_eV=E_start, rmsd_to_ref_A=float(np.sqrt((disp ** 2).mean())), max_shift_A=float(disp.max()), plane_rms_A=float(plane_rms(a0.positions)),
              fmax_start_v7_units=float(fmax_of(F0)), bond_graph_ok=bool(same_bond_graph(a0.positions)), min_pair_distance_A=float(min_pair_distance(a0.positions)))
print(f"D2 ({SHAPE}): RMS shift from the reference {DEFORM['rmsd_to_ref_A']:.3f} A, largest shift {DEFORM['max_shift_A']:.3f} A, ring planarity RMS {DEFORM['plane_rms_A']:.3f} A, closest pair {DEFORM['min_pair_distance_A']:.2f} A, bond pattern unchanged: {DEFORM['bond_graph_ok']}")
print(f"Skala/{BASIS_PATH[0]} energy of D2: {E_start:.6f} eV ({E_start / HARTREE_TO_EV:.6f} Eh); largest force {DEFORM['fmax_start_v7_units']:.2f} (v7 units). The energy released is reported after the run.")
assert DEFORM['bond_graph_ok'], "the deformed start changed the bond pattern: do not run"
del a0, c14; hard_reset_gpu_allocator_pool(verbose=False)


In [ ]:
# @title 15. Run: automatic smoke test first, then the real run (resumable: finished runs are skipped after a Colab disconnect)
import glob
pool = None
if AGENT_MODE == "gemini" and any(ARM_SPEC[a]["agent"] in ("gemini", "hybrid") for a in list(ARMS) + list(N1_ARMS_OVERRIDE or [])):
    pool = GeminiPool()

def _load_summary(run_dir):
    f = os.path.join(run_dir, "summary.json")
    return json.load(open(f)) if os.path.exists(f) else None

def run_grid(starts, arms):
    global FAIL_FAST
    results = []
    index_path = os.path.join(EXPERIMENT_DIR, "results_index.json")
    for start in starts:
        for arm in arms:
            run_dir = os.path.join(EXPERIMENT_DIR, f"{arm}_{start}")
            old = _load_summary(run_dir)
            if old and old.get("status") == "finished" and not (RERUN_DEGRADED and old.get("agent_integrity") == "degraded"):
                print(f"[skip] {arm} / {start}: already finished ({old['gradients_optimization']} gradients)"); results.append(old); continue
            if old and old.get("agent_integrity") == "degraded": print(f"[redo] {arm} / {start}: earlier run lost agent decisions to API limits")
            summ = run_one(arm, start, run_dir, pool)
            results.append(summ)
            if N1_TEST and arm == "A0e" and not summ.get("converged"):
                print("\nFAIL FAST: the expert schedule A0e did not converge, so the remaining arms are skipped. Fix that first (see the go/no-go summary).")
                FAIL_FAST = True
                break
            json.dump(results, open(index_path, "w"), indent=2, default=float)        # readable index, updated after every run
            hard_reset_gpu_allocator_pool(verbose=False)
    json.dump(results, open(index_path, "w"), indent=2, default=float)
    return results

def smoke_gate(results):
    """The smoke test passes when every run finished, made gradients, produced a finite energy, kept the bond graph, and (Gemini arms) the agent was not degraded."""
    rows, ok = [], True
    for r in results:
        fs = r.get("final_structure") or {}
        checks = dict(finished=r.get("status") == "finished", gradients=(r.get("gradients_total") or 0) > 0,
                      energy_finite=r.get("final_energy_eV") is not None and np.isfinite(r["final_energy_eV"]),
                      bond_graph=bool(fs.get("bond_graph_ok")), agent_ok=r.get("agent_integrity") in ("ok", "not_applicable"))
        rows.append((r.get("arm"), checks)); ok = ok and all(checks.values())
    print("\nSMOKE GATE")
    for arm, ch in rows: print(f"   {arm:5s} " + "  ".join(f"{k}={'OK' if v else 'FAIL'}" for k, v in ch.items()))
    return ok

RESULTS = []
FAIL_FAST = False
t_grid = time.time()
if RUN_MODE == "auto":
    print("=" * 100 + "\nAUTO MODE, STEP 1 of 2: smoke test (wiring check only, nothing from it is a result)\n" + "=" * 100)
    smoke_results = run_grid(STARTS, ARMS)
    if not smoke_gate(smoke_results):
        raise RuntimeError("SMOKE TEST FAILED: the real run was not started. Read the messages above (GPU, Skala install, Gemini keys or Drive), fix that, and run all again.")
    print("\nSmoke test passed. Switching to the real run.\n")
    SMOKE_TEST, N1_TEST = False, True
    STARTS, ARMS = [N1_START], list(N1_ARMS_OVERRIDE)
    EXPERIMENT_DIR = os.path.join(RESULTS_ROOT, "N1_" + f"requirement_{REQUIREMENT}" + RUN_TAG + ("" if BACKEND == "skala" else "_STANDIN"))
    os.makedirs(EXPERIMENT_DIR, exist_ok=True)
    GRADIENT_BUDGET_PER_RUN = R0_GRADIENT_BUDGET if REQUIREMENT == "R0" else GRADIENT_BUDGET_PER_RUN
    print("=" * 100 + f"\nAUTO MODE, STEP 2 of 2: real run {ARMS} on start {STARTS} -> {EXPERIMENT_DIR}\n" + "=" * 100)
    RESULTS = run_grid(STARTS, ARMS)
else:
    RESULTS = run_grid(STARTS, ARMS)
if pool is not None:
    usage = {k: (round(v, 1) if isinstance(v, float) else v) for k, v in pool.stats.items()}
    json.dump(dict(gemini_usage=usage, note="waited_s is API back-off time; it is excluded from every wall-clock metric"), open(os.path.join(EXPERIMENT_DIR, "api_usage.json"), "w"), indent=2)
    print("\nGemini usage:", usage)
print(f"\nGrid finished in {(time.time()-t_grid)/60:.1f} min.")


In [ ]:
# @title 15b. LLM call report and download (API usage, retries, full prompt/answer log of the agent)
import json, os, zipfile, collections
log_path = os.path.join(EXPERIMENT_DIR, "llm_calls.jsonl")
rows = [json.loads(l) for l in open(log_path)] if os.path.exists(log_path) else []
if not rows:
    print("No Gemini calls were logged in this session (arms without the agent make no calls).")
else:
    by = collections.defaultdict(lambda: collections.Counter())
    for r in rows:
        m = r.get("meta") or {}; k = (r["label"].split("/")[0] + "/" + r["label"].split("/")[1], m.get("model") or "-")
        c = by[k]; c["calls"] += 1; c["ok"] += int(r["ok"]); c["failed"] += int(not r["ok"])
        c["extra_attempts"] += max(0, (m.get("attempts") or 1) - 1); c["errors"] += len(m.get("errors") or [])
        c["key_or_model_rotations"] += (m.get("key_rotations") or 0) + (m.get("model_rotations") or 0)
        c["waited_s"] += m.get("waited_s") or 0; c["latency_s"] += m.get("latency_s") or 0
        t = r.get("tokens") or {}
        for kk in ("prompt", "output", "thinking", "total"): c["tok_" + kk] += t.get(kk) or 0
    print(f"{'run / model':46s} calls ok fail retries errors rot  wait_s  lat_s  tok_in tok_out tok_think")
    for (run, model), c in sorted(by.items()):
        print(f"{run + ' / ' + model:46s} {c['calls']:5d} {c['ok']:2d} {c['failed']:4d} {c['extra_attempts']:7d} {c['errors']:6d} {c['key_or_model_rotations']:3d} {c['waited_s']:7.0f} {c['latency_s']:6.0f} {c['tok_prompt']:7d} {c['tok_output']:7d} {c['tok_thinking']:8d}")
    errs = [(r["label"], e) for r in rows for e in ((r.get("meta") or {}).get("errors") or [])] + [(r["label"], r["error"]) for r in rows if r.get("error")]
    print(f"\nAll error / retry messages ({len(errs)}):"); [print("  ", l, "->", e) for l, e in errs[:40]]
    # human-readable transcript: every decision with the exact packet the agent saw and what it answered
    md = ["# Agent transcript (prompt in, answer out)\n", "System prompt: llm_system_prompt.txt\n"]
    for i, r in enumerate(rows, 1):
        md.append(f"\n## {i}. {r['label']} | {r['role']} | {(r.get('meta') or {}).get('model')} | ok={r['ok']} | tokens={r.get('tokens')}\n")
        if r.get("error"): md.append(f"ERROR: {r['error']}\n")
        for turn in r["prompt"][-2:]:
            for p in turn["parts"]: md.append(f"**{turn['role']} (sent)**:\n```\n{json.dumps(p, indent=1, default=str)[:6000]}\n```\n")
        for p in (r.get("answer") or {}).get("parts", []): md.append(f"**model (answered)**:\n```\n{json.dumps(p, indent=1, default=str)[:3000]}\n```\n")
    open(os.path.join(EXPERIMENT_DIR, "agent_transcript.md"), "w").write("\n".join(md))
    # ---- proper JSON: llm_calls.json (exact prompts and responses) and llm_calls_brief.json (one short line per call) ----
    def _brief_answer(ans):
        out = []
        for part in (ans or {}).get("parts", []):
            if "function_call" in part:
                a = part.get("args") or {}
                if part["function_call"] == "submit_action":
                    out.append(dict(tool="submit_action", action=a.get("action"), params=a.get("params"), rationale=str(a.get("rationale", ""))[:240], evidence=a.get("evidence")))
                else: out.append(dict(tool=part["function_call"], args=a))
            elif "text" in part: out.append(dict(text=str(part["text"])[:300]))
        return out
    def _brief_prompt(prompt):
        last = (prompt or [{}])[-1]; out = []
        for part in last.get("parts", []):
            if "function_response" in part: out.append(dict(tool_result=part["function_response"], summary=json.dumps(part.get("response"), default=str)[:300]))
            elif "text" in part: out.append(dict(text=str(part["text"])[:300]))
        return dict(turns_in_conversation=len(prompt or []), last_turn_role=last.get("role"), last_turn=out)
    sys_txt = open(os.path.join(EXPERIMENT_DIR, "llm_system_prompt.txt")).read() if os.path.exists(os.path.join(EXPERIMENT_DIR, "llm_system_prompt.txt")) else None
    full, brief = [], []
    for i, r in enumerate(rows, 1):
        m = r.get("meta") or {}; lab = r["label"].split("/")
        head = dict(call=i, time=r.get("t"), arm=lab[0], start=lab[1] if len(lab) > 1 else None, event=lab[2] if len(lab) > 2 else None,
                    round=lab[3] if len(lab) > 3 else None, role=r.get("role"), model=m.get("model"), ok=r.get("ok"), error=r.get("error"),
                    attempts=m.get("attempts"), retries_errors=m.get("errors"), latency_s=m.get("latency_s"), waited_s=m.get("waited_s"), tokens=r.get("tokens"))
        brief.append(dict(head, sent=_brief_prompt(r.get("prompt")), received=_brief_answer(r.get("answer"))))
        full.append(dict(head, sent_last_turn_exact=(r.get("prompt") or [{}])[-1], prompt_full_exact=r.get("prompt"), response_exact=r.get("answer"), response_brief=_brief_answer(r.get("answer"))))
    json.dump(dict(system_prompt_file="llm_system_prompt.txt", system_prompt=sys_txt, n_calls=len(full), calls=full), open(os.path.join(EXPERIMENT_DIR, "llm_calls.json"), "w"), indent=1, default=str)
    json.dump(dict(n_calls=len(brief), calls=brief), open(os.path.join(EXPERIMENT_DIR, "llm_calls_brief.json"), "w"), indent=1, default=str)
    print(f"Wrote llm_calls.json (exact prompts and responses, {len(full)} calls) and llm_calls_brief.json (short form).")
    zp = f"/content/llm_log_{MOLECULE.lower()}_{REQUIREMENT}{RUN_TAG}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in ("llm_calls.json", "llm_calls_brief.json", "llm_calls.jsonl", "llm_system_prompt.txt", "agent_transcript.md", "api_usage.json"):
            if os.path.exists(os.path.join(EXPERIMENT_DIR, f)): z.write(os.path.join(EXPERIMENT_DIR, f), f)
    print(f"\nLog zip written: {zp} ({os.path.getsize(zp)/1e6:.1f} MB): llm_calls.json and llm_calls_brief.json (proper JSON, every call), llm_calls.jsonl (raw), agent_transcript.md (readable), llm_system_prompt.txt, api_usage.json")
    try:
        from google.colab import files as colab_files; colab_files.download(zp)
    except Exception: pass

In [ ]:
# @title 15c. Agent timeline plot: where each decision happened on the convergence curve
import json, os, glob
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
COL = {"CONTINUE": "#9aa0a6", "SET_PARAMS": "#1a73e8", "RESET_LBFGS_MEMORY": "#e8710a", "TIGHTEN_SCF": "#188038", "DISPLACE_ALONG_MODE": "#a142f4", "SWITCH_BASIN": "#d93025"}
rows_all = []
for run in sorted(glob.glob(os.path.join(EXPERIMENT_DIR, "*_*"))):
    sp, ep = os.path.join(run, "steps.jsonl"), os.path.join(run, "events.jsonl")
    if not (os.path.exists(sp) and os.path.exists(ep)): continue
    S = [json.loads(l) for l in open(sp) if l.strip()]; E = [json.loads(l) for l in open(ep) if l.strip()]
    calls = [e for e in E if e.get("kind") == "agent_call"]
    if not calls or not S: continue
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    g = [s["gradients"] for s in S]
    a1.plot(g, [s["fmax"] for s in S], color="#202124", lw=1.4); a1.set_yscale("log"); a1.set_ylabel("fmax"); a1.axhline(0.02, color="#188038", ls=":", lw=1)
    E0 = min(s["energy_eV"] for s in S); a2.plot(g, [(s["energy_eV"] - E0) * 1000 for s in S], color="#202124", lw=1.4); a2.set_yscale("symlog", linthresh=1); a2.set_ylabel("energy above run minimum (meV)"); a2.set_xlabel("gradients")
    for e in calls:
        d = e.get("decision") or {}; act = d.get("action", "?"); c = COL.get(act, "#5f6368")
        for a in (a1, a2): a.axvline(e["gradients"], color=c, alpha=0.55, lw=1.2 if act != "CONTINUE" else 0.7)
    for e in E:
        if e.get("kind") == "rollback":
            for a in (a1, a2): a.axvline(e["gradients"], color="#d93025", lw=2.5)
    a1.set_title(f"{os.path.basename(run)}: agent decisions on the optimization (blue = SET_PARAMS, orange = memory reset, grey = continue, red thick = rollback)")
    fig.tight_layout(); png = os.path.join(run, "agent_timeline.png"); fig.savefig(png, dpi=130); plt.close(fig); print("Saved", png); rows_all.append(png)
if not rows_all: print("No agent decisions in this session, nothing to plot.")
else:
    try:
        import zipfile
        zp = f"/content/agent_timeline_{MOLECULE.lower()}_{REQUIREMENT}{RUN_TAG}.zip"
        with zipfile.ZipFile(zp, "w") as z:
            for p in rows_all: z.write(p, os.path.relpath(p, EXPERIMENT_DIR))
        from google.colab import files as colab_files; colab_files.download(zp)
    except Exception: pass

In [ ]:
# @title 15d. (annulene comparison numbers removed in the 5-FU copy)
REF_EARLIER = {}


# Results

In [ ]:
# @title 16. Results: LLM mode vs LLM + rules mode (and rules only), with who made each decision
import glob, collections
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
GROUPS = {"cap0.005": dict(title="start cap 0.005 A (too small)", arms=["HS", "GS", "RS", "GS2", "PS"]),
          "poor": dict(title="deliberately poor schedule", arms=["H3p", "A3p", "A2p", "P1p", "A0p"])}
MODE = {"hybrid": "LLM + rules", "gemini": "LLM", "rules": "rules only", None: "plain"}
api_calls = collections.Counter()
_lp = os.path.join(EXPERIMENT_DIR, "llm_calls.jsonl")
if os.path.exists(_lp):
    for l in open(_lp):
        try: api_calls[json.loads(l)["label"].split("/")[0]] += 1
        except Exception: pass
NOW, DEC = {}, {}
for run in sorted(glob.glob(os.path.join(EXPERIMENT_DIR, "*_" + STARTS[0]))):
    arm = os.path.basename(run).rsplit("_", 1)[0]
    if arm not in ARM_SPEC or not os.path.exists(os.path.join(run, "summary.json")): continue
    d = load_run(run); S, E, sm = d["steps"], d["events"], d["summary"]; kind = ARM_SPEC[arm]["agent"]
    calls = [e for e in E if e.get("kind") == "agent_call" and e.get("status") == "ok"]
    by = collections.Counter((e.get("decided_by") or ("llm" if kind == "gemini" and AGENT_MODE == "gemini" else "rules")) for e in calls)
    rows_d, last_trig = [], None
    for e in E:
        if e.get("kind") == "trigger": last_trig = e.get("trigger")
        if e.get("kind") == "agent_call" and e.get("status") == "ok":
            dec = e.get("decision") or {}
            rows_d.append(dict(step=e.get("step"), pass_id=e.get("pass_id"), event=e.get("trigger"), trigger=last_trig if e.get("trigger") == "trigger" else None,
                               decided_by=e.get("decided_by") or ("llm" if kind == "gemini" and AGENT_MODE == "gemini" else "rules"), handover=e.get("handover"),
                               action=dec.get("action"), params=dec.get("params"), rules_would_do=e.get("rules_would_do"), rationale=str(dec.get("rationale", ""))[:140]))
    DEC[arm] = rows_d
    NOW[arm] = dict(label=ARM_SPEC[arm]["label"], session="now", mode=MODE.get(kind, str(kind)), converged=bool(sm.get("converged")), steps=sm.get("steps_total"), grads=sm.get("gradients_optimization"),
                    first_cross_step=next((i for i, r in enumerate(S, 1) if r["fmax"] <= ENDPOINT_FMAX), None), final_fmax=sm.get("final_fmax"), final_E=sm.get("final_energy_eV"),
                    decisions=len(calls), by_rules=by.get("rules", 0), by_llm=by.get("llm", 0) + by.get("llm_mock", 0), llm_api_calls=api_calls.get(arm, 0),
                    rollbacks=sum(e.get("kind") == "rollback" for e in E), degraded=sm.get("agent_integrity") == "degraded",
                    cap_changes=[(e.get("step"), round((e.get("params") or {}).get("maxstep"), 4)) for e in E if e.get("kind") == "action_applied" and (e.get("params") or {}).get("maxstep") is not None],
                    fmax=[r["fmax"] for r in S], cap=[r["maxstep"] for r in S],
                    llm_steps=[x["step"] for x in rows_d if x["decided_by"].startswith("llm") and x["event"] == "trigger"])
show_ref = BACKEND == "skala" and STARTS[0] == "R1" and MOLECULE == "C10H6"
TABLE = []
print(f"{MOLECULE}, start {STARTS[0]}, target fmax <= {ENDPOINT_FMAX} (v7 force convention), budget {GRADIENT_BUDGET_PER_RUN} gradients. Each row is a single run.")
for g, G in GROUPS.items():
    print(f"\n=== {G['title']} ===")
    print(f"{'arm':4s} {'when':7s} {'mode':12s} {'reached':7s} {'steps':5s} {'grads':5s} {'final fmax':10s} {'decisions':9s} {'by rules':8s} {'by LLM':6s} {'API calls':9s} {'rollb':5s} cap changes (step, new cap)")
    for arm in G["arms"]:
        cands = ([dict(NOW[arm], arm=arm)] if arm in NOW else []) + ([dict(REF_EARLIER[g][arm], arm=arm)] if show_ref and arm in REF_EARLIER.get(g, {}) else [])
        for r in cands:
            if r["session"] != "now":
                k = "gemini" if arm in ("GS", "GS2", "A3p") else "rules" if arm == "RS" else None
                r.update(mode=MODE[k], by_rules=r["decisions"] if k == "rules" else 0, by_llm=r["decisions"] if k == "gemini" else 0)
            r["group"] = g; TABLE.append(r)
            print(f"{arm:4s} {r['session']:7s} {r['mode']:12s} {('yes' if r['converged'] else 'NO'):7s} {str(r['steps']):5s} {str(r['grads']):5s} {r['final_fmax']:<10.4g} {str(r['decisions']):9s} {str(r['by_rules']):8s} {str(r['by_llm']):6s} "
                  f"{str(r['llm_api_calls']):9s} {str(r['rollbacks']):5s} {r['cap_changes'][:6]}" + ("  DEGRADED" if r.get("degraded") else ""))
print("\n=== Every decision of the LLM + rules runs: who decided, and why the LLM was consulted (actions as proposed; the validator may clip them) ===")
for arm in ("HS", "H3p"):
    if arm not in DEC: continue
    n_llm = sum(x["decided_by"].startswith("llm") for x in DEC[arm]); print(f"\n{arm}: {len(DEC[arm])} decisions, {len(DEC[arm]) - n_llm} by the rules, {n_llm} handed to the LLM; {api_calls.get(arm, 0)} Gemini API calls")
    for x in DEC[arm]:
        line = f"   step {str(x['step']):>3s} pass {x['pass_id']} {x['event']:12s} {str(x['trigger'] or ''):20s} {x['decided_by']:8s} {str(x['handover'] or ''):3s} -> {x['action']} {x['params'] or ''}"
        if x["decided_by"].startswith("llm"):
            rw = x.get("rules_would_do"); same = rw is not None and rw.get("action") == x["action"] and (rw.get("params") or {}) == (x["params"] or {})
            line += f"   | rules would do: {('the same' if same else str(rw.get('action')) + ' ' + str(rw.get('params') or '')) if rw else '(not computed)'}"
        print(line)
    if n_llm == 0: print("   The LLM was never consulted in this run: it is in effect a rules-only run. Report it as such.")
for a, b in (("HS", "GS"), ("H3p", "A3p")):
    if a in NOW and b in NOW:
        x, y = NOW[a], NOW[b]
        print(f"\n{a} (LLM + rules) vs {b} (LLM), same session: reached {('yes' if x['converged'] else 'NO')} / {('yes' if y['converged'] else 'NO')}; steps {x['steps']} / {y['steps']}; gradients {x['grads']} / {y['grads']}; "
              f"Gemini API calls {x['llm_api_calls']} / {y['llm_api_calls']}; rollbacks {x['rollbacks']} / {y['rollbacks']}; final energy difference {1000 * (x['final_E'] - y['final_E']):+.1f} meV.")
json.dump(dict(table=TABLE, decisions=DEC, handover_conditions=HANDOVER), open(os.path.join(EXPERIMENT_DIR, "hybrid_summary.json"), "w"), indent=1, default=float)

STYLE = {"hybrid": ("#1baf7a", "-", 2.4), "gemini": ("#eb6834", "-", 1.8), "rules": ("#2a78d6", "-", 1.8), None: ("#5f6368", "-", 1.2)}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, (g, G) in zip(axes, GROUPS.items()):
    for r in [t for t in TABLE if t["group"] == g]:
        k = {"LLM + rules": "hybrid", "LLM": "gemini", "rules only": "rules"}.get(r["mode"]); c, ls, lw = STYLE[k]; now = r["session"] == "now"
        ax.plot(range(1, len(r["fmax"]) + 1), np.clip(r["fmax"], None, 50), color=c, ls="-" if now else "--", lw=lw if now else 1.1, alpha=1.0 if now else 0.75, label=f"{r['arm']} {r['mode']} ({r['session']})")
        for st in r.get("llm_steps", []) if k == "hybrid" else []:
            if 0 < st <= len(r["fmax"]): ax.plot([st], [r["fmax"][st - 1]], "o", color="#eb6834", ms=7, mec="white")
    ax.axhline(ENDPOINT_FMAX, color="#202124", ls=":", lw=1); ax.set_yscale("log"); ax.set_title(G["title"]); ax.set_xlabel("optimizer step"); ax.legend(fontsize=7)
axes[0].set_ylabel("largest force (fmax, v7 convention)"); fig.tight_layout(); os.makedirs(os.path.join(EXPERIMENT_DIR, "figures"), exist_ok=True)
fig.savefig(os.path.join(EXPERIMENT_DIR, "figures", "hybrid_fmax.png"), dpi=150); plt.show()
print("Reading guide: green = LLM + rules, orange = LLM, blue = rules only, grey = plain. Solid = this session, dashed = embedded earlier runs. Orange dots on a green line = triggers handed to the LLM.")
print("Single runs. Gemini repeats differ by a few steps, so small step differences between modes are not findings.")
if BACKEND != "skala": print("STAND-IN BACKEND: wiring test only. These numbers must not be reported, and the earlier Skala rows are not shown.")


In [ ]:
# @title 16b. Deformed-start result: energy released, and is it the same structure as the first run?
import json, os, numpy as np
from ase.io import read as ase_read
_fu = [r for r in RESULTS if r.get("arm") in ARMS]
if not _fu:
    print("No finished run in this session.")
else:
    summ = _fu[-1]; arm = summ["arm"]; run_dir = os.path.join(EXPERIMENT_DIR, f"{arm}_{STARTS[0]}")
    fin = ase_read(os.path.join(run_dir, "final.xyz")); pos = fin.positions; sym = fin.get_chemical_symbols()
    E_fin = summ.get("final_energy_eV"); E_st = DEFORM["E_start_eV"]
    rel = None if (E_fin is None or E_st is None) else E_st - E_fin
    info = dict(shape=SHAPE, start="D2 (very deformed)", arm=arm, basis="def2-svp", backend=BACKEND, skala_version=SKALA_VERSION,
                E_start_eV=E_st, E_final_eV=E_fin, energy_released_eV=rel, energy_released_kcal=None if rel is None else rel * 23.0605, released_at_least_5_eV=None if rel is None else bool(rel >= 5.0),
                start_rmsd_to_reference_A=DEFORM["rmsd_to_ref_A"], start_plane_rms_A=DEFORM["plane_rms_A"],
                converged=summ.get("converged"), final_fmax_v7_units=summ.get("final_fmax"), gradients=summ.get("gradients_optimization"), minutes=summ.get("minutes_optimization"),
                bond_graph_ok=summ["final_structure"]["bond_graph_ok"], plane_rms_A=summ["final_structure"]["plane_rms_A"], curvature_any_negative=summ.get("curvature_any_negative"), status=summ.get("status"))
    hand = "/content/drive/MyDrive/fu5_handoff" if USE_DRIVE and os.path.isdir("/content/drive/MyDrive") else "/content/fu5_handoff"
    os.makedirs(hand, exist_ok=True)
    jf, xf = os.path.join(hand, f"{SHAPE}_svp_final.json"), os.path.join(hand, f"{SHAPE}_svp_final.xyz")
    if os.path.exists(jf) and os.path.exists(xf):
        first = json.load(open(jf)); p1 = ase_read(xf).positions
        d_rmsd = float(rmsd_sym(p1, pos)); d_meV = 1000.0 * (E_fin - first["final_energy_eV"])
        info.update(first_run_E_final_eV=first["final_energy_eV"], rmsd_to_first_run_A=round(d_rmsd, 4), energy_difference_to_first_run_meV=round(d_meV, 2),
                    same_structure_as_first_run=bool(d_rmsd < 0.02 and abs(d_meV) < 10.0), same_structure_rule="shape difference < 0.02 A and energy difference < 10 meV (rule written before the run)")
    else:
        info.update(note="first notebook's hand-off file not found on Drive; run the first Module 1 notebook for this shape to get the comparison")
    print(json.dumps(info, indent=2, default=float))
    if rel is not None and rel < 5.0: print("\nNOTE: the start released less than 5 eV with Skala. The notebook does not change the start; report the number as it is.")
    if not (summ.get("converged") and info["bond_graph_ok"]): print("\nNOTE: not converged to fmax <= %s or the bond graph changed; read the log." % ENDPOINT_FMAX)
    json.dump(info, open(os.path.join(hand, f"{SHAPE}_svp_deformed6_run.json"), "w"), indent=2, default=float)
    open(os.path.join(hand, f"{SHAPE}_svp_deformed6_run.xyz"), "w").write(f"{len(sym)}\n5FU {SHAPE} from very deformed start D2, Skala-1.1/def2-SVP, E={E_fin} eV\n" + "\n".join(f"{s:2s} {p[0]:14.8f} {p[1]:14.8f} {p[2]:14.8f}" for s, p in zip(sym, pos)) + "\n")
    print("Written to", hand, ":", f"{SHAPE}_svp_deformed6_run.xyz and .json (the first notebook's files are not touched)")
    try:   # energy-release curve
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        rows = [json.loads(l) for l in open(os.path.join(run_dir, "steps.jsonl")) if l.strip()]
        e = np.array([E_st] + [r["energy_eV"] for r in rows]) - E_fin
        fig, ax = plt.subplots(figsize=(7, 3.6)); ax.plot(range(len(e)), e, lw=1.6); ax.axhline(0, color="#888", lw=.8); ax.axhline(5.0, color="#d93025", lw=.8, ls=":")
        ax.set_xlabel("optimizer step"); ax.set_ylabel("energy above final (eV)"); ax.set_title(f"{SHAPE}: energy released from the deformed start"); fig.tight_layout()
        png = os.path.join(hand, f"{SHAPE}_deformed6_energy_release.png"); fig.savefig(png, dpi=150); plt.close(fig); print("Saved", png)
        try:
            from IPython.display import Image, display; display(Image(png))
        except Exception: pass
    except Exception as ex: print("(energy curve skipped:", repr(ex)[:100], ")")


In [ ]:
# @title 16c. Did Run 1 (normal start) and Run 3 (deformed start) reach the same final structure?
# Stand-alone cell: needs only the two hand-off files on Drive (or /content/fu5_handoff). Run it after cell 16b.
import json, os, numpy as np
from ase.io import read as ase_read
from ase.data import covalent_radii, atomic_numbers

RMSD_RULE_A = 0.02      # shape rule, fixed before any run
DE_RULE_MEV = 10.0      # energy rule, fixed before any run
hand = "/content/drive/MyDrive/fu5_handoff" if os.path.isdir("/content/drive/MyDrive") else "/content/fu5_handoff"
f1x, f1j = os.path.join(hand, f"{SHAPE}_svp_final.xyz"), os.path.join(hand, f"{SHAPE}_svp_final.json")
f2x, f2j = os.path.join(hand, f"{SHAPE}_svp_deformed6_run.xyz"), os.path.join(hand, f"{SHAPE}_svp_deformed6_run.json")
missing = [p for p in (f1x, f1j, f2x, f2j) if not os.path.exists(p)]
if missing:
    print("Cannot compare yet. Missing file(s):"); [print("  ", p) for p in missing]
    print("Run 1 files come from the first Module 1 notebook (same SHAPE); Run 3 files from cell 16b of this notebook.")
else:
    a1, a2 = ase_read(f1x), ase_read(f2x); J1, J2 = json.load(open(f1j)), json.load(open(f2j))
    assert a1.get_chemical_symbols() == a2.get_chemical_symbols(), "atom order differs between the two files"
    sym = a1.get_chemical_symbols(); P1, P2 = a1.positions.copy(), a2.positions.copy()
    def kabsch(A, B):                       # rotate and shift B onto A (proper rotation only)
        Ac, Bc = A - A.mean(0), B - B.mean(0); U, S, Vt = np.linalg.svd(Bc.T @ Ac)
        d = np.sign(np.linalg.det(U @ Vt)); D = np.diag([1, 1, d]); R = U @ D @ Vt
        return Bc @ R + A.mean(0)
    P2a = kabsch(P1, P2); dist = np.linalg.norm(P1 - P2a, axis=1)
    heavy = np.array([s != "H" for s in sym])
    rmsd_all = float(np.sqrt((dist ** 2).mean())); rmsd_heavy = float(np.sqrt((dist[heavy] ** 2).mean())); max_atom = float(dist.max())
    # bonds from Run 1 (1.2 x covalent radii), then compare lengths and angles
    Z = [atomic_numbers[s] for s in sym]; n = len(sym); bonds = []
    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(P1[i] - P1[j]) < 1.2 * (covalent_radii[Z[i]] + covalent_radii[Z[j]]): bonds.append((i, j))
    dbond = [abs(np.linalg.norm(P1[i] - P1[j]) - np.linalg.norm(P2[i] - P2[j])) for i, j in bonds]
    nbr = {i: [] for i in range(n)}
    for i, j in bonds: nbr[i].append(j); nbr[j].append(i)
    def ang(P, i, c, k):
        u, v = P[i] - P[c], P[k] - P[c]; return np.degrees(np.arccos(np.clip(u @ v / np.linalg.norm(u) / np.linalg.norm(v), -1, 1)))
    dang = [abs(ang(P1, i, c, k) - ang(P2, i, c, k)) for c in range(n) for ii, i in enumerate(nbr[c]) for k in nbr[c][ii + 1:]]
    E1, E2 = J1["final_energy_eV"], J2["E_final_eV"]; dE_meV = 1000.0 * (E2 - E1)
    same_shape, same_energy = rmsd_all < RMSD_RULE_A, abs(dE_meV) < DE_RULE_MEV; same = bool(same_shape and same_energy)
    rows = [("Run 1 (normal start)  final energy, eV", f"{E1:.6f}"), ("Run 3 (deformed start) final energy, eV", f"{E2:.6f}"),
            ("Energy difference Run 3 - Run 1, meV", f"{dE_meV:+.2f}   (rule: |difference| < {DE_RULE_MEV} meV -> {'ok' if same_energy else 'NOT met'})"),
            ("Shape difference, all 12 atoms, RMSD (A)", f"{rmsd_all:.4f}   (rule: < {RMSD_RULE_A} A -> {'ok' if same_shape else 'NOT met'})"),
            ("Shape difference, heavy atoms only (A)", f"{rmsd_heavy:.4f}"), ("Largest single-atom difference (A)", f"{max_atom:.4f}"),
            ("Largest bond-length difference (A)", f"{max(dbond):.4f}"), ("Largest bond-angle difference (degrees)", f"{max(dang):.2f}"),
            ("Ring planarity RMS, Run 1 / Run 3 (A)", f"{J1.get('plane_rms_A', float('nan')):.4f} / {J2.get('plane_rms_A', float('nan')):.4f}"),
            ("How far the Run 3 start was from the minimum (A RMS)", f"{J2.get('start_rmsd_to_reference_A', float('nan')):.3f}"),
            ("Energy Run 3 released from its start, eV", f"{J2.get('energy_released_eV', float('nan')):.3f}")]
    print(f"=== {SHAPE}: Run 1 vs Run 3 (both Skala-1.1 / def2-SVP, same optimizer) ===")
    for k, v in rows: print(f"{k:58s} {v}")
    verdict = "SAME STRUCTURE" if same else "NOT THE SAME STRUCTURE (by the rules fixed beforehand)"
    print("\nVERDICT:", verdict)
    if same:
        sentence = (f"Starting from two different geometries, a standard start and a start deformed by {J2.get('start_rmsd_to_reference_A', float('nan')):.2f} A RMS "
                    f"(about {J2.get('energy_released_eV', float('nan')):.1f} eV above the minimum), the workflow reached the same 5-fluorouracil {SHAPE} structure: "
                    f"final energies differ by {abs(dE_meV):.1f} meV and the structures differ by {rmsd_all:.3f} A RMSD (largest bond length difference {max(dbond):.3f} A).")
    else:
        sentence = ("The two runs ended at different structures by the rules fixed beforehand "
                    f"(energy difference {dE_meV:+.1f} meV, RMSD {rmsd_all:.3f} A). Do not write that they reached the same structure. "
                    "Report both numbers as they are. A small energy gap with a larger shape gap usually means a very flat energy surface "
                    "(for example a rotating group), which would need a tighter final convergence to settle.")
    print("\nSentence for the paper (check the numbers once more before using it):\n" + sentence)
    out = dict(shape=SHAPE, rule_rmsd_A=RMSD_RULE_A, rule_dE_meV=DE_RULE_MEV, E_run1_eV=E1, E_run2_eV=E2, dE_meV=dE_meV, rmsd_all_A=rmsd_all,
               rmsd_heavy_A=rmsd_heavy, max_atom_diff_A=max_atom, max_bond_diff_A=max(dbond), max_angle_diff_deg=max(dang),
               same_energy=bool(same_energy), same_shape=bool(same_shape), same_structure=same, per_atom_diff_A=[round(float(x), 5) for x in dist], sentence=sentence)
    json.dump(out, open(os.path.join(hand, f"{SHAPE}_run1_vs_run3_same_structure.json"), "w"), indent=2)
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        col = {"H": "#9aa0a6", "C": "#3c4043", "N": "#1a73e8", "O": "#d93025", "F": "#188038"}
        fig, ax = plt.subplots(1, 3, figsize=(13, 3.9), gridspec_kw=dict(width_ratios=[1.2, 1.2, 1]))
        c0 = P1.mean(0)
        for a, (u, v, lab) in zip(ax[:2], [(0, 1, "top view"), (0, 2, "side view")]):
            for i, j in bonds:
                a.plot([P1[i, u] - c0[u], P1[j, u] - c0[u]], [P1[i, v] - c0[v], P1[j, v] - c0[v]], color="#1a73e8", lw=2.2, alpha=.6)
                a.plot([P2a[i, u] - c0[u], P2a[j, u] - c0[u]], [P2a[i, v] - c0[v], P2a[j, v] - c0[v]], color="#f29900", lw=1.2, ls="--")
            for i, s in enumerate(sym): a.scatter(P1[i, u] - c0[u], P1[i, v] - c0[v], s=55, color=col.get(s, "k"), zorder=3)
            a.set_aspect("equal"); a.set_title(f"{SHAPE}, {lab}\nblue = Run 1, orange dashed = Run 3", fontsize=9); a.set_xticks([]); a.set_yticks([])
        ax[2].bar(range(n), dist * 1000, color=[col.get(s, "k") for s in sym]); ax[2].set_xticks(range(n)); ax[2].set_xticklabels(sym, fontsize=8)
        ax[2].set_ylabel("difference between runs (milli-A)"); ax[2].set_title(f"per atom, RMSD {rmsd_all:.3f} A", fontsize=9)
        fig.suptitle(f"{SHAPE}: {verdict}", fontsize=11); fig.tight_layout()
        png = os.path.join(hand, f"{SHAPE}_run1_vs_run3_overlay.png"); fig.savefig(png, dpi=150); plt.close(fig); print("Saved", png)
        try:
            from IPython.display import Image, display; display(Image(png))
        except Exception: pass
    except Exception as ex: print("(overlay plot skipped:", repr(ex)[:100], ")")
    print("Saved", os.path.join(hand, f"{SHAPE}_run1_vs_run3_same_structure.json"))


In [ ]:
# @title 18. Zip everything for download (everything is already on Google Drive too)
import zipfile
zip_path = f"/content/agentic_skala_{MOLECULE.lower()}_{REQUIREMENT}{RUN_TAG}{'_STANDIN' if BACKEND != 'skala' else ''}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(EXPERIMENT_DIR):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), EXPERIMENT_DIR))
print("Zip written:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files as colab_files; colab_files.download(zip_path)
except Exception: pass